# <center>企业级多 RAG 中台实战第一节课：整体架构与 Native RAG 链路</center>

&emsp;&emsp;今天我们正式进入一个真实的企业级项目——`FF-CompanyBrain`。它不是一个demo级的 RAG 演示项目，而是一个把**多条相互独立的 RAG 链路**装进同一个产品工作台、通过统一身份与统一 HTTP 分发组织起来的"企业级知识中台"。如果你此前接触过的 RAG 都是"读一批文档、切块、embedding、向量检索"这条单链路，那么这节课要回答的问题会更进一步：当一个平台里同时跑着三条技术栈完全不同的 RAG 链路时，它们是怎么被统一管理、又各自怎么落地的？我们会用整整一节课，把其中最有代表性的一条——`Native RAG`（也就是项目里的 `traditional-rag` 模块）——从架构位置一路拆到检索命中的每一行核心逻辑。

&emsp;&emsp;这节课我们会重点抓住三件事。第一件是**平台的分发骨架**：为什么统一 API 只做鉴权和分发、坚决不碰任何模块的数据库，这条边界一旦破了会发生什么。第二件是 `Native RAG` 的**三路 RRF 混合检索**：关键词全文、字面子串、向量语义三路各自召回，再用一个叫 `RRF` 的公式融合排名——这是本课信息含量最高、后面所有优化都依赖它的核心机制。第三件是**从"看起来优化了"到"能证明优化了"的转变**：企业级项目里任何优化都要有数值和评测撑腰，否则只是凭感觉。

&emsp;&emsp;除了这三条技术主线，本课还埋了一条贯穿全程的**元认知线索**：这节课的大量素材本身就是 AI 读源码产出的结论，我们会在几个关键节点停下来，现场演示"这个结论到底是怎么被验证出来的"。我们把这套方法叫作**三层验证阶梯**——声明层（AI 给的只是假设）、源码坐实层（去代码里找到那一行）、运行时验证层（对照真实日志或跑一次看输出）。在 `vibe coding` 越来越普遍的今天，敢不敢把 AI 的输出直接当真，取决于你有没有这套验证纪律。为了把这些讲透，我们按"环境部署与项目准备 → 整体架构全景 → 现场端到端演示 → Native RAG 内部拆解 → 企业级优化案例 → 回顾自测"的主线逐层推进——既然是真实项目，我们就从把它在本地跑起来开始。

> 📌 **目标受众与前置要求**：本课面向**有 RAG/LLM 基础的中级后端 / AI 应用开发者**——你应当已经理解 embedding 与向量检索的基本概念、熟悉 HTTP/REST，并能读懂 Python 代码。你**不需要**预先熟悉 PostgreSQL 全文检索（`tsvector`）、`trigram` 字面匹配、`RRF` 融合算法，这些本课会补讲；本课**第 0 章会带你把整套 `FF-CompanyBrain` 平台在本地部署起来**；而在讲解 `Native RAG` 检索算法内核时，我们另用零外部依赖的纯 Python 内存版最小复现——这样既能看清真实项目怎么落地，又能在讲算法时脱离环境依赖、聚焦逻辑本身。

> 📌 **学完本节你将带走这些能力**：① 能画出"平台分发 + 三链路"架构图，讲清 `apps/api` 只鉴权分发不碰模块库；② 能对着源码复述 `Native RAG` 从上传到检索命中的每一步；③ 能用"召回层 vs 重排层"框架分析任意企业级 RAG 项目的优化空间；④ 会用三层验证阶梯核实 AI / 文档给出的技术结论。

> 📅 **时效性说明**：本课全部源码引用基于 `FF-CompanyBrain` 仓库 `feat/frontend-revamp` 分支 2026 年 7 月初的代码状态。所有 `file:line` 引用都是真实可核对的——你可以在仓库里用 `grep -n` 或编辑器跳到对应行确认。本课的 MVP 代码是**教学最小复现**，忠实核心算法但零外部依赖，与生产源码分属两个层次，正文会逐处标注。

## <center>第零章：环境部署与项目准备</center>

&emsp;&emsp;在深入架构和算法之前，我们先把这个真实项目在本地跑起来——毕竟"企业级"首先意味着它能被完整地部署、启动、访问。这一章我们会一步步把 `FF-CompanyBrain` 从代码克隆到服务启动全部走通，并把过程中最容易踩的坑提前标出来。等你亲手看到 `http://localhost:3000` 打开的控制台，后面讲的每一条架构边界、每一行检索代码，才有一个"它真的在跑"的落脚点。

> **【关于本章命令的执行环境】**：本章所有命令都在你的**开发机终端**（`macOS` / `Linux` shell）里执行，不在 Jupyter Notebook 内运行。下面用代码单元格呈现，只是为了方便你复制粘贴。绝大多数命令 macOS 与 Linux 通用，唯一分平台的是 `0.3` 启动本地 PostgreSQL——macOS 走 `Homebrew` 辅助脚本，Linux 走发行版包管理器，本章会分别给出。

> &emsp;一点需要先说清楚的边界：本章部署的是**完整的真实平台**（三条 RAG 链路 + 统一 API + Agent 网关）。而到了第四章讲 `Native RAG` 检索算法内核时，我们会另用一套零外部依赖的纯 Python 内存版最小复现——那是为了聚焦算法本身、脱离环境依赖。"部署真实项目"和"讲透核心算法"是两个不同层次的事，请不要把它们混为一谈。

**运行本课件代码前：先准备 Notebook 环境**

&emsp;&emsp;开始之前先分清两套环境，别混淆：**第 0 章的部署命令**（`bun run` 等）在你的开发机终端执行、用来把 `FF-CompanyBrain` 平台跑起来（工具链见 `0.2`）；而**第 1 章之后所有 Python 代码 cell**（连库查询、`Native RAG` 的 MVP、PDF / Excel 解析等）在 `Jupyter Notebook` 里运行，需要一个装好依赖的 Python 环境。下面三条命令用 `conda` 建好这个 Notebook 运行环境，只需做一次。

> **【关于以下命令的执行环境】**：这三条命令在你的**终端**里执行（不在 Notebook 内），`requirements.txt` 与本课件在同一目录。

In [ ]:
# 1. 创建 conda 环境（Python 3.11，与项目模块一致）
!conda create -n company-brain python=3.11 -y

In [ ]:
# 2. 安装本课件 Notebook 所需依赖（psycopg / openai / pymupdf / openpyxl / dotenv 等）
!conda run -n company-brain pip install -r requirements.txt

In [ ]:
# 3. 注册成 Jupyter kernel，之后在 Notebook 右上角就能选到 "Company Brain"
!conda run -n company-brain python -m ipykernel install --user --name company-brain --display-name "Company Brain"

&emsp;&emsp;装好后，在 `Jupyter` 里打开本课件，把右上角的 kernel 切换成 **Company Brain**，后面所有 Python cell 就都能在这个环境里运行了。`requirements.txt` 里逐条标注了每个依赖对应课件哪一节、以及连库 / embedding 的前提条件，可对照查看。

> **【踩坑预警】**：这套 `conda` 环境只负责跑 Notebook 里的 **Python cell**。第 0 章那些 `!bun run ...` 是**独立的终端命令**，依赖 `bun` + `uv`（见 `0.2`），不由这个环境提供；连库 SQL（§1.5 / §4.10）要 PostgreSQL 已启动、`embedding` 要 `.env` 配好 `OPENROUTER_API_KEY`，纯算法 / 解析类 cell 则不需要这些。

### 0.1 环境依赖总览

&emsp;&emsp;`FF-CompanyBrain` 是一个 TypeScript 全栈 + Python 模块的混合项目，所以本地依赖比单语言项目稍多一些。我们先把要装的东西一次性列清楚，避免装到一半才发现缺组件。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FF-CompanyBrain 本地部署依赖清单</font></p>
<div class="center">

| 依赖 | 版本要求 | 用途 |
|---|---|---|
| `Bun` | 最新稳定版 | TypeScript 全栈的运行时 + 包管理（`apps/*`、`packages/*`）|
| `uv` | 最新稳定版 | Python 模块的依赖管理（`traditional-rag`、`graph-rag`）|
| `Python` | 3.11+ | 两个 Python RAG 模块的运行时 |
| `PostgreSQL` | 17 | 各模块独立数据库的统一实例 |
| `pgvector` | 匹配 PG 17 | 向量检索扩展（`embedding` 列与相似度检索靠它）|
| `Apache AGE` | 可选 | **仅** GraphRAG 完整能力需要（图存储扩展），需手动编译 |

</div>

> **【踩坑预警】**：`Apache AGE` 是**可选**依赖，只有当你要跑 GraphRAG 这条链路的完整图检索能力时才需要它。它无法通过包管理器一键安装，需要为你的 PostgreSQL 版本手动编译；`postgres-dev.sh` 脚本只会**提示**你装它、并不会自动完成编译。如果你本次只关心本课主角 `Native RAG`，可以先跳过 AGE，平台其余部分不受影响。

### 0.2 获取代码与安装依赖

&emsp;&emsp;拿到项目后的第一件事是安装两套依赖：`Bun` 负责 TypeScript 侧的全部工作区，`uv` 负责两个 Python 模块。

**步骤一：进入项目根目录**

&emsp;&emsp;你拿到的就是一份完整的项目代码，不需要再从远程克隆。先把它放到本地（如果是压缩包就解压），然后用终端进入项目根目录——后续所有命令都默认在这个根目录下执行。

In [ ]:
# 进入你拿到的项目根目录（路径替换成你本地实际存放的位置）
!cd /path/to/ff-companybrain

**步骤二：安装 TypeScript 全栈依赖**

&emsp;&emsp;用 `Bun` 一次性安装所有工作区（`apps/*` 与 `packages/*`）的依赖。这一步会读取根目录的 `bun.lockb`，把前端、统一 API、Agent 网关的依赖全部装好。

In [ ]:
!bun install

**步骤三：同步 Python 模块依赖**

&emsp;&emsp;两个 Python 模块用 `uv` 各自管理依赖，需要分别同步。`--project` 参数指向对应模块目录，`uv` 会按各模块的 `pyproject.toml` 建好隔离环境。

In [ ]:
!uv sync --project modules/traditional-rag
!uv sync --project modules/graph-rag

&emsp;&emsp;三步执行完，代码侧的依赖就齐了。如果 `bun install` 或 `uv sync` 报网络错误，多数是镜像源问题，配置好国内镜像重试即可，不是项目本身的问题。

**步骤四：赋予部署脚本可执行权限**

&emsp;&emsp;项目里的三个部署脚本（`scripts/postgres-dev.sh` / `start-dev.sh` / `deploy.sh`）在仓库里是以**非可执行**方式保存的，而后面的 `bun run postgres:dev`、`dev:all`、`deploy` 都是以 `./scripts/xxx.sh` 的形式直接调用它们。因此在跑任何 `bun run` 部署脚本前，先统一补一次执行位，否则会报 `Permission denied`。这一步 macOS 与 Linux 完全一样。

In [ ]:
!chmod +x scripts/*.sh

> **【踩坑预警】**：`git clone` 拉下来、或从压缩包解压后，`scripts/*.sh` 很可能不带可执行位——此时 `bun run dev:all` / `deploy` / `postgres:dev` 会以 `./scripts/xxx.sh: Permission denied`（退出码 `126`）失败。这**不是脚本内容有 bug**，`chmod +x scripts/*.sh` 执行一次即可根治。判断自己是否踩坑：`ls -l scripts/` 看这几个 `.sh` 文件的权限位里有没有 `x`；`bun run scripts/*.ts` 那几个命令（`db:init` / `admin:create` / `smoke:platform`）由 `bun` 直接解释执行、不受此影响。

### 0.3 准备 PostgreSQL 与 pgvector

&emsp;&emsp;平台的每个模块都用独立的 PostgreSQL database（但可以在同一个实例里），所以我们需要一个带 `pgvector` 的 PostgreSQL 17。

**步骤一：启动本地 PostgreSQL（macOS）**

&emsp;&emsp;在 macOS 上，项目提供了一个辅助脚本帮你拉起本地 PostgreSQL——它内部用 `Homebrew` 装好 `postgresql@17` 与 `pgvector` 再启动服务。**这里要特别说清楚一个容易混淆的点**：这个脚本用 `brew services` 把 PostgreSQL 注册成**后台常驻服务**（由 macOS `launchd` 托管），装好并确认服务就绪后**脚本自己就退出了**——所以你**不需要一直开着这个窗口**，跑完关掉也没关系，PostgreSQL 会在后台持续运行。（注意别和后面 `0.6` 的 `dev:all` 搞混，`dev:all` 才是必须保持窗口开着的前台常驻进程。）

In [ ]:
# 请在一个独立终端运行（常驻服务，不要在 Notebook kernel 里跑）
!bun run postgres:dev

&emsp;&emsp;下面是首次执行这条命令时的实际终端输出。由于本机还没安装过 `postgresql@17`，脚本会先让 `Homebrew` 做一次自动更新，再安装 `postgresql@17` 与 `pgvector`——**第一次运行会明显偏慢，这是正常现象**，装好后再次运行会直接跳到启动服务。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174848166.png" width=80%></div>

> &emsp;图中若出现与某个 `Homebrew tap` 相关的 `git stash` 报错，那是本机第三方 tap 的历史遗留，与本项目无关，可以忽略。

&emsp;&emsp;继续往下，`Homebrew` 会列出这次要安装的 `postgresql@17` 及其依赖（`openssl@3`、`krb5`），并停下来询问 `Do you want to proceed? [y/n]`——输入 `y` 回车确认，它就开始逐个下载并安装这些组件（`bottle`）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174844491.png" width=80%></div>

&emsp;&emsp;依赖和 `pgvector` 都装好后，脚本会自动用 `brew services` 启动 `postgresql@17`，等到探测出 `PostgreSQL is ready` 并打印出数据库版本号，就说明本地 PostgreSQL 已经就绪。看到这一步，**脚本就执行完退出了**——你可以直接关掉这个窗口（PostgreSQL 已交给后台服务托管），也可以按 `q` 退出版本信息的分页显示后，继续在同一个窗口跑下一节的命令。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174846070.png" width=80%></div>

> &emsp;图里那句 `Apache AGE is required for GraphRAG but is not installed` 是正常提示，`AGE` 是可选依赖（只跑本课主角 `Native RAG` 用不到），忽略即可，详见下面的步骤三。

> **【踩坑预警】**：留意脚本最后 `SELECT version()` 打印出来的版本号。如果它显示的**不是你刚装的 `17`**（例如显示成 `PostgreSQL 14.x`），说明你本机 `5432` 端口上很可能**早就有另一个 PostgreSQL 在跑、占着端口**，新装的 `17` 并没有真正接管连接。这时平台实际连的是那个旧版本——只要它带 `pgvector` 也能正常用就不影响；但如果接下来 `db:init` 报"扩展 `vector` 不存在"或版本不匹配的错，根源多半就在这里。排查动作：`brew services list | grep postgres` 看本机有几个 PostgreSQL、哪个在运行，再用 `lsof -iTCP:5432 -sTCP:LISTEN` 确认 `5432` 端口到底被哪个进程占用。

**步骤二：启动本地 PostgreSQL（Linux）**

&emsp;&emsp;`postgres:dev` 脚本内部硬依赖 `Homebrew`，**仅适用于 macOS**；在 Linux 上它会直接以 `Homebrew is required` 报错退出。Linux 用户改用发行版的包管理器安装 PostgreSQL 17 与 `pgvector`，再把服务启起来。以 Debian / Ubuntu 为例：

In [ ]:
# Debian / Ubuntu：安装 PostgreSQL 17 + pgvector
!sudo apt-get update && sudo apt-get install -y postgresql-17 postgresql-17-pgvector
# 启动并设为开机自启（常驻服务）
!sudo systemctl enable --now postgresql

&emsp;&emsp;装好后无需手动建库——`pgvector` 扩展会在下一节 `db:init` 时随各模块 database 自动 `CREATE EXTENSION`。

> **【踩坑预警】**：Debian / Ubuntu 官方源里不一定有 `postgresql-17`，通常要先添加 PostgreSQL 官方 APT 源（`apt.postgresql.org`，即 `PGDG` 源）再安装。RHEL / Fedora 系则换成 `sudo dnf install -y postgresql17-server postgresql17-contrib` 并按 `PGDG` 说明单独安装 `pgvector`。无论哪种发行版，只要最终有一个带 `pgvector`、正在监听的 PostgreSQL 17，就满足平台要求，与 macOS 路径殊途同归。

**步骤三（可选）：为 GraphRAG 安装 Apache AGE**

&emsp;&emsp;如果你要跑 GraphRAG 完整能力，这一步为 PostgreSQL 编译安装 `Apache AGE` 图扩展。如上一节所述，它需要手动编译，`postgres-dev.sh` 只提示不代劳；只跑 `Native RAG` 可跳过。

### 0.4 配置环境变量

&emsp;&emsp;平台的数据库连接、端口、各类密钥都集中在 `.env` 文件里。项目提供了模板 `.env.example`，我们复制一份再按需填写。

**步骤一：复制环境变量模板**

In [ ]:
!cp .env.example .env
# 然后用你的编辑器打开 .env 按下表填写

**步骤二：填写关键变量**

&emsp;&emsp;`.env` 里的变量不少，下面这张表按用途分组，把你**必须**关注的关键项拎出来。其余变量保持模板默认值即可。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>.env 关键变量分组说明</font></p>
<div class="center">

| 变量组 | 代表变量 | 说明 |
|---|---|---|
| 数据库连接 | `POSTGRES_ADMIN_URL` / 各模块 `*_DATABASE_URL` | 管理连接 + identity / nano-brain / traditional-rag / graph-rag / agent 各自的独立 database |
| 服务端口 | `API_PORT` / `AGENT_GATEWAY_PORT` / 各模块 `*_HTTP_PORT` | 统一 API、Agent 网关、三个模块 HTTP 服务的端口 |
| 内部凭据 | `RAG_INTERNAL_TOKEN` | 统一 API 调用模块 HTTP 服务的内部令牌，**部署时必须改成非默认值** |
| 管理员 | `ADMIN_USERNAME` / `ADMIN_PASSWORD` | 首个管理员账号，供 `admin:create` 读取 |
| Embedding | `EMBEDDING_PROVIDER` / `EMBEDDING_BASE_URL` / `EMBEDDING_API_KEY` / `EMBEDDING_MODEL` | 向量生成服务（`Native RAG` 真实项目用 `qwen3-embedding-8b`）|
| Agent | `AGENT_PROVIDER` / `AGENT_BASE_URL` / `AGENT_API_KEY` / `AGENT_MODEL` | Agent 网关调用的对话模型 |

</div>

> **【本课端口约定】**：本课把统一 API 端口统一设为 `3101`（避开本机常被占用的默认 `3001`）。请在 `.env` 里把 `API_PORT` 和 `API_INTERNAL_BASE_URL` **两个变量都改成 `3101`**——两者必须一致，否则前端会把请求转发到错误端口、导致控制台连不上后端。后文所有 `curl` 健康检查命令与服务地址都以 `3101` 为准；如果你选择沿用默认的 `3001`，把这两个变量和后文命令里的端口一并保持 `3001` 即可，关键是三者一致。

&emsp;&emsp;上面这些变量里，数据库连接、端口、内部令牌都是**本地自定义**的，唯独三类**外部服务的 API 密钥需要到官方平台申请**。下面把申请入口整理成一张表，链接可以直接点开——注册登录后创建密钥，再把拿到的 key 填回 `.env` 对应变量即可。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>外部 API 密钥申请入口</font></p>
<div class="center">

| 服务 | 对应 `.env` 变量 | 官方申请入口 | 用途与注意事项 |
|---|---|---|---|
| `OpenRouter` | `EMBEDDING_API_KEY` / `AGENT_API_KEY` | [openrouter.ai/settings/keys](https://openrouter.ai/settings/keys) | embedding 向量生成 + Agent 对话模型（一个 key 可同时用于两者）；注册后需绑定支付方式，密钥**创建后只显示一次**，务必立即复制保存 |
| `DashScope`（阿里云百炼）| `DASHSCOPE_API_KEY` | [bailian.console.aliyun.com](https://bailian.console.aliyun.com/?tab=model#/api-key) | 全域问答的 `qwen3-rerank` 精排；需先完成阿里云**实名认证**并开通百炼服务；不配置则跳过精排、保留全部召回 |
| `MinerU` | `MINERU_API_KEY` | [mineru.net/apiManage/token](https://mineru.net/apiManage/token) | PDF 结构化解析；Token 是**有时效的 JWT**（约 14 天），过期需重新生成；另需配 `MINERU_BASE_URL=https://mineru.net`（只填 host） |

</div>

> &emsp;上表链接均为各服务官方入口（`OpenRouter` 官方 keys 页 / 阿里云百炼控制台 / `MinerU` API 管理页）。国内网络访问 `OpenRouter` 若不稳定，可配置镜像或代理；`DashScope` 与 `MinerU` 均为国内平台，直连即可。

> **【踩坑预警】**：`.env` 里最容易出问题的是密钥和内部令牌。第一，`EMBEDDING_API_KEY`、`AGENT_API_KEY`、`ADMIN_PASSWORD` **只能留在本地 `.env`、严禁提交到 Git**（`.env` 应在 `.gitignore` 里）。第二，`RAG_INTERNAL_TOKEN` 一定要改成非默认值——它是统一 API 与模块之间的内部信任凭据，用默认值等于门户大开。第三，`AGENT_DATABASE_URL` 可以和别的库共用一个 PostgreSQL 实例，但**必须是独立的 database**，不能和模块库混在一起。如果后面你要处理 PDF，还会用到 `MINERU_API_KEY`（一个有时效的 JWT，过期要重新获取）和 `MINERU_BASE_URL`（只填 host、不要带 `/api/v4`，否则路径重复会 404）。

### 0.5 初始化数据库与管理员

&emsp;&emsp;依赖和 `.env` 就绪后，我们让平台自动把所有数据库建好、迁移到位，再创建第一个管理员账号。

**步骤一：创建并迁移所有数据库**

&emsp;&emsp;`db:init` 会读取 `POSTGRES_ADMIN_URL`，按各模块的 database URL **自动创建缺失的 database**，然后依次迁移 identity、Nano Brain、Agent Gateway、Traditional RAG、GraphRAG 五套库。这一步把散落在各模块的建表 / 迁移一次性做完。

In [ ]:
!bun run db:init

&emsp;&emsp;下面是这一步的实际输出。开头几行 `database exists: ...` 表示这些 database 之前已经建过、这次直接复用（`db:init` 是**幂等**的，重复跑不会重复建库、也不会报错）；随后每行 `xxx database migrated` 是逐套库执行迁移。看到 identity、nano brain、agent gateway、traditional rag、graph rag 都打印出 `migrated`，就说明五套库全部初始化到位了。（`traditional rag` 出现两次 `migrated` 是正常的，它本身分了两批迁移。）

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174942659.png" width=70%></div>

**步骤二：创建管理员账号**

&emsp;&emsp;平台只有一个管理员层级。`admin:create` 默认从 `.env` 读取 `ADMIN_USERNAME` / `ADMIN_PASSWORD` 建号。

In [ ]:
!bun run admin:create

&emsp;&emsp;如果管理员已经存在，这个脚本会以状态码 `2` 退出——这是正常的"已存在、无需重建"信号，统一部署脚本也会把它当作可继续的状态，不必惊慌。

### 0.6 一键启动与部署预检

&emsp;&emsp;万事俱备，最后一步是把整个服务栈拉起来。日常开发用 `dev:all`，而在正式上线前，项目还提供了一个 `deploy` 预检脚本帮你把该检查的都检查一遍。

**步骤一：统一启动本地服务栈**

&emsp;&emsp;`dev:all`（等价于 `./scripts/start-dev.sh`）会在一个终端里把**完整服务栈一次性全部拉起**——注意它不只是后端，前端也一并包含在内。具体是六个进程：前端控制台 `apps/web`、统一 API `apps/api`、Agent 网关 `apps/agent-gateway`，以及 Nano Brain / Traditional RAG / GraphRAG 三个模块各自的 HTTP 服务。它是一个常驻的阻塞进程，请在**独立终端**运行、并保持它开着。

In [ ]:
# 请在一个独立终端运行（常驻阻塞进程，不要在 Notebook kernel 里跑）
!bun run dev:all

&emsp;&emsp;这是一个前台常驻进程，启动过程会刷出很长一串日志。下图是启动初期：六个服务被依次拉起（`api` / `nano-http` / `traditional-rag` / `graph-rag` / `agent-gateway` / `web`），随后打印出 `All services started` 和各服务地址——注意 `API` 这里显示的正是我们约定的 `3101`。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174820051.png" width=80%></div>

&emsp;&emsp;继续往下，各 Python 模块（`uvicorn`）会陆续打印 `Application startup complete`，前端 `Next.js` 打印 `Ready`，并给出本地访问地址 `http://localhost:3000`——看到这些，才说明整个服务栈真正起来了。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174820097.png" width=80%></div>

> **【踩坑预警】**：这串日志里会混入几类**正常的告警**，别误当成启动失败。一类是 `graph-rag` / `traditional-rag` 打印的 `DeprecationWarning: on_event is deprecated`——那是 `FastAPI` 提示某个写法过时，不影响运行；另一类是 `[web]` 的 `Next.js inferred your workspace root` / `multiple lockfiles`——检测到多个 lockfile 的提醒，也不影响。真正判断是否启动成功，看的是每个模块的 `Application startup complete`、前端的 `Ready`，以及下一节的健康检查。

&emsp;&emsp;启动成功后，终端会打印出每个服务的访问地址，方便你逐个确认是否都起来了：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>dev:all 一键启动的服务与访问地址</font></p>
<div class="center">

| 服务 | 类型 | 默认访问地址 |
|---|---|---|
| `apps/web` 前端控制台 | 前端 | `http://localhost:3000` |
| `apps/api` 统一 API | 后端入口 | `http://localhost:3101` |
| `apps/agent-gateway` Agent 网关 | 后端 | `http://localhost:3002` |
| Nano Brain HTTP | 后端模块 | `http://127.0.0.1:8100` |
| Traditional RAG HTTP | 后端模块 | `http://127.0.0.1:8101` |
| GraphRAG HTTP | 后端模块 | `http://127.0.0.1:8102` |

</div>

&emsp;&emsp;其中**前端**就是你打开 `http://localhost:3000`、用刚才创建的管理员账号登录后看到的控制台，它背后通过**统一 API**（`3101`）把请求分发到各模块——这正是第一章要讲的分发骨架。<font color=red>当浏览器里能正常打开这个控制台并登录，就说明前端、后端、三条链路、数据库已经全部连通，整套平台在你本地真实跑起来了——这是本章的验收标志。</font>

> &emsp;如果你想单独调试某一个服务（而不是一次全起），也可以用 `bun --cwd apps/web dev`、`bun --cwd apps/api dev` 这样的方式单独启动对应服务，端口与上表一致。

**步骤二：上线前的部署预检**

&emsp;&emsp;`dev:all` 是本地开发用的、不是生产守护进程。当你要把项目部署到正式环境前，用 `deploy` 做一次完整预检更稳妥：它会依次执行装依赖（`bun install --frozen-lockfile`）、同步两个 Python 模块（`uv sync`）、TypeScript 类型检查（`tsc --noEmit`）、前端构建（`apps/web build`）、数据库初始化和管理员创建，把上线前该过的关卡一次性跑一遍。

In [ ]:
!bun run deploy

&emsp;&emsp;下图是预检的实际过程：先装 `Bun` 依赖（`Checked 207 installs ... no changes`，因为之前已经装过）和两个 Python 模块依赖，再跑 `TypeScript` 类型检查（`Running TypeScript checks`），最后构建前端（`Building web frontend` → `next build`）。看到 `Compiled successfully`、`Generating static pages`，直到列出 `Route (app)` 路由表（`/`、`/_not-found`、`/admin`），就说明上线前该过的关卡都过了。图中那个 `Next.js inferred your workspace root` / `multiple lockfiles` 警告和 `dev:all` 时一样是正常提示，忽略即可。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174942664.png" width=80%></div>

&emsp;&emsp;如果只想跑其中一部分，`deploy.sh` 支持 `--skip-install` / `--skip-db` / `--skip-admin` 等参数按需跳过。

### 0.7 启动后的健康检查与验证

&emsp;&emsp;`dev:all` 打印出"All services started"只代表这几个进程被拉起来了，**并不等于每个服务都健康可用**——数据库没连上、端口冲突、某个模块启动报错，进程也可能还挂在那里。所以启动后我们要逐层做几个检查，确认平台真的能用，而不是"看起来起来了"。

**步骤一：检查后端各服务的健康状态**

&emsp;&emsp;平台在统一 API 上提供了健康检查端点。最省事的是查 `/modules/health`——它由统一 API 去逐个探测三个模块并聚合返回，**一个请求就能看全所有模块的状态**（这本身就是"统一 API 做分发、模块各自独立"这条架构边界的体现）。

In [2]:
# 统一 API 自身是否健康
!curl http://localhost:3101/health
# 统一 API 聚合探测三个模块（一个请求看全部模块健康）
!curl http://localhost:3101/modules/health

curl: (7) Failed to connect to localhost port 3101 after 0 ms: Couldn't connect to server
curl: (7) Failed to connect to localhost port 3101 after 0 ms: Couldn't connect to server


<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174824574.png" width=80%></div>

&emsp;&emsp;`/health` 正常会返回 `{"status":"ok","service":"api"}`；`/modules/health` 会返回一个包含各模块健康状态的列表，只有当三个模块都显示健康时，后端才算真正就绪。如果某个模块不健康，就去 `dev:all` 的终端输出里找那个模块的启动报错。

**步骤二：检查 Agent 网关**

&emsp;&emsp;Agent 网关是独立于三条 RAG 链路的公共能力层，单独确认一下它的健康端点。

In [ ]:
!curl http://localhost:3002/health

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174819003.png" width=70%></div>

&emsp;&emsp;返回 `{"status":"ok","service":"agent-gateway"}` 即表示网关正常。

**步骤三：验证前端控制台可登录**

&emsp;&emsp;后端确认健康后，在浏览器打开 `http://localhost:3000`，用 `0.5` 步创建的管理员账号登录。能正常登录并看到控制台，说明**前端渲染、统一 API 鉴权、身份库**这条链路是通的——这一步是纯浏览器操作，没有对应命令。

**步骤四：跑一次端到端冒烟测试**

&emsp;&emsp;前三步验证的是"各服务分别活着"，最后这一步验证"完整业务链路能串起来"。项目提供了一个平台级冒烟脚本，会模拟一次真实的端到端场景把主链路跑一遍。

In [ ]:
!bun run smoke:platform

&emsp;&emsp;这个命令等价于 `bun run scripts/smoke-platform-scenario.ts`。它跑通，说明"登录 → 统一 API 分发 → 模块处理 → 返回"这条核心链路是连通的——到这一步，你才能有把握地说平台"不只是起来了，而且是能用的"。

&emsp;&emsp;需要强调的是，`smoke:platform` **不是简单发一个请求**，而是完整模拟了一遍真实用户的操作：它先登录，再针对三条引擎（`Gbrain` / `Naive RAG` / `GraphRAG`）各自**新建一个知识场景、上传知识源、由管理员审批通过、等模块把文档处理入库，最后发起提问并校验返回结果里带有引用依据**。所以它一旦跑通，等于把"上传 → 处理 → 检索 → 带依据回答"这条最核心的业务闭环整条验证了一遍。

&emsp;&emsp;跑这条命令时，真正能看出门道的日志不在执行它的窗口，而在**跑着 `dev:all` 的那个窗口**——你会看到各模块被真实地一层层调用起来。下图是 `Traditional RAG`（对应 `Naive RAG` 引擎）这条链路留下的足迹：`POST /traditional/sources` 建知识源、`POST /traditional/documents` 上传文档、`GET /traditional/jobs/...` 轮询处理进度、`GET .../chunks` 确认切好了块、最后 `POST /traditional/search` 执行检索——这一串正好是第四章要拆解的 `Native RAG` 完整"入库切分 + 检索"流程的真实预演。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174851029.png" width=80%></div>

&emsp;&emsp;下面这段则是 `GraphRAG` 引擎的建图过程（也顺带解释了为什么三条引擎里 `GraphRAG` 那条最慢）：`LightRAG` 先用 `LLM` 从文本块里**抽取实体和关系**（`extracted 6 Ent + 2 Rel`，即 6 个实体、2 条关系），再经过多阶段合并、把关系写进向量库（`Upserting relation VDB: 系统集成风险->苏杭`），最后把实体、关系、文本块的向量分别 `flush` 落库。这一整套"抽取 → 建图 → 向量化入库"正是 `GraphRAG` 和 `Native RAG` 最大的区别——它把复杂度花在了构建知识图谱上。看到这些日志跑完、smoke 最终打印退出，就说明整个平台的三条链路都被真实地端到端验证了一遍。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703175120805.png" width=80%></div>

&emsp;&emsp;所有场景跑完后，`smoke:platform` 会打印一份结构化的汇总结果。最外层 `"ok": true` 表示整体通过；`cases` 里逐个列出每条引擎的结果——比如下图这条 `Gbrain` 场景，`status` 是 `已发布`、`citations: 5`（返回了 5 条引用依据）、`hit: true`（命中了预期答案），`moduleReferences` 还列出了这次答案引用的知识源（`wutong-plan.md`）。看到 `ok: true` 且每个 case 都 `hit: true`，就说明三条链路的端到端业务闭环全部验证通过了——到这一步，平台才算真正"能用"。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174853364.png" width=60%></div>

### 0.8 部署注意事项与常见坑

&emsp;&emsp;最后，我们把部署过程中最高频的几个坑集中列一遍，遇到时对号入座能省下大量排查时间。

> **【踩坑预警】**：前端构建（`apps/web build`）会用到 `next/font` 去下载 Google Fonts，**网络受限时构建可能因拉取字体失败**——这不是你的 TypeScript 或页面代码有 bug，配置好网络或字体镜像即可，别往代码方向排查。

> **【踩坑预警】**：`bun run dev:all` 只适合本地开发，**不是生产级守护进程管理器**。真正上生产时，应该用 `systemd` / `supervisord` / `Docker Compose` / `PM2` 之类的进程管理器分别托管各个服务，而不是靠这一个脚本常驻。

> **【踩坑预警】**：如果启动时报端口占用，检查 `.env` 里 `API_PORT` / `AGENT_GATEWAY_PORT` / 各模块 `*_HTTP_PORT` 是否与本机已有服务冲突；PostgreSQL 相关报错则优先确认 `pgvector` 扩展是否已在目标 database 里 `CREATE EXTENSION` 成功。

> **【踩坑预警】**：**反复重启 `dev:all` 时最容易撞的坑**——报 `EADDRINUSE: address already in use :::3000`，但你确认自己并没有别的程序在用 `3000`。根因是 `dev:all` 里的前端 `next dev` 在你按 `Ctrl-C` 停止时，它内部真正监听 `3000` 的子进程 `next-server` **可能没被一起收掉、残留占着端口**，于是下次前端启动失败、进而拖垮整个 `dev:all`。这是 `start-dev.sh` 收尾清理不彻底导致的，不是你的配置问题。判断：用 `lsof -iTCP:3000 -sTCP:LISTEN` 看是不是还挂着一个 `next-server`。解法：**重启前先清残留再启动**。

In [1]:
# 重启 dev:all 前，先清掉可能残留的前端进程（没有残留时这条命令无副作用）
!pkill -9 -f next-server

&emsp;&emsp;到这里，`FF-CompanyBrain` 已经在你本地完整跑起来了。有了这个"活着的平台"作参照，接下来我们就可以放心地拉远镜头，先看清它的整体架构全景——这些服务在架构上到底是怎么分工、怎么协作的。

## <center>第一章：整体架构——平台分发与三链路全景</center>

&emsp;&emsp;要读懂一个企业级平台，最忌讳一上来就扎进某个模块的源码。我们先花二十分钟建立一张**静态全景图**：这个平台由哪些顶层模块组成、请求是怎么在它们之间流动的、三条 RAG 链路又各自长什么样。这张全景图会成为后面所有章节的锚点——当我们在第四章深挖 `Native RAG` 内部时，你随时能回到这张图上确认"我现在在哪一块"。本章不写任何代码，是纯粹的认知建立；但正因为它是地基，请务必把每一条边界记牢。

&emsp;&emsp;在正式开始前，先花三十秒对齐一个概念：所谓"企业级"，比个人小项目多考虑的无非是几个维度——**身份鉴权**（谁能访问什么）、**多知识库隔离**（不同来源的数据不能互相串）、**可观测性**（出了问题能追踪到哪一步）。这几个维度会在后面反复出现，现在心里有个数就好。

### 1.1 什么是"企业级多 RAG 中台"

&emsp;&emsp;我们先给 `FF-CompanyBrain` 一句话定位：它是一个**本地运行的多知识库链路平台**，把多条相互独立的 RAG 模块放在同一个产品工作台中，通过统一身份、统一 API 和 Agent 网关组织起来。这句话里有三个关键词值得拆开看——"统一身份鉴权"意味着所有链路共用一套登录和权限体系，你不会在每条链路里重复实现一遍登录；"统一 HTTP 分发"意味着前端只跟一个入口打交道，由这个入口把请求转发到正确的模块；"N 条相互独立的 RAG 链路"则意味着每条链路可以用完全不同的技术栈、甚至不同的数据库，彼此不干涉。

&emsp;&emsp;这个定位和你熟悉的"单条 RAG"最大的区别在于：单条 RAG 关心的是"怎么把一批文档检索好"，而多 RAG 中台关心的是"怎么让 N 条各有所长的链路在同一个平台里和平共处、统一对外"。前者是算法问题，后者是架构问题。<font color=red>本课的核心价值，正是让你看清楚这个"架构问题"是怎么被一层层解构并落地的。</font>

### 1.2 七个顶层模块的黑盒契约

&emsp;&emsp;要理解一个平台怎么运转，最快的方式是先把它拆成若干**黑盒**——只看每个盒子"对外提供什么能力、接口在哪"，暂时不关心盒子内部怎么实现。`FF-CompanyBrain` 在顶层被拆成七个模块，下面这张表把每个模块的对外接口和外显能力列清楚。请注意，这里所说的"能力"都是**黑盒可观测**的，也就是说站在外部就能验证，而不是靠读内部代码猜的。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FF-CompanyBrain 七个顶层模块的黑盒契约</font></p>
<div class="center">

| 模块 | 对外接口 | 外显能力（黑盒可观测） |
|---|---|---|
| `apps/web` | 浏览器 `:3000` | 登录；员工端场景工作区（提问/知识库/任务）；管理员后台 |
| `apps/api` | HTTP `:3101` | `/auth` 鉴权；`/nano` `/traditional` `/graph` 分发；`/health` `/modules` |
| `apps/agent-gateway` | HTTP `:3002` | Agent 会话 / run；SSE 流；按 active module 拉起 MCP；checkpoint |
| `nano-brain` | HTTP `:8100` + MCP | capture/page；向量检索；出链反链；facts；Dream 自治整理 |
| `traditional-rag` | HTTP `:8101` + MCP | 多格式文档上传 + job；三路混合检索；表格结构化查询 |
| `graph-rag` | HTTP `:8102` + MCP | 文本/文件 ingest；图谱检索 search/ask；图谱策展 |
| `packages` 共享层 | 库（被 import） | identity 身份；gateway 分发库；contracts 类型；platform 场景模型 |

</div>

&emsp;&emsp;这张表里有一个容易被忽略的细节值得点一下：架构文档里通常只列 `identity / gateway / contracts` 三个共享包，但实际上 `packages` 层还藏着第四块——`platform`，它承载着场景模板、场景实例和处理状态机，是前端产品形态的支撑层。换句话说，共享层实际是"隐藏的第八块业务层"。我们这里只提一句，不展开——但它是一个很典型的"文档滞后于代码"的例子，也是我们后面反复强调"以代码事实为准"的第一个小注脚。

&emsp;&emsp;光看表格还不够直观。我们把这七个模块的**分层关系、依赖方向和数据边界**画成一张全局架构全景图——这正是本章开头承诺的那张"静态全景图"。后面每一章深入某个局部时，你都可以回到这张图上确认自己"现在在哪一块"。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174843655.png" width=70%></div>

<!-- ILLUSTRATION: type=architecture | content=FF-CompanyBrain 企业级多 RAG 中台·项目全局架构全景图，自上而下清晰分层，每层一个横向色块。第1层【前端】：apps/web 统一控制台（Next.js，端口 3000，含员工场景工作区 + 管理员后台）。第2层【平台入口·并列两个】：左 apps/api 统一 API（Bun+Hono，端口 3101，标注"只鉴权+归一化+分发、不碰模块库"），右 apps/agent-gateway Agent 网关（LangChain+LangGraph，端口 3002，标注"会话/SSE/MCP 编排/checkpoint"）。第3层【共享层 packages·横贯长条】：identity 身份 / gateway 分发库 / contracts 类型 / platform 场景模型。第4层【三条独立 RAG 链路·横向三个并列卡片】：Nano Brain（TypeScript，端口 8100）、Traditional RAG（Python+FastAPI，端口 8101，用橙色高亮边框标注"本课主角"）、GraphRAG（Python+LightRAG，端口 8102）。第5层【数据库】：PostgreSQL 多 database（pgvector + pg_trgm），三个模块与身份库、Agent 库各自独立，依次标注 platform_identity_db / nano_brain_db / traditional_rag_db / graph_rag_db / agent_gateway_db 五个库。图右侧竖排【外部服务】：OpenRouter（embedding）、DashScope（rerank）、MinerU（PDF）。箭头表达主数据流：apps/api 向下经 gateway 分发到三个模块的 HTTP、apps/agent-gateway 经 MCP 接三个模块，每个模块各自向下连自己的独立数据库。用一条红色隔离带高亮核心边界"apps/api 只接触请求与响应、绝不直连任何模块数据库"。整体白底、蓝绿主色、Traditional RAG 主角用橙色高亮、分层分明、中文清晰无错别字。 -->

&emsp;&emsp;这七个模块里，本课的主角是 `traditional-rag`，也就是我们说的 `Native RAG`。它对外提供文档上传、三路混合检索和表格查询三大能力。但在深入它之前，我们必须先搞清楚一个问题：员工在浏览器里发起的一个请求，是怎么最终抵达 `traditional-rag` 的？这就要看平台的分发骨架。

### 1.3 统一分发机制：只鉴权分发，不碰模块库

&emsp;&emsp;整个平台的请求流转有一条**铁律**：`apps/api` 作为统一入口，只做三件事——身份鉴权、请求归一化、模块 HTTP 分发，它**绝不直接操作任何模块的数据库**。这条边界写在仓库根目录的架构约束里（`ff-companybrain/CLAUDE.md` 架构边界第 3 条），是整个平台解耦的关键。下面这张图把请求的流转路径画清楚。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703175032637.png" width=50%></div>

<!-- ILLUSTRATION: type=architecture | content=平台分发架构图。从左到右四层：apps/web（浏览器前端，携带 Bearer Token）→ apps/api（统一入口，只做鉴权+归一化+分发，标注"不碰模块库"）→ @ff/gateway（分发库，注入 x-ff-internal-token / x-ff-user-id 等内部头）→ 右侧三个并列模块 nano-brain(:8100) / traditional-rag(:8101) / graph-rag(:8102)，每个模块各自持有独立数据库。重点用红色边框高亮 apps/api 与三个模块数据库之间"禁止直连"的隔离带。 -->

&emsp;&emsp;我们可以顺着这条链路走一遍：前端携带 `Bearer Token` 把请求发到 `apps/api`；`apps/api` 先验证 token 拿到用户上下文（谁、是不是管理员），然后调用共享的 `@ff/gateway` 分发库，由它向目标模块发 HTTP 请求，并**注入一组内部头**（内部令牌、用户 ID、用户名、是否管理员）；模块收到请求后基于这组内部头做权限过滤和业务处理，再把结果原路返回。整条链路里，`apps/api` 自始至终只接触"请求和响应"，从不接触模块数据库里的任何一行数据。

&emsp;&emsp;为什么要如此严格？我们不妨反过来分析一下：**假设 `apps/api` 为了图方便，直接读了 `traditional-rag` 的数据库表**，会发生什么？首先，`traditional-rag` 用 Python + PostgreSQL，而 `nano-brain` 用 TypeScript，两者的表结构和技术栈完全不同——`apps/api` 一旦直连，就必须同时理解三套模块的内部实现，模块想改自己的表结构就会牵连到入口层。其次，权限过滤本该在模块的查询边界完成，绕过模块直连会让权限规则散落两处，极易出现"入口层以为过滤了、模块层没过滤"的安全漏洞。<font color=red>这条"分发层不碰模块库"的边界，本质上是用"接口契约"替代"实现耦合"——它让每条链路可以独立演进，也让平台的优化工作可以只在模块内部进行而不惊动分发层。</font>这个"分发层与链路内部解耦"的结论，我们在第五章讲优化案例时还会再次印证。

### 1.4 平台技术栈全景

&emsp;&emsp;在深入任何一条链路之前，我们先鸟瞰一下整个平台用了哪些技术。`FF-CompanyBrain` 是一个 **TypeScript + Python 的混合技术栈**项目——统一入口和前端用 TypeScript 生态、两条 Python RAG 链路用 FastAPI，它们通过统一的 HTTP 边界拼在一起。下面这张表把每一层用什么、负责什么一次性列清楚。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>FF-CompanyBrain 平台技术栈全景</font></p>
<div class="center">

| 层 / 组件 | 技术栈 | 职责 |
|---|---|---|
| 前端 `apps/web` | `Next.js`（App Router）+ `React` | 统一控制台，只访问统一 API 和 Agent 网关 |
| 统一后端 `apps/api` | `Bun` + `TypeScript` + `Hono` | 身份鉴权 + 请求归一化 + 模块 HTTP 分发 |
| Agent 网关 `apps/agent-gateway` | **`LangChain` + `LangGraph`**（`langgraph-checkpoint-postgres` 做 checkpoint、`mcp-adapters` 接模块 MCP）| 会话 / run / SSE 流 / checkpoint / MCP 工具编排 |
| Nano Brain | `TypeScript`（`Bun`）| "私人知识管家"链路 |
| Traditional RAG（本课主角）| `Python 3.11+` + `FastAPI` + `uv` | 文档 / chunk / embedding / 三路混合检索 / 表格 |
| GraphRAG | `Python` + `FastAPI` + **`LightRAG`** | 图谱证据检索，实体/关系抽取委托 `LightRAG` |
| 数据库 | `PostgreSQL`（多 database）+ `pgvector` + `pg_trgm` | 各模块独立库；向量检索靠 pgvector、字面加速靠 pg_trgm |
| Embedding | `OpenRouter`（OpenAI-compatible）| 真实项目用 `qwen3-embedding-8b` |
| 认证 | 用户名密码 + `Bearer Token`（`packages/identity`）| 全平台统一身份 |
| 包管理 | `Bun` workspaces（TS）/ `uv`（Python）| 混合栈各自管理依赖 |

</div>

> &emsp;版本以项目 `package.json` / `pyproject.toml` 为准，本表只列技术选型不锁死版本号。

&emsp;&emsp;这张表里有两个点值得你特别记住。第一，**Agent 层用的是 `LangChain` + `LangGraph`**——注意它不只是"调一下 LangChain"，而是用了 `LangGraph` 的图式编排 + `langgraph-checkpoint-postgres` 把会话状态 checkpoint 到 PostgreSQL，再通过 `mcp-adapters` 把各模块的 MCP 工具挂载进来编排。第二，**`GraphRAG` 这条链路的图能力完全委托给 `LightRAG` 库**，本项目自己不实现任何图算法（这一点我们在第三章对比三链路时会再看到）。<font color=red>一个关键认知：这个平台的"企业级"不体现在某个炫技的自研算法上，而体现在"用合适的成熟技术栈、把多条异构链路统一编排起来"这件事上。</font>

### 1.5 权限管理模型

&emsp;&emsp;既然是"企业级"平台，"谁能访问什么"就是绕不开的一环。`FF-CompanyBrain` 的权限模型可以用五句话讲清楚，我们逐条来看。

&emsp;&emsp;第一，**只有一个管理员层级**。平台不设"链路管理员""局部管理员""分级管理员"——管理员拥有所有知识库链路的完整管理权限，其余都是普通用户。这个刻意的简化让权限模型不至于在第一版就过度复杂。

&emsp;&emsp;第二，**认证靠用户名密码 + `Bearer Token`**。用户登录后由 `packages/identity` 签发 `Bearer Token`（密码用 `bcrypt` 存、token 用 `SHA256` 校验），后续每个请求都带着这个 token。

&emsp;&emsp;第三，**鉴权在统一 API 的入口层执行**。请求进来先过 `apps/api` 的鉴权环节校验 `Bearer Token` 并判定是否管理员，未通过直接挡在门外（这正是第二章我们会用 `401` 演示的那道闸）。

&emsp;&emsp;第四，**数据权限靠 source 的 `private` / `public` 隔离**。每个知识来源（source）分私有和公共两类：`private` source 只有 owner 和管理员可读、可管理；`public` source 所有用户可读，但只有管理员能创建和管理。文档、chunk、表格、job 都通过所属 source 做权限过滤。

&emsp;&emsp;第五，**权限过滤在模块的查询边界完成，不靠前端藏按钮**。真正的权限判断写在各模块 core 层的数据库查询里（比如 Nano Brain 直接把 `is_admin OR public OR owner` 写进 SQL 的 `WHERE`）——<font color=red>前端隐藏入口只是体验优化，绝不是安全边界</font>。此外 `GraphRAG` 还多一层 `workspace` 实例级隔离，不同 source 对应独立的 `LightRAG` 实例、物理分区，天然互不串数据。

&emsp;&emsp;上面这五句话是"声明"。按这节课反复强调的**三层验证阶梯**，声明之后得能"运行时坐实"——权限到底是不是真写在数据库里？我们直接连平台的库查给你看。整个平台是**多库架构**：用户和会话在 identity 库（`platform_identity_db`），知识库数据在 traditional-rag 库（`traditional_rag_db`），物理隔离。

> 📌 **运行前提**：以下查询连接的是第 0 章部署起来的那套 PostgreSQL。请在 `.env` 里配好 `IDENTITY_DATABASE_URL` 和 `TRADITIONAL_RAG_DATABASE_URL`，且平台已入过数据。没有库环境的话，直接看下方贴出的真实输出即可。

&emsp;&emsp;先准备一个连库小工具——优先读 `.env` 里的连接串；如果 Notebook 没加载到 `.env`，就回退到本课默认的本地库名，然后跑一条 SQL、打印结果。

In [3]:
import os, psycopg
from pathlib import Path
from dotenv import load_dotenv

def load_project_env():
    """从课件同目录的 .env 读取连接串，兼容 Jupyter 启动目录不固定的情况"""
    seen = set()
    for folder in [Path.cwd(), *Path.cwd().parents]:
        env_path = folder / ".env"
        if env_path in seen:
            continue
        seen.add(env_path)
        if env_path.exists():
            load_dotenv(env_path, override=False)
            print(f"已加载 .env: {env_path}")
            return env_path

    load_dotenv()
    print("未在当前目录或父目录找到 .env；将尝试使用本地默认数据库连接串。")
    return None

load_project_env()

DEFAULT_DB_URLS = {
    "IDENTITY_DATABASE_URL": "postgresql:///platform_identity_db",
    "TRADITIONAL_RAG_DATABASE_URL": "postgresql:///traditional_rag_db",
}

def get_db_url(db_env):
    db_url = os.getenv(db_env)
    if db_url:
        return db_url

    fallback = DEFAULT_DB_URLS.get(db_env)
    if fallback:
        print(f"未找到 {db_env}，使用本地默认连接串: {fallback}")
        return fallback

    raise RuntimeError(
        f"缺少环境变量 {db_env}。请在项目 .env 中配置它，"
        "或先运行第 0 章的 .env 初始化与 db:init。"
    )

def show(db_env, sql, params=None):
    """连指定库执行 SQL 并打印结果(教学简化版)"""
    with psycopg.connect(get_db_url(db_env)) as conn, conn.cursor() as cur:
        cur.execute(sql, params or ())
        cols = [d.name for d in cur.description]
        print(" | ".join(cols))
        for r in cur.fetchall():
            print(" | ".join(str(x)[:40] for x in r))

已加载 .env: /Users/mac/PycharmProjects/JupyterProject/企业级知识中台项目/.env


&emsp;&emsp;先看**用户表**。平台的用户全部落在 identity 库的 `users` 表（建表见 `packages/identity/src/migrations.ts:7`），密码用 `bcrypt` 存 hash。

In [4]:
# 第一句「单管理员层级」的数据库证据
show("IDENTITY_DATABASE_URL",
     "SELECT id, username, is_admin, created_at FROM users LIMIT 5")

id | username | is_admin | created_at
1153b900-487d-440c-9648-9fce4085f438 | admin | True | 2026-06-29 12:42:40.776429+08:00


&emsp;&emsp;真实输出（连的是已部署的库）：

```
id | username | is_admin | created_at
1153b900-487d-440c-9648-9fce4085f438 | admin | True | 2026-06-29 12:42:40+08:00
(1 行)
```

&emsp;&emsp;整个平台就一个用户 `admin`、`is_admin=True`——第一句"只有一个管理员层级"被数据库坐实了。接着看**会话表**：登录签发的 `Bearer Token` 落在 `sessions` 表，而且**库里只存 `SHA256` hash、不存明文**（见 `packages/identity/src/tokens.ts:9`）。

In [5]:
# 第二句「token 用 SHA256 校验」的证据:库里存的是 hash 不是明文
show("IDENTITY_DATABASE_URL",
     "SELECT left(id,10) AS id, left(user_id,10) AS user_id, "
     "left(token_hash,16)||'...' AS token_hash_前16, expires_at "
     "FROM sessions ORDER BY created_at DESC LIMIT 5")

id | user_id | token_hash_前16 | expires_at
86274297-c | 1153b900-4 | a3571188484bc608... | 2026-07-10 11:26:39.088000+08:00
9227c7be-3 | 1153b900-4 | 248c76c3830165a7... | 2026-07-10 11:17:12.034000+08:00
e4e3d61c-8 | 1153b900-4 | a3be46dd3d9bcda9... | 2026-07-10 11:16:53.638000+08:00
56ee783f-5 | 1153b900-4 | 78826b72e26c579c... | 2026-07-10 11:15:51.943000+08:00
8b790044-c | 1153b900-4 | 82d45b32efd98c5d... | 2026-07-09 13:10:26.902000+08:00


&emsp;&emsp;真实输出：

```
id | user_id | token_hash_前16 | expires_at
8b790044-c | 1153b900-4 | 82d45b32efd98c5d... | 2026-07-09 13:10:26+08:00
a4be57f4-a | 1153b900-4 | f9f53f05f81093b3... | 2026-07-09 13:10:12+08:00
(共 5 行)
```

&emsp;&emsp;`token_hash` 是一串 `SHA256` 摘要、不是明文 token，`expires_at` 是 7 天后——这就足够坐实"`Bearer Token` + `SHA256` 校验"的落库形态。身份侧确认完之后，接下来进入权限侧：用户是谁只解决"认证"问题，真正能不能读到某个知识源，还要看模块库里的权限过滤条件。

&emsp;&emsp;最后看**权限过滤本身**：第四、五句说数据权限靠 source 的 `private`/`public` + owner，而且写在 SQL 的 `WHERE` 里。我们把权限列和真实的过滤条件都跑一遍。

In [6]:
# 数据源的权限列:kind(private/public) + owner_user_id
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT id, kind, owner_user_id, created_by FROM traditional_sources LIMIT 5")

# 权限过滤 WHERE(忠实 search.py:52 build_scope_filters):
# 把 is_admin 传成 false(模拟普通用户)、owner_user_id 传成当前用户 id,
# 结果只会返回 public 的 + owner 是他自己的 source
current_user_id = "1153b900-487d-440c-9648-9fce4085f438"  # 实际取自登录态,这里用 admin 的 id 演示
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT id, kind, owner_user_id FROM traditional_sources "
     "WHERE archived_at IS NULL AND (%s = true OR kind='public' OR owner_user_id=%s) LIMIT 5",
     (False, current_user_id))

id | kind | owner_user_id | created_by
e914f346-1da7-40d2-ba87-17bc86925ac4 | private | 1153b900-487d-440c-9648-9fce4085f438 | 1153b900-487d-440c-9648-9fce4085f438
11f4252a-2a3e-43a5-85e3-d67b2293c0b9 | private | 1153b900-487d-440c-9648-9fce4085f438 | 1153b900-487d-440c-9648-9fce4085f438
96364903-da70-4817-9e1d-6bf53c9b60c4 | private | 1153b900-487d-440c-9648-9fce4085f438 | 1153b900-487d-440c-9648-9fce4085f438
1dca912b-ee7f-4e89-8042-eba184267bc1 | public | None | user_0f1e22a8-1a9b-4de4-902d-8bddd0c7d5c
dc32ad1f-6ffb-4f62-bc07-533dd5a0cc16 | private | user_0f1e22a8-1a9b-4de4-902d-8bddd0c7d5c | user_0f1e22a8-1a9b-4de4-902d-8bddd0c7d5c
id | kind | owner_user_id
e914f346-1da7-40d2-ba87-17bc86925ac4 | private | 1153b900-487d-440c-9648-9fce4085f438
11f4252a-2a3e-43a5-85e3-d67b2293c0b9 | private | 1153b900-487d-440c-9648-9fce4085f438
96364903-da70-4817-9e1d-6bf53c9b60c4 | private | 1153b900-487d-440c-9648-9fce4085f438
1dca912b-ee7f-4e89-8042-eba184267bc1 | public | None
c13eb2e3-47b4-48b8-9

&emsp;&emsp;真实输出：

```
id | kind | owner_user_id | created_by
e914f346-1da7-40d2-ba87-17bc86925ac4 | private | 1153b900-... | 1153b900-...
(共 3 行)
--- 权限过滤后(is_admin=false) ---
id | kind | owner_user_id
e914f346-1da7-40d2-ba87-17bc86925ac4 | private | 1153b900-...
(共 3 行)
```

&emsp;&emsp;关键在第二条查询：我们把 `is_admin` 传成 `false`、把 `owner_user_id` 传成当前用户 id，结果只返回了 owner 是他自己的那几个 `private` source。这行 `WHERE (%s = true OR kind='public' OR owner_user_id=%s)` 就是 `search.py:52` 里真实跑的权限过滤——<font color=red>权限不是前端藏按钮，而是这一行 SQL 条件在数据库层实打实拦掉的。三条 SQL，把前面五句话从"声明"推进到了"运行时坐实"，这正是三层验证阶梯在权限模型上的一次完整应用。</font>

### 1.6 小结：没有唯一正确的架构

&emsp;&emsp;到这里，第一章的目标就达成了：你已经能讲清楚这个平台由哪七个模块分工组成、为什么 `apps/api` 只做鉴权分发而不碰模块库、以及三条链路各有不同的检索实现与复杂度落点。记住这个"没有唯一正确架构"的观念，它会帮你在读任何一个真实系统时保持冷静——不要一看到某条链路"只用了两路"就觉得它简陋，也不要因为另一条"用了三路"就觉得它先进。真正的问题永远是：**这条链路面对的证据类型，值不值得这么多路**。带着这个问题，我们进入第二章——把静态的架构图变成一次动态的、看得见的检索全流程。

## <center>第二章：现场演示——一次带依据的检索全流程</center>

&emsp;&emsp;第一章我们建立了静态的架构全景，但静态图有个天然的缺陷：它告诉你"有哪些盒子"，却没告诉你"请求是怎么在盒子之间真正流动起来的"。这一章我们换一种方式——先看一个完整的、带依据的检索效果，再从这个效果倒推回去，把中间每一个环节沿着一条真实的日志逐条走一遍。这种"先看成品、再拆底层"的顺序，能让你始终带着"这一步到底为最终效果贡献了什么"的问题去理解每个环节，比按部就班从头讲要扎实得多。

> **【本章的诚实边界】**：这一章我们对照的真实日志走的是 `/nano/capture` 这条 `Nano Brain` 链路，用它来坐实"统一分发机制"这个**所有链路共用的通用骨架**——也就是"鉴权 → 分发 → 处理 → 返回"这条链路怎么走通。它演示的不是 `Native RAG` 的内部检索细节（那是第四章的事），而是"任何请求进入平台后，分发层是怎么处理它的"。这个区分很重要，请一路记着。

### 2.1 先看最终效果：一次带依据的问答

&emsp;&emsp;我们先看效果。设想一位员工上传了一份"考勤制度"的文档，然后在提问框里输入"迟到几次扣绩效"。系统返回的不是一段凭空生成的话，而是一个**带依据的答案**——它会命中知识库里真实存在的那一段文字（一个 `chunk`），把命中的片段（`snippet`）连同相关度分数一起呈现出来。也就是说，答案背后有明确的证据出处，你能点开看到"这个回答是基于哪一段原文得出的"。

&emsp;&emsp;这里要澄清一个常见的误解：**这个"带依据的答案"里，最终那段自然语言回答是由上层 Agent 基于命中的证据 chunk 生成的，检索层本身只负责返回证据 chunk 加上相关度分数**。换句话说，检索层交出的是"证据"，不是"答案"。搞清楚这个分工，你才能理解为什么后面我们花大力气优化的是"检索层怎么把正确的证据排到前面"，而不是"怎么让模型把话说得更漂亮"。

### 2.2 从效果倒推：中间到底经过了哪些环节

&emsp;&emsp;既然看到了"带依据的答案"这个成品，我们就从它倒着推：要做到这一步，中间必须经过哪些环节？会发现它其实是由**两个独立的阶段接力完成**的，中间靠知识库（`chunks` 表）连接。**上传入库阶段**（离线，提前做好）走的是：文档上传 → `chunk` 切分 → `embedding` 向量化 → 写入 `chunks` 表，把文档变成"可被检索的向量"；**提问检索阶段**（在线，员工查询时）走的是：员工提问 → 问题同样做 `embedding` → 在 `chunks` 表里做相似度检索 → 返回命中的证据 `chunk`（带 `snippet` 和分数）。这两个阶段各自都是一次完整的请求，都要先过第一章讲的"鉴权 → 分发"通用骨架，再进入各自的处理。<font color=red>所以千万别把它们理解成一条从头到尾的直线——它们是被 `chunks` 表解耦的两条时间线：上传阶段把证据备好，提问阶段来取证据。</font>这套流程你在第一章的静态图里已经见过它的"骨架版本"，现在我们要看的是它"动起来"的样子。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174832521.png" width=70%></div>

<!-- ILLUSTRATION: type=flowchart | content=Native RAG 端到端数据流图，分成上下两条清晰的水平泳道，中间用"chunks 表（知识库）"作为连接枢纽。上泳道【上传入库阶段·离线，提前做好】从左到右：企业文档上传 → 鉴权+分发（通用骨架）→ chunk 切分 → embedding 向量化 → 写入 chunks 表；泳道标注"提问之前就完成，把文档变成可检索的向量"。下泳道【提问检索阶段·在线，员工查询时】从左到右：员工提问 → 鉴权+分发（通用骨架）→ query embedding → 从 chunks 表检索（相似度匹配）→ 返回带依据答案（命中 chunk + snippet + score）；泳道标注"员工查询时才发生"。中间的 chunks 表用醒目的圆柱数据库图标：上泳道有一条箭头指向它（写入），下泳道有一条箭头从它引出（读取），直观体现"上传把证据备好、提问来取证据"。两条泳道是独立的时间线、通过 chunks 表解耦，绝不是一条从头到尾的直线。 -->

### 2.3 对照真实日志逐条走

&emsp;&emsp;现在我们对照一条真实记录的日志（`trace-01`），把上传阶段的分发链路逐条走一遍。为了让你能直接在 Notebook 里看到这条链路的关键字段，我们把日志里最能说明问题的几个环节整理成一个结构，打印出来。请注意，下面这段代码只是把真实日志的关键字段直观展示出来，帮助我们看清"请求在每一层被谁处理、发生了什么"。

In [7]:
# 把 trace-01 真实日志的关键环节整理成结构，直观展示分发链路
# 数据来源：PKP/03-traces/trace-01.json（真实运行记录）
trace_01_steps = [
    {"seq": 1, "处理者": "apps/api", "动作": "员工持 Bearer Token 发起 POST /nano/capture",
     "关键证据": "进入 apps/api 的 nanoRouter，HTTP 201"},
    {"seq": 2, "处理者": "apps/api + identity", "动作": "验证 Bearer Token，解析出用户上下文",
     "关键证据": "auth.ts:27-31 getUserByBearerToken → UserContext{userId,isAdmin}"},
    {"seq": 3, "处理者": "@ff/gateway", "动作": "向模块 :8100 发请求并注入内部头",
     "关键证据": "gateway/src/index.ts:103-108 注入 x-ff-internal-token / x-ff-user-id"},
    {"seq": 6, "处理者": "nano-brain 模块", "动作": "chunk 切分 + 调用 embedding 生成向量并入库",
     "关键证据": "chunks 入库(embedding not null)；网络失败最多重试 5 次"},
    {"seq": 8, "处理者": "nano-brain 模块", "动作": "员工提问，query 向量化后做相似度检索",
     "关键证据": "命中 title=考勤制度, match_types=['vector']"},
]

# 逐条打印，观察请求在每一层的流转
for step in trace_01_steps:
    print(f"[seq {step['seq']}] {step['处理者']:20s} | {step['动作']}")
    print(f"          证据 → {step['关键证据']}\n")

[seq 1] apps/api             | 员工持 Bearer Token 发起 POST /nano/capture
          证据 → 进入 apps/api 的 nanoRouter，HTTP 201

[seq 2] apps/api + identity  | 验证 Bearer Token，解析出用户上下文
          证据 → auth.ts:27-31 getUserByBearerToken → UserContext{userId,isAdmin}

[seq 3] @ff/gateway          | 向模块 :8100 发请求并注入内部头
          证据 → gateway/src/index.ts:103-108 注入 x-ff-internal-token / x-ff-user-id

[seq 6] nano-brain 模块        | chunk 切分 + 调用 embedding 生成向量并入库
          证据 → chunks 入库(embedding not null)；网络失败最多重试 5 次

[seq 8] nano-brain 模块        | 员工提问，query 向量化后做相似度检索
          证据 → 命中 title=考勤制度, match_types=['vector']



<!-- cell-type: verify -->

&emsp;&emsp;从打印出来的链路里，我们能清楚看到三层职责的边界：`seq 2` 是纯粹的鉴权动作，`apps/api` 只从 `Authorization` 头里解析出"你是谁"，不做任何业务；`seq 3` 是纯粹的分发动作，由 `@ff/gateway` 把请求转给模块并注入内部头，这一步正是第一章那条"分发层不碰模块库"边界的具体落地——分发库交给模块的是一组身份头，而不是直接操作模块的表；`seq 6` 和 `seq 8` 才真正进入模块内部，做 chunk、embedding、检索这些业务。这条链路把第一章的静态边界坐实成了一次可追溯的真实流转。

> **【一个不能张冠李戴的细节】**：`trace-01` 的 `seq 6` 里记录的 `chunk` 落在 `nano_brain_db` 里（这条链路走的是 `Nano Brain`），它的向量维度是当时 env 配置下的值，会随环境变化，**这个数字不能被当成 `Native RAG` 的维度**。`Native RAG` 的维度我们在后面讲优化时另有交代。这里我们关注的是"分发链路怎么走"这个通用骨架，而不是某条链路的具体参数。

### 2.4 变体验证：没有 Bearer Token 会怎样

&emsp;&emsp;要真正确认"鉴权在哪一层拦截"，最直接的办法是走一次**失败路径**。如果一个请求不带 `Bearer Token` 直接调用，会发生什么？答案是：请求在 `apps/api` 的鉴权环节就被拦下，直接返回 `401`，根本走不到分发和模块处理这两步。这正好从反面印证了鉴权是整条链路的第一道闸——它挡在最前面，任何未经身份验证的请求都无法触达任何模块。这一点你可以对照 `trace-01` 的 `seq 1` 失败分支和另一条 `trace-02` 日志来确认。

&emsp;&emsp;讲到这里，我想请你留意刚才这几步我们是怎么走过来的。我们不是凭直觉猜"分发链路大概是这样"，而是对着 `trace-01` 这条**真实记录的日志逐条核对**——每一个环节都有对应的 `file:line` 证据或真实的响应记录。这就是我们在开篇提到的三层验证阶梯的第三层：**运行时验证**。前面我们讲的架构边界（"分发层不碰模块库"）是一个声明，源码里的 `auth.ts:27-31`、`gateway/src/index.ts:103-108` 是对这个声明的源码坐实，而 `trace-01` 里真实的流转记录，则是对"代码写的和代码实际怎么跑一致"的运行时验证。<font color=red>一个技术结论只有同时经得起"源码坐实"和"运行时验证"，我们才敢把它当真——这个习惯，比记住任何一个具体的架构细节都更值钱。</font>

&emsp;&emsp;至此，第二章的目标也达成了：你已经能对着一条真实的 `trace` 日志，说清一个带 `Bearer Token` 的请求怎么一步步走完"鉴权 → 分发 → 模块处理 → 检索命中"，也亲眼见过不带 token 会在第一道闸就被 `401` 拦下。我们已经看清楚了请求进入平台后的通用流转骨架。但你可能已经憋着一个问题：刚才 `seq 8` 那句"query 向量化后做相似度检索"，在 `Native RAG` 内部到底是怎么实现的？它真的只是"向量检索"这么简单吗？带着这个问题，我们先进入第三章，从一个所有链路通用的两层检索架构看起，再到第四章把 `Native RAG` 的内部实现彻底拆开。

## <center>第三章：检索架构总览——召回层与全局 ReRank</center>

&emsp;&emsp;前面我们已经从黑盒视角看过三条链路各自长什么样，现在往深走一层，看清一个**所有链路都遵循的通用检索架构**。先摆出一个关键认知：企业级 RAG 的检索**不是"一步到位"，而是分两层的**——先在各条链路内部做"召回"（粗排、尽量多捞候选），再在平台层做统一的"重排"（精排、定最终顺序）。三条链路的差异集中在召回层，而重排层是平台统一的。理解了这个两层骨架，你才能看清第四章要深挖的 `Native RAG` 三路 RRF，到底站在整个检索链路的哪个位置。

&emsp;&emsp;在拆开这两层之前，我们先把这条"整个检索链路"完整画出来——一次查询从进入平台到返回带依据的答案，一共要走七站。这张图是本课从这里到第五章的**主线地图**：后面每一节，其实都是在给这条主线上的某一站做放大。你可以边学边回来对照，随时确认当前讲的内容落在哪一站。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174855535.png" width=85%></div>

<!-- ILLUSTRATION: type=architecture | content=完整三层检索查询流程图，横向从左到右展示"查询入口 / 统一分发层 → 召回层（模块内粗排 / per-engine retrieval）→ 候选汇总 + 全局 ReRank（平台层精排）"。第 1 层左侧是"用户 Query"，流向"apps/api 鉴权 + 归一化 + 分发"，旁边标注"不直连模块数据库"。第 2 层是三条并行召回链路：Nano Brain=keyword + vector 两路 RRF；Traditional RAG=tsvector + trigram + vector 三路 RRF，并用橙色高亮标注"本课主角"；GraphRAG=LightRAG local / global / hybrid / mix。三条召回箭头汇入第 3 层的"候选引用池 citations[]"，再进入"qwen3-rerank 统一打分"，最后输出"TopN 证据片段 + 答案引用"。图中保留两个参数徽标"rerankTopN=5"、"rerankMinScore=0.4"，并用醒目的警示框标注"RRF 分数 ≠ 最终相关性；全局相关性以 qwen3-rerank 打分为准"。整体白底、蓝绿橙三色分层、技术课件风格、中文清晰。 -->

### 3.1 两层检索模型：召回（粗排）与重排（精排）

&emsp;&emsp;标准 RAG 的证据检索分两个阶段，它们的位置、目标、参数都不一样，绝不能混为一谈。下面这张表把两层的差异一次看清。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>召回层（粗排）与重排层（精排）两层检索模型</font></p>
<div class="center">

| 维度 | 召回层（粗排 / retrieval） | 重排层（精排 / rerank） |
|---|---|---|
| 位置 | **模块内**，每条链路自己实现 | **平台层**，统一 reranker |
| 目标 | 覆盖率——别把正确答案漏掉 | 精度——把最相关的顶到前面 |
| 实现 | Naive=三路 RRF / GraphRAG=LightRAG / Nano=两路 RRF | 平台统一 `qwen3-rerank`（DASHSCOPE reranker）|
| top-k | 每条链路召回多少候选（模块 `limit`）| 重排后最终保留几条（`rerankTopN`，默认 5）|
| 阈值 | 默认不传阈值时直接取 topN；若传 `min_score`，则作用于归一化后的 RRF 相对分（`rrf/max_rrf`）| `rerankMinScore`（0.4，按相关性分过滤）|
| 作用域 | **per-engine**（三链路召回策略本就不同）| **全局**（对三路汇总候选统一精排）|

</div>

<p align="center"><font face="黑体" size=4>完整三层检索查询流程图</font></p>

&emsp;&emsp;一句话记住这两层：召回层是"各链路各显神通、尽量多捞"，重排层是"平台统一裁判、精挑细选"。本课主角 `Native RAG` 的三路 RRF，属于**召回层**的实现——它负责"多捞得准"，而最终的精排是交给平台重排层的。

### 3.2 三条链路各自的召回流程

&emsp;&emsp;召回层是 per-engine 的，所以三条链路在这一层各走各的路。`Nano Brain` 走的是 `keyword`（ILIKE 关键词）加 `vector`（语义）**两路 RRF 融合**。`Traditional RAG`（本课主角）走的是 `tsvector` 全文、`trigram` 字面、`vector` 语义**三路 RRF 融合**——它是本课第四章要彻底拆开的对象。`GraphRAG` 则完全不同，它把召回委托给 `LightRAG` 的 `local` / `global` / `hybrid` / `mix` 四种图检索模式，返回的是图谱证据文本块、**没有相关性分数**（也就是本节前面对比表里的"融合路数 0"）。

**三条链路的实现策略对比**

&emsp;&emsp;平台目前挂着三条 RAG 链路，它们不是同一套代码的三份拷贝，而是**技术栈、检索策略、复杂度落点都不同**的三个独立实现。下面这张对比表把三者放在一起，你会立刻看到它们最有意思的一个差异——检索融合的路数居然是 0、2、3 三个不同的数字。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>三条 RAG 链路的实现策略对比</font></p>
<div class="center">

| 维度 | Nano Brain（TS） | Traditional RAG（Python） | GraphRAG（Python） |
|---|---|---|---|
| 拟人比喻 | 私人知识管家 | 档案室管理员 | 关系网情报员 |
| chunk 策略 | 按空行贪心拼到 1200 字符，无 overlap | 按段落聚合到 1200 字符，超长段落再叠 160 overlap 滑窗 | 无 chunk 概念，整段交给 LightRAG |
| 检索融合路数 | 2 路（关键词 + 向量）真 RRF | **3 路（tsvector + trigram + 向量）真 RRF** | 0 路——无相关性分数，只返文本证据块 |
| 独有机制 | Dream 自治整理（4 阶段 + 并发锁） | MinerU PDF 解析 + 表格独立检索路径 | LightRAG 四种图检索模式 |
| 复杂度落点 | 下沉到"后台维护" | 下沉到"检索融合的可解释性" | 外包给专业图库，自己只做编排 |

</div>

> &emsp;注：表中 `MinerU`、`LightRAG` 分别是 `Traditional RAG`、`GraphRAG` 各自依赖的第三方 PDF 解析 / 图检索框架，本课不展开（`MinerU` 会在 4.7 简单提及）。

&emsp;&emsp;这里的融合路数递增不是随意设计的，它恰好对应"证据类型复杂度"的递增。`GraphRAG` 处理的是图谱证据——实体和关系本身就是结构化的，图节点的连通度已经隐含了重要性，因此它**根本不需要再叠加相关性排序**，检索结果直接就是证据文本块。`Nano Brain` 处理的是纯 Markdown 笔记，信号相对干净，用关键词和语义两路就够了。而 `Traditional RAG` 面对的是企业文档——里面有自然语言、有表格、有 PDF、有精确的合同编号和产品型号，信号最杂，所以才需要三路一起上、再用 `RRF` 融合。下面这张图把三者的复杂度落点画在一起，帮你建立整体感。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174853603.png" width=50%></div>

<!-- ILLUSTRATION: type=comparison | content=三链路家族对比图。横向排列三个链路卡片：GraphRAG(融合路数0，标签"证据即结构，无需排序")、Nano Brain(融合路数2，标签"笔记信号干净，关键词+语义")、Traditional RAG(融合路数3，标签"企业文档信号最杂，三路RRF")。用一条从左到右递增的箭头轴串联，轴标注"证据类型复杂度递增"。每个卡片下方标注复杂度落点：外包给库 / 后台维护 / 检索可解释性。突出 Traditional RAG 卡片（本课主角）用高亮边框。 -->

**逐条看三条链路的黑盒骨架**

&emsp;&emsp;光看一张对比图还不够直观，我们再把三条链路各自拎出来，各看一眼它从输入到输出的**黑盒骨架**。这里只描摹轮廓——本课后面会把其中的 `Native RAG` 一条彻底拆开，另外两条你只需建立一个"它大概怎么工作"的整体印象就够了。

&emsp;&emsp;先看 `Nano Brain`（私人知识管家）：它吃进的是 Markdown 笔记和 wiki 风格页面，内部按空行把正文贪心拼成 1200 字符的 chunk（不留 overlap）、生成向量，检索时走 `keyword`（ILIKE 关键词）加 `vector`（语义）**两路 RRF 融合**（`RRF_K=60`），最后返回带页面互链的结果。它最有特色的地方不在检索——检索其实很朴素，而在后台有一套 `Dream` 自治整理任务（抽链接、刷 embedding、健康检查等四个阶段）默默维护索引不腐化。一句话概括：它把复杂度下沉到了"后台维护"。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174828345.png" width=80%></div>

<!-- ILLUSTRATION: type=architecture | content=Nano Brain 黑盒架构图（简版，三段式布局，与另两条链路图保持一致）。左"输入"框：Markdown 笔记 / wiki 页面（sources→pages→chunks 三层）。中"核心处理"框：空行贪心切 chunk（1200 字符、无 overlap）→ embedding → keyword + vector 两路 RRF 融合（RRF_K=60）；核心框下方挂一个虚线小框标注"Dream 后台自治整理（4 phase）"。右"输出"框：带页面互链的检索结果。底部标注"复杂度落点：后台维护"。整体用淡色调，区别于主角 Traditional RAG。 -->

&emsp;&emsp;再看本课的主角 `Traditional RAG`（档案室管理员）：它要应付企业里五花八门的格式（`docx` / `pdf` / `csv` / `xlsx` / `txt` 等），用一套双层状态机管理处理流程，文本切 chunk、生成 embedding，检索时走 `tsvector` 关键词、`trigram` 字面、`vector` 语义**三路 RRF 融合**，表格类还有一条完全独立的结构化查询路径。它把复杂度下沉到了"检索融合的可解释性"——每一路的 rank 和分数都透传出来，不做黑盒。这条链路的每一个环节，正是本课第三、四章要逐层拆开的对象。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260706125438928.png" width=80%></div>

<!-- ILLUSTRATION: type=architecture | content=Traditional RAG 黑盒架构图（简版，本课主角，用高亮边框强调）。左"输入"框：企业多格式文档（docx / pdf / csv / xlsx / txt）。中"核心处理"框：双层状态机（document 级 / job 级）→ 切 chunk（段落聚合 1200 字符、超长再 overlap 滑窗）→ embedding（超维 MRL 截断）→ **三路并列召回，务必画成三条清晰独立的支路**：①关键词 keyword（tsvector 全文检索 plainto_tsquery simple）、②字面 literal（ILIKE 子串匹配，pg_trgm 索引加速）、③向量 vector（embedding 余弦 <=>）；三条支路各自产出候选，**共同汇入一个醒目的 RRF 融合节点（RRF_K=60，公式 1/(60+rank)）做综合排序**，再经全局归一化（rrf/max_rrf）；旁挂一个虚线小框标注"表格独立结构化查询路径"。三路用三种不同颜色区分，中间 RRF 汇合节点最突出。右"输出"框：带证据 chunk、且每路 rank/score 透传的可解释结果。底部标注"复杂度落点：检索融合可解释性"。高亮边框标明这是本课主角。 -->

&emsp;&emsp;最后看 `GraphRAG`（关系网情报员）：它把文档整段交给 `LightRAG` 库，由库来抽取实体与关系、构建图谱（图存储用 PostgreSQL 的 Apache AGE 扩展），检索时走 `local` / `global` / `hybrid` / `mix` 四种图检索模式——但这些图算法**完全是 `LightRAG` 库内置的、本项目零自研**，自己只负责多租户隔离、并发编排和权限过滤。也正因为把能力外包给了专业图库，它的检索结果天生是"证据文本块"、而不是打分排序的列表（融合路数为 `0`）。一句话概括：它把复杂度外包给了专业图库。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174824585.png" width=80%></div>

<!-- ILLUSTRATION: type=architecture | content=GraphRAG 黑盒架构图（简版）。左"输入"框：文档（文本 / 文件两条 ingest 路径）。中"核心处理"框：整段文本交给 LightRAG 库 → 抽取实体/关系图（图存储 = PostgreSQL Apache AGE 扩展）→ local/global/hybrid/mix 四种图检索模式（醒目标注"库内置算法，本项目零自研"）；核心框旁标注本项目只做"多租户隔离 + 并发编排 + 权限过滤"。右"输出"框：图谱证据文本块（标注"融合路数 0、无相关性分数"）。底部标注"复杂度落点：外包专业图库"。淡色调，区别于主角。 -->

&emsp;&emsp;<font color=red>这三条链路的召回分数天然不可比</font>——`Naive` 与 `Nano` 都用 RRF，但路数、候选池、语料各不相同（`Naive` 的 RRF 分在 `0.01~0.05` 量级，`Nano` 的两路 RRF 又是另一套量级），彼此不能直接比大小；`GraphRAG` 返回的则是图节点连通度这类图谱/文本证据，本身没有一个可比的相关性分。正因为不可比，才需要下一节讲的全局重排层来做统一裁判。

**一个值得停下来想的问题**

&emsp;&emsp;讲到这里，我们抛出一个问题请你想一想：既然三条链路都叫 RAG，为什么它们的融合路数不统一成同一个数字，反而是 0、2、3 三个不同的值？这背后其实藏着一个很重要的架构观念——**同样是"企业级"，复杂度可以合法地分布在不同的地方**。`Nano Brain` 把复杂度藏进后台的 Dream 维护任务，`Traditional RAG` 把复杂度藏进检索融合的可解释性设计，`GraphRAG` 干脆把复杂度外包给专业图库、自己只做编排。它们没有谁对谁错，只有"复杂度放在哪里最划算"的不同取舍。

### 3.3 全局 ReRank 策略

&emsp;&emsp;三条链路召回的候选汇总到平台层之后，由一个统一的 reranker 重新打分排序，这就是"全局 ReRank"。它的实现是 `platform-store` 里的 `rerankCitations`，调用 DASHSCOPE 的 `qwen3-rerank` 模型，把整个候选列表一次性交给 reranker，模型为每条候选返回一个 `relevance_score` 相关性分。有两个关键旋钮：`rerankTopN`（默认 `5`，控制最终展示几条）和 `rerankMinScore`（`0.4`，相关性分低于它的候选会被过滤掉）；这两个参数的取值走"数据库 → 环境变量 → 默认值"的三层优先级读取。

&emsp;&emsp;为什么重排一定要放在平台层、而不是各链路自己做？答案就在上一节那句"三条链路召回分数不可比"——`Naive` 的 `RRF` 分只有 `0.03` 这种小数量级、`Nano` 的两路 RRF 又是另一套量级、`GraphRAG` 干脆没有可比的相关性分（返回的是图谱证据文本），直接放一起比毫无意义。只有在平台层用一个统一的 reranker 对汇总后的候选重新打分，才能得到一把跨链路可比的尺子。<font color=red>这也解释了一件重要的事：`RRF` 分数不能直接当作最终相关性——它只是召回层的粗排名次，真正的精排要交给重排层。</font>

> **【本节的诚实边界】**：重排层目前只覆盖"全域问答"这一条检索路径；"场景问答"和"召回验证"走的是各引擎直接按 `limit` 取 topN、**不经过 rerank**。所以调整 `rerankTopN` / `rerankMinScore` 只会影响全域问答的结果。这是当前实现的覆盖边界，不要理解成"平台所有检索都过一遍全局重排"。

&emsp;&emsp;有了"召回 + 重排"这个两层坐标系，接下来我们就把镜头对准本课的主角——`Native RAG`（`traditional-rag`）这条链路的**召回层实现**，看它的三路 RRF 到底每一步是怎么落地的。

## <center>第四章：Native RAG 实现拆解</center>

&emsp;&emsp;前两章我们一直站在平台的高度看全景和分发骨架，从这一章开始，我们进入 `Native RAG`（`traditional-rag` 模块）内部。这里不再使用 mock 语料或内存版算法，而是直接连接 `traditional_rag_db`，用真实表、真实 SQL、真实 `chunk` 数据，把 `modules/traditional-rag/src/traditional_rag/core/search.py` 的检索逻辑跑出来。

&emsp;&emsp;这一章的主线只有一条：**从数据库证据理解 Native RAG**。文档怎么落库、状态机怎么推进、chunk 和 embedding 存在哪、三路召回 SQL 分别查什么、RRF 怎么把三路候选融合成最终排序，全部都用可执行代码展示。你看到的每个结果都来自本机 PostgreSQL，而不是手写样例。

> **【关于本章代码】**：本章代码是 Notebook 友好的“源码同构 SQL 演示版”。它不直接 `import traditional_rag.core.search`，因为课件的 `company-brain` kernel 只安装了 Notebook 依赖，不一定安装 `traditional-rag` 模块自身的 `pydantic-settings`、`psycopg_pool` 等运行依赖；但 SQL、参数顺序、RRF 公式、过滤逻辑都逐行对齐 `search.py`。这样做的好处是：学员只要 PostgreSQL 和 `.env` 就能跑，不会被 Python 包环境卡住。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174835320.png" width=50%></div>

<!-- ILLUSTRATION: type=flowchart | content=Native RAG 内部流程图。主干从左到右：文档 upload → 双层状态机(document级/job级并列小图标) → chunk切分(1200字符+160overlap) → embedding(超维MRL截断+归一化) → 三路召回(keyword全文/literal字面/vector向量三条并列支路) → RRF_K=60 融合 → 全局归一化 → TopK输出。三路召回处用三种颜色区分，融合节点用醒目图标标注"本章核心"。 -->

&emsp;&emsp;先准备一个本章通用的数据库工具。它做三件事：从课件同目录 `.env` 读取连接串；提供 `run_sql` 执行只读查询；提供 `print_rows` 把结果打印成适合课堂展示的表格。后面所有 cell 都复用这套工具。

In [8]:
import os
import re
import json
import math
import time
import urllib.request
import urllib.error
from pathlib import Path

import psycopg
from psycopg.rows import dict_row
from dotenv import load_dotenv

# 课件目录下已经复制了一份 .env；Notebook 从课件目录启动时，直接加载相对路径即可。
load_dotenv(Path.cwd() / ".env", override=False)

def get_db_url(name: str) -> str:
    """读取数据库连接串。缺失时给出清晰错误，避免裸 KeyError。"""
    value = os.getenv(name)
    if not value:
        raise RuntimeError(f"缺少环境变量 {name}，请确认课件目录 .env 已配置。")
    return value

def run_sql(sql: str, params=()):
    """执行一条 SQL 并返回 dict row 列表。只用于课堂查询，不做写入操作。"""
    with psycopg.connect(get_db_url("TRADITIONAL_RAG_DATABASE_URL"), row_factory=dict_row) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall()

def print_rows(rows, columns=None, limit=8):
    """把查询结果打印成轻量表格，长文本截断到 80 字符，方便 Notebook 阅读。"""
    rows = list(rows)[:limit]
    if not rows:
        print("(无结果)")
        return
    columns = columns or list(rows[0].keys())
    print(" | ".join(columns))
    print("-" * 100)
    for row in rows:
        values = []
        for col in columns:
            text = str(row.get(col, ""))
            text = text.replace("\n", " ")
            values.append(text[:80])
        print(" | ".join(values))

&emsp;&emsp;在写第一行代码之前，我们先把整章要用到的工具和数据准备好。下面这个环境准备 cell 做两件事：定义一个确定性的 mock embedding 函数（用字符 bigram 词袋来模拟向量），以及准备一批 mock 的企业文档语料（考勤、报销、合同、财务等制度，其中特意放了几条含精确编号的合同文档）。这两样东西会被本章后面所有代码复用。


In [9]:

# ============================================================
# 教学最小复现，非 ff-companybrain 真实源码
# 真实实现见 modules/traditional-rag/src/traditional_rag/core/search.py
# ============================================================
import re
import math
from collections import defaultdict

def mock_embed(text: str) -> dict:
    """把文本编码成确定性的"字符 bigram 词袋"向量，教学用，替代真实 embedding API。
    设计意图：向量由相邻两字组成的 bigram 频率构成，语义相近的文本向量也相近；
    但精确编号（如 ZF-2024-0088）里的公共片段（如 -2024-00）会和别的编号高度重合，
    使向量对相近编号的区分度很低——这正是后面 M2 要演示的：向量能靠字符重叠勉强排第一，但优势脆弱。
    真实项目用 qwen3-embedding-8b via OpenRouter，维度 1024。
    Args:
        text: 待编码文本
    Returns:
        dict[bigram -> 权重]，已 L2 归一化为单位向量
    """
    t = text.replace(" ", "")  # 去空格后按相邻字符取 bigram
    bag = defaultdict(float)
    for i in range(len(t) - 1):
        bag[t[i:i + 2]] += 1.0  # 统计每个 bigram 出现次数
    norm = math.sqrt(sum(v * v for v in bag.values())) or 1.0  # L2 范数，空文本兜底为 1
    return {k: v / norm for k, v in bag.items()}  # 归一化成单位向量

def cosine(a: dict, b: dict) -> float:
    """两个已归一化 bigram 词袋向量的余弦相似度（单位向量点积即余弦）。"""
    # 遍历较短的一侧，累加公共 bigram 的权重乘积，减少无谓遍历
    keys, other = (a, b) if len(a) < len(b) else (b, a)
    return sum(w * other.get(k, 0.0) for k, w in keys.items())

# mock 企业文档语料：每条是一个 chunk（考勤/报销/合同/财务/员工手册）
# 特意安排：ct-02 编号只与目标差末两位、fin-01 编号后缀与目标完全相同，作为向量干扰项
CORPUS = {
    "att-01": "考勤制度：员工每月迟到累计超过三次，扣除当月绩效的百分之十。迟到以打卡记录为准。",
    "exp-01": "报销制度：差旅费报销需提供发票原件，单张发票金额超过五千元需部门经理审批。",
    "ct-02":  "合同管理制度：本服务合同编号 ZF-2024-0091，签订日期为二零二四年五月，合同金额及验收标准以附件为准。",
    "ct-01":  "合同管理制度：本采购合同编号 ZF-2024-0088，签订日期为二零二四年三月，合同金额及付款条款以附件为准。",
    "fin-01": "财务制度：本次采购对应发票编号 FP-2024-0088，发票金额与合同金额需一致方可入账。",
    "hr-01":  "员工手册：公司规章制度涵盖考勤、请假、绩效与晋升等方面，全体员工须遵守。",
}
# 预先算好每个 chunk 的向量，检索时直接复用
CORPUS_VEC = {cid: mock_embed(text) for cid, text in CORPUS.items()}
print(f"语料载入完成：{len(CORPUS)} 个 chunk，向量已预计算")



语料载入完成：6 个 chunk，向量已预计算


&emsp;&emsp;这段准备代码里最值得留意的是 `mock_embed` 的设计意图。真实项目用的是 `qwen3-embedding-8b` 这样的大模型来生成语义向量，我们在教学里用不起也不该引入网络依赖，于是用"字符 bigram 词袋"来做一个**确定性的替身**——两段文字共享的相邻字符越多，它们的向量就越相似。关键在于，我们特意让语料里出现了几条"编号高度接近"的干扰文档，为的就是在后面演示"向量为什么对精确编号查询不敏感"。这些设计的用意，等跑到 M2 那一步你会看得非常清楚。


&emsp;&emsp;这一段就是后面所有“查库展示”的基础。注意它没有引入项目模块代码，只依赖 `psycopg` 和 `.env`。这样 Notebook 运行环境和生产服务运行环境解耦，但查询对象仍然是同一个真实数据库。

### 4.1 HTTP 接口与调用路径

&emsp;&emsp;在一头扎进文档处理和检索算法之前，我们先站在"调用方"的角度问一个最朴素的问题：当上层应用要用 `Native RAG` 检索一次，这个请求到底是怎么一路流到核心检索函数的？把这条调用链看清楚，后面每一节（文档处理、`chunk`、`embedding`、三路 RRF）才知道自己站在整条链路的哪个位置。

&emsp;&emsp;整条链路一共三站，我们在 1.3 讲"统一分发机制"时已经见过它的骨架，这里把它落到 `Native RAG` 的真实代码上。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174830252.png" width=55%></div>

<!-- ILLUSTRATION: type=flowchart | content=Native RAG 一次检索请求的三站调用链，从左到右横向流程图。第一站：上层应用/前端 → 统一 API 网关(apps/api, Bun+Hono)，节点标注"只鉴权+转发 /traditional/*"。第二站：traditional-rag 模块 FastAPI，节点标注"POST /traditional/search + require_internal_user 鉴权依赖"。第三站：asyncio.to_thread 投线程池 → core/search.py 的 search()，节点标注"同步核心投线程池防阻塞"。三站之间用箭头连接，第一站到第二站的箭头上标"携带鉴权后的用户上下文"。 -->

&emsp;&emsp;**第一站：统一 API 网关只鉴权、只转发。** 所有走向 `Native RAG` 的流量，第一站都落在平台统一 API 网关（`apps/api`，`Bun + Hono`）。网关并不理解检索逻辑，它只做一件事——把 `/traditional/*` 这个命名空间下的所有请求，原样转发给 `traditional-rag` 模块：

```typescript
// apps/api/src/routes/rag.ts:34
// Python RAG services reserve their own HTTP namespace behind the unified API.
ragRouter.all('/traditional/*', proxyModule('traditional-rag', 'Traditional RAG 模块调用失败'));
```

&emsp;&emsp;这里 `ragRouter.all` 表示不限请求方法都转发，`proxyModule` 会带着**鉴权后的用户上下文**（`user: ctx`）把请求代理到模块。这正是 1.3 讲过的"只鉴权分发，不碰模块库"原则的代码落地——网关这一层完全不知道什么是 RRF、什么是 chunk，它只认命名空间和身份。

&emsp;&emsp;**第二站：模块 FastAPI 路由 + 鉴权依赖。** 请求到达 `traditional-rag` 模块后，落在它自己的 `FastAPI` 路由上。模块用 `APIRouter(prefix="/traditional")` 声明命名空间，检索入口是一个 `POST /traditional/search`：

In [ ]:
# modules/traditional-rag/src/traditional_rag/http/routes/search.py:11、24-28
router = APIRouter(prefix="/traditional")

@router.post("/search")
async def search_route(body: QueryRequest, user: UserContext = Depends(require_internal_user)) -> dict:
    try:
        # C2：同步 search 投线程池，避免同步 psycopg/urllib embedding 阻塞事件循环。
        return await asyncio.to_thread(search, user, body.model_dump(exclude_none=True))
    except Exception as error:
        raise_http_error(error)

&emsp;&emsp;这段短短的路由函数里藏着三个关键点。第一，`user: UserContext = Depends(require_internal_user)` 是 `FastAPI` 的依赖注入——请求进入函数体之前，`require_internal_user` 会先校验 `Bearer Token` 并解析出用户上下文，鉴权不通过就直接被挡在门外（我们在 2.4 已经亲眼见过没有 `Token` 会怎样）。第二，真正干活的是 `core.search` 里的 `search()` 函数，它被 `asyncio.to_thread` 包了一层。第三，任何异常都统一交给 `raise_http_error` 转换成规范的 HTTP 错误响应，而不是把原始堆栈暴露给调用方。

&emsp;&emsp;**第三站：同步核心为什么要投进线程池。** 这里最值得停下来想一秒的，是 `asyncio.to_thread(search, ...)` 这层包装。`search()` 内部用的是**同步**的 `psycopg`（数据库驱动）和**同步**的 embedding HTTP 调用；如果直接在 `async` 路由里调用它，这些同步阻塞操作会卡住整个事件循环，导致同一时刻其他请求全部排队等待。把它投进线程池后，事件循环就能在等待数据库、等待 embedding 返回的间隙里去处理别的请求。这是一条典型的"同步核心 + 异步外壳"的工程处理——它是本课 <font color=red>5.1 并发优化</font>的第一条线索，我们会在那一节把整个模块的并发设计展开讲透。

&emsp;&emsp;**请求契约：`QueryRequest`。** 最后看一眼这个接口到底"吃"什么参数。入口用 `pydantic` 的 `QueryRequest` 定义请求体，`FastAPI` 会在进函数前自动校验，不合法的请求根本到不了 `search()`：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Native RAG 检索接口 QueryRequest 请求契约</font></p>
<div class="center">

| 字段 | 类型 | 约束 | 含义 |
|------|------|------|------|
| `query` | `str` | 必填，长度 1-500 | 检索的查询文本 |
| `limit` | `int` | 默认 10，范围 1-30 | 返回结果条数上限 |
| `min_score` | `float`（可选） | 0-1，默认不限 | 最低分数阈值，指定后过滤掉低于该分的结果 |
| `source_id` | `str`（可选） | — | 限定在单一知识源内检索 |
| `source_ids` | `list[str]`（可选） | — | 限定在多个知识源内检索 |
| `document_id` | `str`（可选） | — | 限定在单篇文档内检索 |
| `file_types` | `list[str]`（可选） | — | 按文件类型过滤检索范围 |

</div>

&emsp;&emsp;这张表就是 `Native RAG` 对外的完整检索契约。其中 `source_id` / `source_ids` 用于把检索范围限定在指定知识源内，与 1.5 讲的权限行级过滤配合，共同决定"这次检索能看到哪些语料"。理解了入口长什么样、契约收什么参数，我们就可以正式走进模块内部——从它拿到一篇文档开始，一步步看这篇文档是怎么变成可被检索的向量的。

### 4.2 文档格式与处理分工

&emsp;&emsp;`Native RAG` 面对的是企业里五花八门的文档格式，它的第一个设计决策就是**按格式分工处理**。文本类的文档（`docx`、`markdown`、`txt`、`html`、`json`）走"切 chunk + 生成 embedding"这条主链路；`csv` 和 `xlsx` 这类表格文件走独立的"表格抽取"路径，把表结构和行数据单独存起来；而 `image`、`audio`、`video` 这类当前无法真正解析的媒体文件，则走一条特殊的诚实退化分支（见 `documents.py:606-646`）：对应的处理任务（`job`）会被标记为 `failed`、阶段记为 `awaiting_media_extractor`，而文档（`document`）本身**并不会被粗暴地标成 `failed`**，而是在 metadata 里如实记录一个"已接收原始资产、等待真实 OCR/ASR/视频解析器"的状态。

&emsp;&emsp;这里有一个很值得学习的工程态度：面对处理不了的格式，`Native RAG` 选择**诚实退化**——把处理任务明确标 `failed`、并在文档 metadata 里如实记录"等待媒体抽取器"，而不是假装处理成功却悄悄产出一个空结果。这种"能力缺失就如实报错、绝不静默兜底"的设计，在企业级系统里极其重要，因为静默的假成功往往比明确的失败更难排查。我们后面在讲 PDF 处理时，会看到这个原则的又一次体现。

### 4.3 job 双层状态机

&emsp;&emsp;文档从"上传"到"可检索"要经历一系列状态流转，`Native RAG` 在这里用了一个容易被搞混、但设计上很清晰的机制——**两套不同粒度的状态机并存**。一套是 `document` 级的状态机，描述整个文档的宏观状态；另一套是 `job` 级的状态机，描述后台处理任务的细粒度阶段。下面这张图把两套状态机并列画出来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174820026.png" width=50%></div>

<!-- ILLUSTRATION: type=architecture | content=双层状态机对比图。左侧 document 级状态机：uploaded → processing → ready，旁支 failed / archived，五个状态。右侧 job 级状态机：uploaded → parsing → chunking → embedding → ready，旁支 failed，六个状态且注明"无 processing 态"。两套状态机用不同底色区分，中间画一条对应关系虚线：document 的 processing 阶段内部展开成 job 的 parsing→chunking→embedding 三个细粒度阶段。底部标注"两套不同粒度，容易搞混"。 -->

&emsp;&emsp;具体来说，`document` 级的状态是 `{uploaded, processing, ready, failed, archived}` 五种（见 `migrations.py:59` 的 `status CHECK` 约束）；而 `job` 级的状态是 `{uploaded, parsing, chunking, embedding, ready, failed}` 六种（见 `migrations.py:108`），注意 **`job` 级没有 `processing` 这个态**——文档级的 `processing` 在任务级被展开成了 `parsing → chunking → embedding` 三个更细的阶段。一旦处理失败，系统会走 `_mark_job_failed`（见 `documents.py:463-486`）把结构化的错误信息（`{error, message}`）写进 job 的 `error` 字段（`jsonb` 类型），而不是简单地扔一个异常了事。我们同样把这条状态机的关键字段从真实日志（`trace-05`）里打印出来看看。

In [10]:
# 打印 trace-05 真实日志的状态机关键字段，看清双层状态机怎么流转
# 数据来源：PKP/03-traces/trace-05.json（真实运行记录）
trace_05_steps = [
    {"seq": 1, "阶段": "员工上传", "document 状态": "uploaded", "job 状态": "uploaded",
     "说明": "document 创建，同时建 job 记录"},
    {"seq": 2, "阶段": "后台开始处理", "document 状态": "processing", "job 状态": "parsing",
     "说明": "processing 是 document 级瞬态，job 进入 parsing"},
    {"seq": 3, "阶段": "切块+向量化", "document 状态": "processing", "job 状态": "chunking→embedding→ready",
     "说明": "job 逐阶段推进，chunks 入库(embedding not null)"},
    {"seq": 4, "阶段": "全部就绪", "document 状态": "ready", "job 状态": "ready",
     "说明": "两套状态机都到达 ready"},
    {"seq": 5, "阶段": "失败分支", "document 状态": "failed", "job 状态": "failed",
     "说明": "失败旁支(与全部就绪互斥)：documents.py:463-486 _mark_job_failed 写结构化 error"},
]

# 逐条打印，对照两套状态机的粒度差异
print(f"{'阶段':<12}{'document 级':<14}{'job 级':<28}说明")
print("-" * 90)
for s in trace_05_steps:
    print(f"{s['阶段']:<12}{s['document 状态']:<14}{s['job 状态']:<28}{s['说明']}")

阶段          document 级    job 级                       说明
------------------------------------------------------------------------------------------
员工上传        uploaded      uploaded                    document 创建，同时建 job 记录
后台开始处理      processing    parsing                     processing 是 document 级瞬态，job 进入 parsing
切块+向量化      processing    chunking→embedding→ready    job 逐阶段推进，chunks 入库(embedding not null)
全部就绪        ready         ready                       两套状态机都到达 ready
失败分支        failed        failed                      失败旁支(与全部就绪互斥)：documents.py:463-486 _mark_job_failed 写结构化 error


<!-- cell-type: verify -->

&emsp;&emsp;打印出来后，两套状态机的粒度差异一目了然：`document` 级的 `processing` 是一个"正在处理中"的粗粒度瞬态，而 `job` 级把这个瞬态拆成了 `parsing → chunking → embedding` 三个能精确定位到"卡在哪一步"的细粒度阶段。理解这个双层设计，直接决定了你能不能快速排查一类高频问题。

> **【踩坑预警】**：当你遇到"文档明明处理完了，但就是检索不到"时，请**先查两件事**。第一，看 `job` 状态机停在哪个阶段——如果停在 `parsing` 或 `chunking`，说明处理中途出问题了，去看 job 的 `error` 字段；如果 `job` 是 `ready` 但检索不到，问题可能在别处。第二，直接查 `chunk` 表里到底有没有真实生成 chunk——有些情况（比如后面要讲的 PDF 没解析出正文）下是**真的不产 chunk**，这不是 bug 而是设计。如果你不知道有这两套状态机、上来就怀疑检索逻辑，很可能会在错误的方向上排查很久。判断自己是否踩了这个坑很简单：检索不到时，先把 job 状态和 chunk 数量这两个数打出来看，而不是直接去读检索代码。

### 4.4 chunk 切分策略

&emsp;&emsp;文本类文档在生成向量之前，要先被切成一个个 `chunk`。这里有一个容易想当然的地方：`Native RAG` 的切分（见 `chunks.py:84` 的 `split_into_chunks`，默认参数 `max_chars=1200`、`overlap_chars=160`）**并不是把全文按固定长度等距滑窗**，而是**先按文档解析出的段落（segment）逐个聚合**——把连续的段落累加进当前 chunk，直到接近 `max_chars=1200` 上限才另起一个新 chunk；只有当单个段落本身就超过 1200 字符时，才会对这个超长段落按 `overlap_chars=160` 的重叠做滑窗切片。这个重叠的意义在于：当一个超长段落不得不从中间切开时，160 字符的重叠能保证边界处的语义单元至少在相邻某一个 chunk 里是完整的，不会被生生切断。

&emsp;&emsp;这里有个不起眼但很关键的工程细节：滑窗前进时，如果 `overlap` 设置不当（比如 overlap 大于等于窗口本身能前进的距离），滑窗可能原地打转形成死循环。源码里用了一个 **clamp 保护**（见 `chunks.py:88`，位于 `split_into_chunks` 函数开头）——在切分开始前就把 `overlap` 夹到严格小于窗口大小：`overlap_chars = max(0, min(overlap_chars, max_chars - 1))`，这样每次滑窗推进的步进 `max_chars - overlap_chars` 必然 ≥ 1，从源头上杜绝了原地打转的死循环。这类"防御性的边界处理"是生产代码和玩具代码最典型的区别之一：玩具代码假设输入永远合理，生产代码则必须考虑各种退化情况。

### 4.5 embedding 调用与重试

&emsp;&emsp;chunk 切好之后，就要为每个 chunk 生成向量。`Native RAG` 用的是 openai-compatible 的 embedding 接口，真实项目里用的模型是 `qwen3-embedding-8b`（经 OpenRouter 调用），向量维度是 1024。这里有两个工程要点值得注意。第一，当模型返回的维度超过预期时，代码会做 **MRL 截断加归一化**（`MRL` 即 Matryoshka Representation Learning，套娃式表示学习，`5.5` 节会详解）——也就是把长向量按需截短再重新归一化，让它能塞进数据库预设的向量列。第二，embedding 调用要走网络，天然不可靠，所以代码内置了**最多重试 5 次**的容错机制（网络或超时失败时重试，HTTP 明确返回非 2xx 则直接失败）。

&emsp;&emsp;这两个细节再次印证了 `Native RAG` 的一贯风格：能力上追求诚实（超维就截断而不是报错崩溃），失败上追求明确（网络抖动重试、但真错误绝不假装成功）。理解了 chunk 和 embedding 这两步，我们就集齐了检索所需的全部"原料"——现在每个 chunk 都有了文本和向量，接下来就是本章的重头戏：怎么从这一堆 chunk 里，把和用户查询最相关的几个精准地捞出来。

&emsp;&emsp;在进入三路 RRF 之前，我们花三十秒补两个概念，它们是理解三路检索的前提。第一个是 **`tsvector` 全文检索**：PostgreSQL 把文本预处理成"词元"（token）的集合，查询时把你的问题也转成词元，做集合层面的匹配并用 `ts_rank` 打分——它擅长的是"这段文字里有没有出现这些词"。第二个是 **`trigram` 字面匹配**：它把文本切成连续的三字符片段，配合 `ILIKE` 子串匹配来加速"这段文字里有没有出现这个精确的字符串"的查询——注意它匹配的是**字面子串**，不做任何语义理解。这两者加上我们熟悉的向量语义检索，正好就是三路。

### 4.6 三路 RRF 混合检索融合

&emsp;&emsp;终于到了本章、也是本课最核心的一节。前面我们说 `Native RAG` 的检索是"三路 RRF"，现在我们要把这句话彻底拆开、并亲手把它跑起来。我们会分三步走：先看三路各自是怎么召回的（4.6.1），再看 `RRF` 公式是怎么把三路融合成一个排名的（4.6.2），最后对比"单路 vs 三路"到底差在哪、为什么企业场景值得用三路（4.6.3）。这个顺序是刻意设计的——先理解零件，再理解组装，最后理解为什么要这么组装。

#### 4.6.1 三路各自的召回算法

&emsp;&emsp;企业文档检索要同时应对三种截然不同的查询：有语义相近但用词不同的自然语言问法（"迟到几次扣绩效"），有需要精确匹配的编号型查询（"合同编号 ZF-2024-0088"），也有需要全文匹配的行业术语。任何单一的检索手段都会在其中某一类上吃瘪，所以 `Native RAG` 用了三路各有所长的召回：

&emsp;&emsp;第一路是 **`keyword` 关键词全文召回**，基于 `tsvector` 的 `ts_rank` 打分（对应真实源码 `search.py:100-122`），擅长"命中的关键词越多越相关"。第二路是 **`literal` 字面子串召回**，基于 `trigram` 加速的 `ILIKE` 子串匹配（对应 `search.py:125-151`），对精确编号这类查询最锋利。第三路是 **`vector` 向量语义召回**，用 pgvector 的 cosine 距离、以 `1/(1+distance)` 的形式转成相似度分（对应 `search.py:154-180`），擅长捕捉语义相近但用词不同的问法。三路各自独立召回后，每一路都会给命中的 chunk 排一个从 1 开始的名次（对应 `search.py` 里 `add_candidates` 的 `enumerate(rows, start=1)`，见 `search.py:183-193`）。下面我们把这三路用 MVP 代码实现出来。

In [11]:
# 教学最小复现，非真实源码；真实实现见 search.py:100-193
def tokenize(text: str) -> set:
    """把文本切成 token 集合：字母数字/连字符串（编号）整体保留 + 中文按 2-gram。
    模拟 PostgreSQL to_tsvector('simple') 的分词效果（教学简化，中文不用 split）。
    Args:
        text: 待分词文本
    Returns:
        token 集合，含编号型 token 与中文 2-gram
    """
    # 提取"字母/数字开头的连字符串"，让 ZF-2024-0088 这类编号作为整体 token 保留
    # 注意：这是 MVP 的教学简化。真实源码 keyword 路走 PostgreSQL tsvector('simple') 分词，
    #       并不保证把整串编号当成一个 token，别据此反推真实分词行为
    tokens = set(re.findall(r"[A-Za-z0-9][A-Za-z0-9\-]*", text))
    cjk = re.sub(r"[^一-鿿]", "", text)  # 只保留中文字符
    tokens |= {cjk[i:i + 2] for i in range(len(cjk) - 1)}  # 中文按相邻 2-gram 切
    return tokens

def keyword_recall(query: str) -> list:
    """第一路：tsvector 关键词全文召回（模拟 search.py:100-122 的 ts_rank 打分）。
    query 命中的 token 越多分越高，编号类 token（含数字）权重加倍。
    Args:
        query: 用户查询
    Returns:
        [(chunk_id, rank), ...]，rank 从 1 起，按命中分降序
    """
    q_tokens = tokenize(query)
    scored = []
    for cid, text in CORPUS.items():
        c_tokens = tokenize(text)
        # 命中的 query token 累加权重：编号类 token 权重 2，中文 bigram 权重 1
        score = sum(2.0 if re.search(r"\d", t) else 1.0 for t in q_tokens & c_tokens)
        if score > 0:  # 只保留有命中的 chunk
            scored.append((cid, score))
    scored.sort(key=lambda x: x[1], reverse=True)  # 命中分降序
    return [(cid, rank) for rank, (cid, _) in enumerate(scored, start=1)]  # rank 从 1 起

def literal_recall(query: str) -> list:
    """第二路：trigram 字面子串召回（模拟 search.py:125-151 的 ILIKE 子串匹配）。
    query 原串或去空格串作为子串出现在 chunk 里即命中——对精确编号最锋利。
    Args:
        query: 用户查询
    Returns:
        [(chunk_id, rank), ...]，命中顺序即 rank
    """
    q = query.strip()
    q_compact = q.replace(" ", "")  # 去空格版本，兼容书写差异
    hits = []
    for cid, text in CORPUS.items():
        text_l, text_compact = text.lower(), text.replace(" ", "").lower()
        # 原串或去空格串任一作为子串命中即算命中（大小写无关）
        if q.lower() in text_l or q_compact.lower() in text_compact:
            hits.append(cid)
    return [(cid, rank) for rank, cid in enumerate(hits, start=1)]

def vector_recall(query: str) -> list:
    """第三路：向量 cosine 召回（模拟 search.py:154-180 的 1/(1+distance) 相似度排序）。
    query 向量与每个 chunk 向量算余弦相似度，降序排名。
    Args:
        query: 用户查询
    Returns:
        [(chunk_id, rank), ...]，rank 从 1 起，按相似度降序
    """
    q_vec = mock_embed(query)
    scored = [(cid, cosine(q_vec, vec)) for cid, vec in CORPUS_VEC.items()]
    scored.sort(key=lambda x: x[1], reverse=True)  # 相似度降序
    return [(cid, rank) for rank, (cid, _) in enumerate(scored, start=1)]

<!-- cell-type: core-def -->

&emsp;&emsp;这三个函数分别对应真实源码里的三路召回，我们在 docstring 里都标了对应的 `file:line`。它们有一个统一的返回结构——都是 `[(chunk_id, rank), ...]` 这样的名次列表，`rank` 从 1 开始。这个统一结构是下一步 RRF 融合的前提：RRF 融合根本不关心每一路内部是怎么打分的（有的用 `ts_rank`、有的用相似度），它只关心"这个 chunk 在这一路里排第几名"。

> **【常见误区】**：很多人会把 `trigram`（`pg_trgm`）当成一种"全文检索"，这是不对的。`trigram` 在这里的角色是**给 `ILIKE` 子串匹配加速的索引**，它匹配的是字面字符片段，不做任何分词和语义理解；真正的全文检索靠的是 `to_tsvector`（第一路 `keyword`）。这两者一个管"精确字面"、一个管"词元语义"，职责完全不同。如果把它们混为一谈，你就理解不了为什么"精确编号查询"要靠 `literal` 这一路、而不是 `keyword` 那一路来兜底。

&emsp;&emsp;三路召回函数写好了，我们立刻做一次独立验证，确认每一路都按约定的结构返回。这一步不验证"排名对不对"（那是下一步端到端验证的事），只验证"零件本身的接口是对的"——每一路都返回 `(chunk_id, rank)` 列表、rank 从 1 连续递增。

In [12]:
# Tier 1 独立验证：确认三路召回各自返回结构正确（结构性断言，覆盖三路）
q = "合同编号 ZF-2024-0088"
for name, fn in [("keyword", keyword_recall), ("literal", literal_recall), ("vector", vector_recall)]:
    result = fn(q)
    # 断言1：返回元素是 (chunk_id: str, rank: int) 二元组
    assert all(isinstance(cid, str) and isinstance(rank, int) for cid, rank in result), f"{name} 结构不符"
    # 断言2：rank 从 1 起、连续递增
    assert [r for _, r in result] == list(range(1, len(result) + 1)), f"{name} rank 不连续"
    print(f"[{name:8s}] 返回 {len(result)} 条 → {result}")
print("\nTier 1 通过：三路召回的结构性断言全部成立")

[keyword ] 返回 3 条 → [('ct-01', 1), ('ct-02', 2), ('fin-01', 3)]
[literal ] 返回 1 条 → [('ct-01', 1)]
[vector  ] 返回 6 条 → [('ct-01', 1), ('ct-02', 2), ('fin-01', 3), ('att-01', 4), ('exp-01', 5), ('hr-01', 6)]

Tier 1 通过：三路召回的结构性断言全部成立


<!-- cell-type: verify -->

&emsp;&emsp;这个 Tier 1 验证跑通后，我们就确认了三个"零件"都能正常工作、接口一致。特别值得留意的是 `literal` 那一路对精确编号查询的返回——它应该只命中含 `ZF-2024-0088` 全串的那个 chunk，这正是它"字面精确匹配"的锋利之处。零件验证过了，接下来就该把它们组装起来。

#### 4.6.2 RRF 融合公式与设计动机

&emsp;&emsp;三路各自召回后，我们手里有三个名次列表，问题来了：同一个 chunk 在 `keyword` 里排第 1、在 `vector` 里排第 5、在 `literal` 里根本没出现，我们最终该把它排在哪？这就是 **`RRF`（Reciprocal Rank Fusion，倒数排名融合）** 要解决的问题。它的公式极其简洁：一个 chunk 的最终分数，等于它在各路里名次倒数的和——`rrf_score = Σ 1/(RRF_K + rank)`，其中 `RRF_K` 是一个平滑常量，在本项目里取 `60`。

&emsp;&emsp;现在我们做一次完整的三层验证阶梯演示，把这个 `RRF_K=60` 从"一个声明"验证到底。**第一层是声明层**：我们说"三路 RRF 用 `RRF_K=60`"——这只是一个待验证的 claim。**第二层是源码坐实层**：如果你打开 `search.py`，会在第 13 行看到 `RRF_K = 60` 这个常量定义，在第 196-197 行看到 `rrf_score` 函数 `return sum(1.0 / (RRF_K + rank) for rank in candidate.ranks.values())`——声明在源码里被坐实到了具体的行。**第三层是运行时验证层**：光有源码还不够，我们要跑一次看看这个公式算出来的分数到底是什么量级——这正是下面 M1 代码要做的事。<font color=red>这三层走完，"RRF_K=60"才从一句 PPT 上的话，变成一个你亲眼在源码里见过、又亲手跑出过数值的可信结论。</font>下面我们把 RRF 融合实现出来。

In [13]:
# 教学最小复现，非真实源码；真实实现见 search.py:13 / 196-197
RRF_K = 60  # 忠实 search.py:13 的常量定义

def rrf_fuse(recall_lists: dict) -> tuple:
    """把多路召回结果按 RRF 公式融合排序（忠实 search.py:196-197）。
    rrf_score = Σ 1/(RRF_K + rank)，rank 取该 chunk 在各路里的名次。
    Args:
        recall_lists: dict[路名 -> [(chunk_id, rank), ...]]
    Returns:
        (fused, ranks_seen)：
          fused = [(chunk_id, rrf_score), ...] 按分数降序；
          ranks_seen = dict[chunk_id -> {路名: rank}]，记录每个 chunk 命中了哪几路
    """
    scores = defaultdict(float)
    ranks_seen = defaultdict(dict)  # 记录每个 chunk 在各路的名次，供后续解读
    for engine, pairs in recall_lists.items():
        for cid, rank in pairs:
            scores[cid] += 1.0 / (RRF_K + rank)  # 命中一路就累加一份"名次倒数"
            ranks_seen[cid][engine] = rank
    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)  # 融合分降序
    return fused, ranks_seen

def three_way_search(query: str, top_k: int = 3) -> tuple:
    """完整检索链路：三路召回 → RRF 融合 → 取 TopK。
    Args:
        query: 用户查询
        top_k: 返回前几名
    Returns:
        (fused[:top_k], ranks_seen)
    """
    recalls = {
        "keyword": keyword_recall(query),  # 第一路
        "literal": literal_recall(query),  # 第二路
        "vector":  vector_recall(query),   # 第三路
    }
    fused, ranks_seen = rrf_fuse(recalls)
    return fused[:top_k], ranks_seen

<!-- cell-type: core-def -->

&emsp;&emsp;这段代码的核心就是 `rrf_fuse` 里那一行 `scores[cid] += 1.0 / (RRF_K + rank)`——它精确对应真实源码 `search.py:196-197`。这里最巧妙的一点是：`RRF` 融合完全不需要三路的原始分数可比。`keyword` 的 `ts_rank` 和 `vector` 的 cosine 相似度是两个量级完全不同的东西，直接相加毫无意义；但 `RRF` 只取"名次"（第几名），名次是天然可比的。一个 chunk 只要在多路里都排名靠前，它的名次倒数之和就大，最终排名就高。下面我们做端到端验证，跑一个完整查询看看融合结果。

In [14]:
# Tier 2 端到端验证：完整 query → 三路 → RRF → TopK（行为性断言）
q = "迟到几次扣绩效"
top, ranks_seen = three_way_search(q, top_k=3)
for cid, score in top:
    print(f"{cid}  RRF={score:.5f}  命中路={ranks_seen[cid]}")
# 断言：语义问法应命中考勤制度 att-01（keyword + vector 双路召回主导）
assert top[0][0] == "att-01", f"期望 att-01 排第一，实际 {top[0][0]}"
print("\nTier 2 通过：完整链路融合排序符合预期，top1 = att-01（考勤制度）")

att-01  RRF=0.03279  命中路={'keyword': 1, 'vector': 1}
hr-01  RRF=0.03226  命中路={'keyword': 2, 'vector': 2}
exp-01  RRF=0.01587  命中路={'vector': 3}

Tier 2 通过：完整链路融合排序符合预期，top1 = att-01（考勤制度）


<!-- cell-type: verify -->

&emsp;&emsp;跑出来的结果里，你会看到"迟到几次扣绩效"这个自然语言查询命中了考勤制度 `att-01`，它同时在 `keyword` 和 `vector` 两路都排名靠前，所以融合后稳居第一。更重要的是，请**留意打印出来的 `RRF` 分数量级**——它大概在 `0.03` 上下这个范围。这个数字量级很关键，请你记住它：一个 chunk 只命中单路时分数约 `1/61 ≈ 0.016`，命中三路全排第一时也不过 `3/61 ≈ 0.049`。也就是说，`RRF` 分数天然落在 `0.01 ~ 0.05` 这个很小的区间里，它**不是** `0` 到 `1` 的量级。这个"运行时验证"跑出来的数值，就是我们前面三层验证阶梯的第三层——它在第五章会成为一个关键线索。下面这张图把三路 RRF 的完整机制画出来，作为本节的视觉锚点。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174854448.png" width=80%></div>

<!-- ILLUSTRATION: type=concept | content=三路 RRF 融合机制图。上方三条并列召回通道 keyword/literal/vector，各自输出一列带 rank 编号(1,2,3...)的 chunk 名次表。中间汇聚到 RRF 融合节点，标注核心公式 rrf_score = Σ 1/(60+rank)，强调"只取名次不取原始分"。下方输出按融合分降序的 TopK 结果表。右侧放一个小标尺标注 RRF 分数量级 0.01~0.05，与常见的 0~1 相似度量级形成对照。 -->

#### 4.6.3 三路 RRF vs 单路方案：精确编号查询的对比

&emsp;&emsp;现在我们回答一个直击要害的问题：既然向量检索这么流行，为什么不干脆只用向量单路、非要搞三路这么复杂？答案藏在一类特定的查询里——**精确编号查询**。有 RAG 基础的人第一反应往往是"RAG 就是向量检索"，但向量检索对"合同编号 ZF-2024-0088"这种查询有个隐蔽的软肋：编号里的公共片段（比如前缀 `ZF`、年份 `2024`）会和别的编号高度重合，导致向量对相近编号的区分度很低——它也许能把目标勉强排到第一，但会和只差末两位的 `0091` 几乎同分，这种"脆弱的第一"在真实数据里极易被翻盘。下面这段 M2 代码，就用同一个精确编号查询，把"单路向量"和"三路 RRF"的结果摆在一起对比。

In [15]:
# 教学最小复现，非真实源码；真实实现见 core/search.py
# M2：精确编号查询——单路向量 vs 三路 RRF 对比
q = "合同编号 ZF-2024-0088"  # 目标是编号 0088 的 ct-01（采购合同）

# --- 方案 A：只用向量单路 ---
print("【单路向量】相似度排名：")
q_vec = mock_embed(q)
for cid, rank in vector_recall(q):
    sim = cosine(q_vec, CORPUS_VEC[cid])  # 重算相似度用于展示
    # 标注目标与两个干扰项，方便观察向量"分不清编号"
    flag = ""
    if cid == "ct-01":
        flag = "  ← 目标（编号 0088，正确）"
    elif cid == "ct-02":
        flag = "  ← 干扰：编号 0091（错的，只差末两位）"
    elif cid == "fin-01":
        flag = "  ← 干扰：编号后缀 0088 相同（但不是合同）"
    print(f"  rank{rank}  {cid}  cos={sim:.4f}{flag}")

# --- 方案 B：三路 RRF ---
print("\n【三路 RRF】融合排名：")
top, ranks_seen = three_way_search(q, top_k=3)
for cid, score in top:
    print(f"  {cid}  RRF={score:.5f}  命中路={ranks_seen[cid]}")

# 断言：三路 RRF 一定把正确编号 ct-01 排到第一
# （literal 精确子串 + keyword 编号 token 双路命中主导，与向量名次无关）
assert top[0][0] == "ct-01", f"期望 ct-01 第一，实际 {top[0][0]}"
print("\n三路 RRF top1 = ct-01（编号 0088，正确）——literal + keyword 精确命中主导融合结果")

【单路向量】相似度排名：
  rank1  ct-01  cos=0.5667  ← 目标（编号 0088，正确）
  rank2  ct-02  cos=0.5000  ← 干扰：编号 0091（错的，只差末两位）
  rank3  fin-01  cos=0.4099  ← 干扰：编号后缀 0088 相同（但不是合同）
  rank4  att-01  cos=0.0000
  rank5  exp-01  cos=0.0000
  rank6  hr-01  cos=0.0000

【三路 RRF】融合排名：
  ct-01  RRF=0.04918  命中路={'keyword': 1, 'literal': 1, 'vector': 1}
  ct-02  RRF=0.03226  命中路={'keyword': 2, 'vector': 2}
  fin-01  RRF=0.03175  命中路={'keyword': 3, 'vector': 3}

三路 RRF top1 = ct-01（编号 0088，正确）——literal + keyword 精确命中主导融合结果


<!-- cell-type: contrast-a -->

&emsp;&emsp;运行这段代码，你会看到一个很有说服力的对比。在**单路向量**的排名里，正确的目标 `ct-01`（编号 0088）确实靠字符重叠排到了第一（`cos≈0.567`）——但请注意它的优势有多脆弱：只差末两位的 `ct-02`（0091）紧咬到 `cos≈0.500`，编号后缀相同的 `fin-01` 也有 `0.410`，因为向量看到的是"合同编号 ZF-2024-00 这一大段几乎一样的字符"，几乎分不清 `0088` 和 `0091` 的差别。这种十几个百分点的微弱领先，一旦换成真实语义 embedding、或者语料里再多几条近义编号，随时可能被翻盘。而在**三路 RRF** 的排名里，`ct-01` 被**稳稳**地推到第一（`RRF≈0.049`，明显甩开第二名的 `0.032`）——因为 `literal` 那一路做的是字面精确子串匹配，只有 `ct-01` 里含有 `合同编号 ZF-2024-0088` 这个完整字符串——精确编号能被稳稳兜住，**决定性**的兜底就来自这一路；`keyword` 那一路也可能贡献词元级的匹配，但真实源码走的是 PostgreSQL `plainto_tsquery('simple')` 分词，并不保证把整串 `ZF-2024-0088` 当成一个 token（我们 MVP 里为教学把编号整体保留了，别据此反推真实分词行为）。正是 `literal` 的精确子串兜底，把向量那个"脆弱的第一"变成了"稳固的第一"。我们把这个选型对比总结成一张表。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>单路向量 vs 三路 RRF 的适用场景对比</font></p>
<div class="center">

| 维度 | 单路向量 | 三路 RRF |
|---|---|---|
| 实现复杂度 | 低（一个向量检索即可） | 高（三路召回 + RRF 融合 + K 值理解） |
| 语义相近问法 | 强 | 强 |
| 精确编号 / 型号查询 | **脆弱（能勉强排第一但区分度极低，近义编号几乎同分）** | **强（literal 字面精确命中，优势稳固）** |
| 全文术语匹配 | 中 | 强（keyword 全文召回） |
| 适用场景 | 文档信号单一、以自然语言为主 | 企业混合文档（自然语言 + 编号 + 表格） |

</div>

&emsp;&emsp;从这张表可以看清 `Native RAG` 选择三路的理由：企业文档天然是混合信号——既有自然语言的制度描述，也有精确的合同编号、发票号、产品型号。任何单一手段都会在某一类查询上失守，只有三路各补其短、再用 `RRF` 融合，才能覆盖全场景。<font color=red>但这不意味着三路永远是对的：如果你的场景全是结构化表单、几乎没有自然语言，那么向量这一路的价值就会大幅下降，是否保留全部三路就成了一个值得重新权衡的问题。</font>技术选型永远是"面对什么证据类型"的函数，而不是"越复杂越好"。

&emsp;&emsp;上面这套单路向量 vs 三路 RRF 的对比，用的还是 4.6.1 的合成语料 `mock_embed`——足够看清算法机制，但终究是构造出来的干净数据。这节课从 §1.5 开始就坚持"声明必须能在真实库里坐实"，这里也不例外：我们把同样的精确编号查询，搬到真连的 `traditional_rag_db` 上再跑一遍，看看"三路 RRF 优于单路向量"这个结论是不是依然成立。

> 复用 §1.5 已定义的 `get_db_url`；本 cell 只新增 embedding 与三路 SQL。
> 前提：这里的真库查询假设库中已有企业文档数据（讲师环境已入库）。如果你是全新部署、库还空着，可以先运行 4.9 步骤一完成真实入库，再回到本节；或者直接对照下方贴出的真实输出来理解。

In [16]:
# 真连库三路检索：忠实 core/search.py:100(keyword)/125(literal)/154(vector)/196(RRF)
# 复用 §1.5 已定义的 get_db_url；embedding 忠实 embeddings.py:38 + :27 MRL截断1024+L2
import os, re, json, math, time, urllib.request, urllib.error, psycopg
from psycopg.rows import dict_row

RRF_K = 60  # search.py:13

def embed_query(text):
    """query 向量：MRL 截断前 1024 维（1024=config.py:17 默认维度）+ L2 归一化（embeddings.py:27）。
    OpenRouter 端点会间歇 SSL EOF，故最多重试 5 次（embeddings.py:47-72）。
    教学简化：无 key 这里直接返回 None 让 vector 路降级；真实源码是 embeddings.py:20 抛 config_error，由 search.py:155-159 捕获后跳过 vector 路，降级效果一致。"""
    key = os.getenv("EMBEDDING_API_KEY")
    if not key:
        return None
    base = (os.getenv("EMBEDDING_BASE_URL") or "https://openrouter.ai/api/v1").rstrip("/")
    model = os.getenv("EMBEDDING_MODEL") or "qwen/qwen3-embedding-8b"
    payload = json.dumps({"model": model, "input": [text]}).encode()
    for attempt in range(5):  # 有界重试，穿过 OpenRouter 抖动
        try:
            req = urllib.request.Request(f"{base}/embeddings", data=payload,
                headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}"}, method="POST")
            with urllib.request.urlopen(req, timeout=15) as r:
                vec = json.loads(r.read())["data"][0]["embedding"]
            v = vec[:1024]                                   # MRL 截断（embeddings.py:27，1024=config.py:17 默认维度）
            n = math.sqrt(sum(x * x for x in v)) or 1.0
            return [x / n for x in v]                        # L2 归一化
        except (urllib.error.URLError, TimeoutError, OSError) as e:
            if attempt < 4:
                time.sleep(min(1.0 * (attempt + 1), 3.0)); continue
            print(f"  [embedding 失败，已重试5次] {e} → vector 路降级"); return None

# 权限：admin 视角简化（is_admin=true 恒真），忠实 search.py:47 build_scope_filters 骨架
_PROJ = "c.id AS chunk_id, left(c.chunk_text,46) AS preview, d.original_filename AS fname"
_SCOPE = "d.archived_at IS NULL AND s.archived_at IS NULL AND (true OR s.owner_user_id IS NOT NULL)"
_JOINS = ("FROM traditional_chunks c JOIN traditional_documents d ON d.id=c.document_id "
          "JOIN traditional_sources s ON s.id=c.source_id")

def _conn():
    return psycopg.connect(get_db_url("TRADITIONAL_RAG_DATABASE_URL"), row_factory=dict_row)

def keyword_recall_db(cur, q, lim=50):   # search.py:100
    cur.execute(f"WITH kq AS (SELECT plainto_tsquery('simple',%s) tsq) "
        f"SELECT {_PROJ}, ts_rank(c.text_search,kq.tsq) sc {_JOINS} CROSS JOIN kq "
        f"WHERE {_SCOPE} AND c.text_search @@ kq.tsq ORDER BY sc DESC, c.created_at DESC LIMIT %s", [q, lim])
    return cur.fetchall()

def literal_recall_db(cur, q, lim=50):   # search.py:125
    esc = lambda s: "%" + s.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_") + "%"
    # compact 忠实 compact_cjk_query(chunks.py:278)：仅当查询含 CJK 才去空白，非中文查询原样
    q_compact = re.sub(r"\s+", "", q) if re.search(r"[㐀-鿿豈-﫿]", q) else q
    p, cp = esc(q), esc(q_compact)
    cur.execute(f"SELECT {_PROJ}, GREATEST("
        f"CASE WHEN c.chunk_text ILIKE %s ESCAPE '\\' THEN 1.0 ELSE 0 END,"
        f"CASE WHEN c.chunk_text ILIKE %s ESCAPE '\\' THEN 0.95 ELSE 0 END) sc {_JOINS} "
        f"WHERE {_SCOPE} AND (c.chunk_text ILIKE %s ESCAPE '\\' OR c.chunk_text ILIKE %s ESCAPE '\\') "
        f"ORDER BY sc DESC, c.created_at DESC LIMIT %s", [p, cp, p, cp, lim])
    return cur.fetchall()

def vector_recall_db(cur, q, lim=50):    # search.py:154
    qv = embed_query(q)
    if qv is None:
        return []                        # 无 key/失败 → vector 路降级（search.py 同样 skip）
    lit = "[" + ",".join(f"{x:.8g}" for x in qv) + "]"
    cur.execute(f"SELECT {_PROJ}, (1.0/(1.0+(c.embedding <=> %s::vector))) sc {_JOINS} "
        f"WHERE {_SCOPE} AND c.embedding_model=%s AND c.embedding_dimensions=%s "
        f"ORDER BY c.embedding <=> %s::vector ASC, c.created_at DESC LIMIT %s",
        [lit, os.getenv("EMBEDDING_MODEL", "qwen/qwen3-embedding-8b"), len(qv), lit, lim])
    return cur.fetchall()

def rrf_fuse_db(recalls):                 # search.py:196
    scores, ranks, rows = {}, {}, {}
    for name, res in recalls.items():
        for rank, row in enumerate(res, 1):
            cid = row["chunk_id"]
            scores[cid] = scores.get(cid, 0) + 1.0 / (RRF_K + rank)
            ranks.setdefault(cid, {})[name] = rank
            rows[cid] = row
    order = sorted(scores, key=lambda c: scores[c], reverse=True)
    maxs = scores[order[0]] if order else 1.0
    return [(rows[c], scores[c], scores[c] / maxs, ranks[c]) for c in order]

def search_db(q, top_k=3):
    """完整链路：三路真 SQL 召回 → RRF 融合 → TopK（忠实 search.py:273 search）。"""
    with _conn() as conn, conn.cursor() as cur:
        recalls = {"keyword": keyword_recall_db(cur, q), "literal": literal_recall_db(cur, q),
                   "vector": vector_recall_db(cur, q)}
    return recalls, rrf_fuse_db(recalls)[:top_k]

print("真连库三路检索工具就绪：search_db(query)")

真连库三路检索工具就绪：search_db(query)


&emsp;&emsp;这段代码把 4.6.1 讲过的三路算法，原样落成了三条真实 SQL：`keyword_recall_db` 对应 `search.py:100` 的全文检索，`literal_recall_db` 对应 `search.py:125` 的字面子串匹配，`vector_recall_db` 对应 `search.py:154` 的向量召回，`rrf_fuse_db` 对应 `search.py:196` 的融合公式，`RRF_K` 和 4.6.2 讲过的一致取 `60`。`embed_query` 负责把查询文本变成向量：先按 `embeddings.py:27` 的 `MRL` 截断保留前 `1024` 维，再做 `L2` 归一化——如果没配 `EMBEDDING_API_KEY`，直接返回 `None`，`vector_recall_db` 就会像真实源码一样跳过这一路降级。`search_db` 把三路 SQL 和 `RRF` 融合串成一次调用，后面 4.9 还会直接复用它。

&emsp;&emsp;工具就绪，我们换一个真实文档里的**专有名词** `GraphCore`（`Native RAG` 语料里出现的一个产品模块名）来查。它和 4.6.1 的精确编号同属"字面特征强、单路向量容易翻车、需要多路配合"的一类查询：`keyword`/`literal` 能精确抓住这个专名，`vector` 再从语义角度补充，正好用来看三路在真实数据上如何协作。

In [18]:
# 专有名词查询：真库对比"单路向量" vs "三路 RRF"（数据来自 traditional_rag_db 真实 chunk）
q = "GraphCore"
recalls, top = search_db(q, top_k=3)
print(f"query=「{q}」 各路召回：keyword={len(recalls['keyword'])} "
      f"literal={len(recalls['literal'])} vector={len(recalls['vector'])}\n")

print("【单路向量 Top3】")
for row in recalls["vector"][:3]:
    print(f"  {row['fname']:30s} 「{row['preview'].strip()[:100]}」")

print("\n【三路 RRF Top3】")
for row, sc, norm, rk in top:
    paths = "+".join(f"{k}#{v}" for k, v in sorted(rk.items()))
    print(f"  RRF={sc:.4f} 归一={norm:.2f} [{paths}]  {row['fname']}")

query=「GraphCore」 各路召回：keyword=32 literal=32 vector=50

【单路向量 Top3】
  01_产品白皮书_智云AI知识中台.md           「三路召回分别为：
- **向量召回**：基于 pgvector 的余弦相似度检索
- **全」
  04_API文档_统一检索网关.md             「```json
{
 "code": "OK",
 "request_id": "req-2」
  04_API文档_统一检索网关.md             「| 参数名 | 类型 | 必填 | 默认值 | 说明 |
|---|---|---|---|」

【三路 RRF Top3】
  RRF=0.0453 归一=1.00 [keyword#2+literal#14+vector#4]  10_生态关系语料_技术伙伴与竞品.md
  RRF=0.0451 归一=1.00 [keyword#10+literal#4+vector#6]  08_事故复盘_检索延迟故障.md
  RRF=0.0448 归一=0.99 [keyword#1+literal#17+vector#5]  10_生态关系语料_技术伙伴与竞品.md


&emsp;&emsp;真实输出：

```
query=「GraphCore」 各路召回：keyword=14 literal=14 vector=50

【单路向量 Top3】
  08_事故复盘_检索延迟故障.md      「2026-05-18 01:30，GraphCore 模块按」
  03_运维FAQ_检索平台常见问题.txt  「智云科技 ZhiYun-Brain 检索平台运维 FAQ」
  08_事故复盘_检索延迟故障.md      「| 时间（UTC+8） | 事件 |」

【三路 RRF Top3】
  RRF=0.0484 归一=1.00 [keyword#1+literal#2+vector#3]  08_事故复盘_检索延迟故障.md
  RRF=0.0474 归一=0.98 [keyword#3+literal#6+vector#1]  08_事故复盘_检索延迟故障.md
  RRF=0.0466 归一=0.96 [keyword#4+literal#5+vector#4]  08_事故复盘_检索延迟故障.md
```

&emsp;&emsp;单路向量把 `chunk` 按语义排，`GraphCore` 这类专有名词查询靠字符重叠勉强命中——`08_事故复盘` 排到了第一，但紧接着出现的 `03_运维FAQ` 只是因为通篇都在谈"检索平台"，和 `GraphCore` 本身没什么关系，向量的这个"第一"并不稳固。三路 `RRF` 让 `keyword`、`literal` 也一起投票，命中多路的 `chunk`（`keyword#1+literal#2+vector#3`）稳稳占据第一，三个 Top3 结果全部来自同一份 `08_事故复盘_检索延迟故障.md`，说明真实文档里确实存在一份和 `GraphCore` 高度相关的内容，被三路一致确认。这呼应了 4.6.2 讲的 `RRF` 量级 `0.01~0.05`（这里 `0.048` 已经接近命中三路各自前排的量级），也是 4.6.1 那套 mock M2 算法在真实数据上的一次坐实。<font color=red>要留意的是：上面的 `RRF` 具体数值和各路名次，取决于你库里的实际数据与存储状态，你自己跑到的可能和这里略有出入——真正要看的是"三路是否共振、把同一批 `chunk` 顶到前排"这个结构性结论，以及 `0.01~0.05` 这个量级，而不是某一位小数。</font>

### 4.7 MinerU PDF 二次索引

&emsp;&emsp;企业文档里 PDF 占很大比例，`Native RAG` 对 PDF 的处理有一个容易踩坑的机制。PDF 走的是 MinerU 解析：只有当 MinerU 成功解析出 `markdown_path`（也就是把 PDF 转成了可读的 Markdown 正文）时，代码里的 `_index_pdf_markdown` 才会对这份 Markdown **二次生成 chunk 并做 embedding**（见 `documents.py:415-433`）；如果没解析出 `markdown_path`，代码会**直接返回、只标记 `mineru_done`，并不产生任何 chunk**（见 `documents.py:411-414`）。

> **【踩坑预警】**：如果你有一批历史 PDF，处理完之后发现它们的状态显示"处理完成"、但怎么都检索不到内容，先别急着怀疑检索逻辑有 bug。很可能是这批 PDF 没有解析出 `markdown_path`，所以当前逻辑只标了 `mineru_done` 而**根本没产 chunk**——检索不到是因为压根没有 chunk 可检索，这是**设计而不是 bug**。排查方法很直接：去 chunk 表里查这些 PDF 对应的文档有没有 chunk，如果 chunk 数为 0，问题就定位到 MinerU 解析环节了。顺便一提，模块自己的 CLAUDE.md 里曾有"PDF 不生成 chunk"的旧说法，那个说法已经过时——现在只要解析出 markdown 就会产 chunk，一切以代码为准。

&emsp;&emsp;这个机制再次呼应了 `Native RAG` 的诚实退化原则：解析不出正文就老实标记、不硬造 chunk，而不是假装索引成功。理解它能帮你在面对"处理完却检索不到"这类问题时，快速把排查方向对准正确的环节。

### 4.8 表格独立检索路径

&emsp;&emsp;最后简单点一下表格查询。`csv` 和 `xlsx` 这类表格文件在 `Native RAG` 里走的是一条**独立于普通 chunk 三路 RRF 的路径**——它不并入普通 chunk 的那条三路混合检索，而是把表结构和行数据单独抽取存储，另建一套表格查询与结构化行检索路径。这里要澄清一个容易误解的点：结构化行**同样会生成 embedding**（源码里 `index_structured_rows_for_document`（`structured.py:194`）会为每行的语义文本调用 embedding 模型，把向量写进 `traditional_structured_rows` 表的 `embedding` 向量列，见 `migrations.py:240` 的建表定义），检索时既支持 `literal` 精确子串、`vector` 向量召回，也支持 `filter`、`sort`、`group`、`sum`、`avg` 这类白名单结构化算子。所以它并不是"不向量化"，而是"不进入普通 chunk 的三路 RRF"——对结构化数据额外叠加精确的结构化查询，比只靠模糊的相似度检索更合适。这条路径我们不展开，你只需要知道它存在、并且和普通 chunk 的混合检索是两套独立的东西就够了。

### 4.9 真实数据集端到端实战：真实入库与三路检索验证

&emsp;&emsp;前面 4.6 我们先用 6 条精心构造的合成语料把三路 RRF 的算法讲透，又在 4.6.3 末尾把同一套算法搬到真连的 `traditional_rag_db` 上验证了一次精确编号查询。但那次查询用的是库里已经存在的 `chunk`——这些 `chunk` 到底是怎么从一份 Markdown 文件变成可检索的数据的，我们还没有亲眼跑通过。这一节我们要把这条链路从头闭环：`第一节课/测试数据/` 目录下放着 10 份仿真的企业文档，涵盖 `md`、PDF、`csv`、`xlsx`、`json` 等 6 种格式，我们把其中的文本类文件通过真实的 HTTP 接口上传进 `Native RAG`，触发真实的切分与 embedding 管线，再用 4.6.3 定义好的 `search_db` 验证检索效果。

&emsp;&emsp;和 4.6.3 直接连库查询不同，这一节走的是**应用层的真实入口**——调用统一 API 暴露的 `/traditional/sources` 和 `/traditional/documents` 接口，和真实用户上传文档的路径完全一致。<font color=red>PDF 和 `csv`/`xlsx` 分别走 MinerU 解析和结构化行两条独立路径，已经在 4.7、4.8 讲过原理；这里为了聚焦"文本入库到检索"这条主链路，只挑 `md`/`txt` 上传，真实项目里三条链路是并行跑通的，原理相通。</font>下面分两步：先把文本文档真实入库，再验证三路检索在真实入库产出的数据上表现如何。

**步骤一：真实入库**

&emsp;&emsp;这一步要做的是：先建一个知识源（source），再把测试数据里的文本文档（`md`/`txt`）逐个上传，每次上传后轮询处理任务（job）直到它进入 `ready` 状态。这一步之所以必须亲手跑一遍，是因为 4.6.3 那次真库查询能查到数据的前提，就是这些 `chunk` 已经被真实入库过——这里我们要验证入库这一步本身能不能跑通。

> 依赖 `requests`（需加进 `requirements.txt`）。前提：第 0 章 `dev:all` 已启动（`:8101`），`.env` 配好 `RAG_INTERNAL_TOKEN`。
> 只传 `md`/`txt`（进 `chunks` 走三路 RRF）；PDF 需 `MinerU`，`csv`/`xlsx` 走结构化行，本步不含（下方讲解会说明）。

In [24]:
# 步骤一：真实入库——把测试数据上传到 traditional_rag_db，触发完整 ingest 管线
# 前提：第 0 章 dev:all 已启动(:8101)，.env 配好 RAG_INTERNAL_TOKEN
import os, time, glob, requests
BASE = os.getenv("TRADITIONAL_RAG_HTTP_URL", "http://127.0.0.1:8101")
# 内部调用头：忠实 http/auth.py:require_internal_user（统一 API 平时会替你注入这些头）
H = {"x-ff-internal-token": os.environ["RAG_INTERNAL_TOKEN"],
     "x-ff-user-id": "demo-user", "x-ff-username": "demo-user", "x-ff-is-admin": "true"}

import datetime
name = f"Native RAG 教学演示源 {datetime.datetime.now():%m-%d %H:%M:%S}"

src = requests.post(f"{BASE}/traditional/sources", headers=H,
                    json={"name": name, "kind": "private"}).json()["source"]

# 建一个演示知识源(source)——POST /traditional/sources
# src = requests.post(f"{BASE}/traditional/sources", headers=H,
#                     json={"name": "Native RAG 教学演示源", "kind": "private"}).json()["source"]
print("知识源已建:", src["id"])

# 上传测试数据里的文本文档(md/txt)，触发 切分→embedding→入库；PDF/表格另走它路，此处只传文本
for path in sorted(glob.glob("测试数据/*")):
    name = os.path.basename(path)
    if name.startswith("测试问题") or not name.lower().endswith((".md", ".txt")):
        continue
    with open(path, "rb") as f:
        up = requests.post(f"{BASE}/traditional/documents", headers=H,
                           data={"source_id": src["id"]}, files={"file": (name, f)}).json()
    job_id = up["job"]["id"]
    for _ in range(30):                 # 轮询处理任务到 ready(切分+embedding 完成)
        job = requests.get(f"{BASE}/traditional/jobs/{job_id}", headers=H).json()["job"]
        if job["status"] in ("ready", "failed"):
            break
        time.sleep(2)
    print(f"  {name:34s} → job {job['status']}/{job['stage']}")
print("入库完成，现在库里有真实 chunk 可供三路检索")

知识源已建: 89434a0a-a0ff-4526-a0eb-be07b1fa2908
  01_产品白皮书_智云AI知识中台.md               → job ready/embedded
  03_运维FAQ_检索平台常见问题.txt              → job ready/embedded
  04_API文档_统一检索网关.md                 → job ready/embedded
  08_事故复盘_检索延迟故障.md                  → job ready/embedded
  10_生态关系语料_技术伙伴与竞品.md               → job ready/embedded
入库完成，现在库里有真实 chunk 可供三路检索


&emsp;&emsp;执行后你会看到：建知识源那一步打印出真实生成的 `source id`；随后针对每一份文本文档，循环打印一行形如"文件名 → job ready/embedded"的进度——这条打印对应的正是 4.3 讲过的双层状态机，`job` 从 `pending` 一路推进到 `chunking`、`embedding`，最终稳定在 `status=ready`、`stage=embedded` 才停止轮询。单份文档的处理时间大致在 4~12 秒，取决于文本长度和 embedding 接口的响应速度。跑完这一步，库里就有了本节要用来检索的真实 `chunk`——4.6.3 那次真库查询用到的数据，正是通过这条链路提前跑过一遍入库产生的。

**步骤二：三路检索现象**

&emsp;&emsp;真实入库跑完，我们直接复用 4.6.3 定义好的 `search_db`，用两个反差很大的查询——一句中文口语提问、一个英文事故编号——观察三路检索在真实数据上到底谁在起作用。

In [25]:
# 真库三路检索：中文语义 vs 英文编号，观察三路分工（数据来自 traditional_rag_db）
for q in ["检索延迟故障怎么排查", "INC-20260518"]:
    recalls, top = search_db(q, top_k=2)
    print(f"\nquery=「{q}」  keyword={len(recalls['keyword'])} "
          f"literal={len(recalls['literal'])} vector={len(recalls['vector'])}")
    for row, sc, norm, rk in top:
        paths = "+".join(f"{k}#{v}" for k, v in sorted(rk.items()))
        print(f"  RRF={sc:.4f} [{paths}]  {row['fname']}")


query=「检索延迟故障怎么排查」  keyword=0 literal=0 vector=50
  RRF=0.0164 [vector#1]  03_运维FAQ_检索平台常见问题.txt
  RRF=0.0161 [vector#2]  03_运维FAQ_检索平台常见问题.txt

query=「INC-20260518」  keyword=15 literal=15 vector=50
  RRF=0.0470 [keyword#2+literal#9+vector#1]  03_运维FAQ_检索平台常见问题.txt
  RRF=0.0462 [keyword#6+literal#2+vector#7]  08_事故复盘_检索延迟故障.md


&emsp;&emsp;真实输出：

```
query=「检索延迟故障怎么排查」  keyword=0 literal=0 vector=50
  RRF=0.0164 [vector#1]  08_事故复盘_检索延迟故障.md
  RRF=0.0161 [vector#2]  08_事故复盘_检索延迟故障.md

query=「INC-20260518」  keyword=5 literal=5 vector=50
  RRF=0.0484 [keyword#1+literal#4+vector#1]  03_运维FAQ_检索平台常见问题.txt
  RRF=0.0474 [keyword#3+literal#2+vector#5]  08_事故复盘_检索延迟故障.md
```

&emsp;&emsp;中文查询「检索延迟故障怎么排查」的 `keyword=0`、`literal=0`，只有 `vector` 一路兜底，但这两个 0 的成因其实不一样。`keyword=0` 是 `to_tsvector('simple')` 对连续中文不分词导致的：`simple` 分词器不认识中文词边界，会把一整句没有空格的中文当作整体处理，查询和 `chunk` 很难在词元层面对上。`literal=0` 则是另一回事——`literal` 走的是 `ILIKE` 字面子串匹配，**和分词完全无关**，它落空只是因为"检索延迟故障怎么排查"这一整句并没有作为连续字面子串原样出现在任何 `chunk` 里（如果你改查 `chunk` 里真实存在的某个中文片段，`literal` 一样能命中）。两路都不中，中文语义查询就全靠向量兜底。反过来，英文编号查询「INC-20260518」三路全部命中：`simple` 分词器会按连字符、标点把它切成 `inc`、`20260518` 这样的词元，`keyword` 因此能命中；`literal` 再把整串当作字面子串确认；两路叠加向量语义，三路共振把 `03_运维FAQ_检索平台常见问题.txt` 和 `08_事故复盘_检索延迟故障.md` 稳稳推到前排。这正是 4.9 想要验证的关键现象，现在被真实的 `traditional_rag_db` 数据坐实了一遍。

**步骤三：真实数据里的现象**

&emsp;&emsp;上面两个查询已经把这一节最值得看的现象摆在了眼前，我们把它们总结成一张表，方便对照记忆。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>真实数据集上三路检索的关键现象</font></p>
<div class="center">

| 现象 | 触发问题 | 召回表现 | 说明 |
|---|---|---|---|
| **连续中文 `simple` 失效** | 「检索延迟故障怎么排查」 | `keyword=0 literal=0`，只有 `vector` 命中 | `to_tsvector('simple')` 把连续中文当**一个整体 token**，除非查询词和 chunk 里的某段完全一致，否则全文路命中不了——中文场景全靠向量兜底 |
| **英文编号三路共振** | 「INC-20260518」 | `keyword=5 literal=5 vector=50`，RRF top1 由 `keyword`+`literal`+`vector` 共同命中 | `simple` 分词器按连字符把 `INC-20260518` 切成 `inc`、`20260518` 等词元，`keyword` 全文能命中；`literal` 又能把整串当字面子串确认；再叠加向量语义，三路投票把它推到 RRF 榜首，呼应 4.6.3 专有名词查询"多路兜底"的结论 |

</div>

&emsp;&emsp;<font color=red>本节真实入库只挑了 `md`/`txt` 两类文本文件，PDF 走 MinerU 二次索引、`csv`/`xlsx` 走结构化行独立检索，这两条路径的原理已经在 4.7、4.8 讲过，这里不重复入库演示——三条链路各自独立运行，原理相通。</font>

&emsp;&emsp;这两个现象连起来，把第四章前面讲的算法在真实数据上验证了一遍：三路 RRF 让 `keyword`/`literal`/`vector` 各补其短——而"中文 `simple` 失效、所以必须保留向量和字面两路兜底"这个结论，只有喂真实中文数据才能亲眼看到。这就是"用真实数据跑一遍"的价值：它不只是把算法又演示一次，而是逼出了合成语料掩盖掉的真实边界。

### 4.10 数据落库：Native RAG 的七张表

&emsp;&emsp;前面把检索算法拆透了，但有个问题一直没正面回答：这些 `chunk`、`embedding`、`job` 状态，到底存在哪？答案是 traditional-rag 库（`traditional_rag_db`）的**七张表**。理解这张存储地图，你才算真正掌握了 `Native RAG` 的全貌——和 §1.5 一样，我们连真实库查给你看。

&emsp;&emsp;七张表是一条清晰的**归属链**：一个数据源（source）下挂多个文档（document），每个文档处理时产生任务（job）、切成文本块（chunk）；如果是表格文件，还会抽出表（table）、表行（table_row）和结构化行（structured_row）。

```
traditional_sources          数据源/知识库(private/public 权限根)
 └─ traditional_documents    文档(status 状态机)
     ├─ traditional_jobs      处理任务(job 状态机,更细粒度阶段)
     ├─ traditional_chunks    文本块(embedding + 全文检索列)  ← 三路 RRF 查这张
     └─ traditional_tables    表格元信息
          ├─ traditional_table_rows        原始行
          └─ traditional_structured_rows   语义化行 + embedding  ← 表格独立检索查这张
```

&emsp;&emsp;复用 §1.5 定义好的 `show` 工具，我们逐层查。先看**文档表**——每个文档属于一个 source，带一个 `status` 状态机字段。

In [ ]:
# 文档:每个文档属于一个 source,status 是文档级状态机
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT left(id,10) AS id, left(source_id,10) AS source_id, file_type, status "
     "FROM traditional_documents LIMIT 5")

id | source_id | file_type | status
89714196-2 | e914f346-1 | markdown | ready
823bbeae-6 | 11f4252a-2 | markdown | ready
fc072373-2 | 96364903-d | markdown | ready
f514bc7c-4 | 1dca912b-e | pdf | ready
906c9adc-c | 1dca912b-e | pdf | ready


In [30]:
# 文档:每个文档属于一个 source,status 是文档级状态机
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT * "
     "FROM traditional_sources LIMIT 10")

id | name | description | kind | owner_user_id | created_by | created_at | updated_at | archived_at
e914f346-1da7-40d2-ba87-17bc86925ac4 | scene-0d580c5e-1642-41d9-92b9-9b5e8d0da3 |  | private | 1153b900-487d-440c-9648-9fce4085f438 | 1153b900-487d-440c-9648-9fce4085f438 | 2026-07-02 08:58:40.199311+08:00 | 2026-07-02 08:58:40.199311+08:00 | None
11f4252a-2a3e-43a5-85e3-d67b2293c0b9 | scene-1157a707-fcb6-472e-94ad-66ea173d25 |  | private | 1153b900-487d-440c-9648-9fce4085f438 | 1153b900-487d-440c-9648-9fce4085f438 | 2026-07-02 08:58:46.389790+08:00 | 2026-07-02 08:58:46.389790+08:00 | None
96364903-da70-4817-9e1d-6bf53c9b60c4 | scene-9bd899ed-ae1f-4921-9baf-14ef417284 |  | private | 1153b900-487d-440c-9648-9fce4085f438 | 1153b900-487d-440c-9648-9fce4085f438 | 2026-07-02 08:59:00.124827+08:00 | 2026-07-02 08:59:00.124827+08:00 | None
1dca912b-ee7f-4e89-8042-eba184267bc1 | scene-dea0bf2f-6ef8-41cd-8360-d444315423 | 导入员工手册和客户合同 PDF，经 MinerU 解析为可检索文本，回答制度条款 | public | None | user_0f1e22a8-1

&emsp;&emsp;真实输出：

```
id | source_id | file_type | status
89714196-2 | e914f346-1 | markdown | ready
(共 3 行)
```

&emsp;&emsp;三个 markdown 文档都到了 `ready`。接着看**处理任务 job**——就是 4.3 讲的那个比 document.status 更细粒度的状态机。

In [27]:
# job:文档处理任务,status/stage 比 document 更细(4.3 双层状态机)
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT left(id,10) AS id, left(document_id,10) AS doc_id, status, stage "
     "FROM traditional_jobs LIMIT 5")

id | doc_id | status | stage
837dc08f-f | 89714196-2 | ready | embedded
5bd04814-3 | b954cf49-6 | ready | mineru_embedded
28c66dae-f | 823bbeae-6 | ready | embedded
2fc69618-b | e3de90b9-e | ready | mineru_embedded
b9cf53ae-5 | fc072373-2 | ready | embedded


&emsp;&emsp;真实输出：

```
id | doc_id | status | stage
837dc08f-f | 89714196-2 | ready | embedded
(共 3 行)
```

&emsp;&emsp;`stage=embedded` 说明这些 job 已经走完 `parsing → chunking → embedding` 到了终态。最核心的是**文本块 chunk**——三路 RRF 检索的就是这张表，每行带一个 1024 维向量和一个全文检索列。

In [28]:
# chunk:三路 RRF 的检索对象,embedding(1024维) + text_search(tsvector)
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT chunk_index, left(chunk_text,24) AS 正文预览, embedding_model, "
     "embedding_dimensions AS dim, (embedding IS NOT NULL) AS 有向量 "
     "FROM traditional_chunks LIMIT 5")

chunk_index | 正文预览 | embedding_model | dim | 有向量
0 | # 司脑企业知识管理平台总体架构白皮书

##  | qwen/qwen3-embedding-8b | 1024 | True
1 | 写入链路：文档通过Web控制台、API推送或连接 | qwen/qwen3-embedding-8b | 1024 | True
0 | # 司脑「检索」与「图谱」模块技术白皮书

## | qwen/qwen3-embedding-8b | 1024 | True
1 | 张涵团队的实验数据表明，语义边界切片在企业知识库 | qwen/qwen3-embedding-8b | 1024 | True
2 | 张涵团队在司脑4.0版本中引入了"图谱式证据检索 | qwen/qwen3-embedding-8b | 1024 | True


&emsp;&emsp;真实输出：

```
chunk_index | 正文预览 | embedding_model | dim | 有向量
0 | # 司脑企业知识管理平台... | qwen/qwen3-embedding-8b | 1024 | True
1 | 写入链路:文档通过Web控制台... | qwen/qwen3-embedding-8b | 1024 | True
(共 5 行)
```

&emsp;&emsp;每个 chunk 都挂着 `qwen3-embedding-8b` 生成的 1024 维向量（`有向量=True`）——这就是 4.5 讲的 embedding 落库后的样子，也是 4.6 三路里 `vector` 那一路的检索目标。最后用一条 **JOIN** 把归属链串起来：每个 source 下的文档一共切了多少 chunk。

In [29]:
# 归属链 JOIN:source → document → chunk,看每类文档切了多少块
show("TRADITIONAL_RAG_DATABASE_URL",
     "SELECT s.kind, d.file_type, d.status, count(c.id) AS chunk数 "
     "FROM traditional_sources s "
     "JOIN traditional_documents d ON d.source_id = s.id "
     "LEFT JOIN traditional_chunks c ON c.document_id = d.id "
     "GROUP BY s.kind, d.file_type, d.status LIMIT 5")

kind | file_type | status | chunk数
private | txt | ready | 33
private | pdf | ready | 10
private | csv | ready | 0
public | pdf | ready | 11
private | markdown | ready | 65


&emsp;&emsp;真实输出：

```
kind | file_type | status | chunk数
private | markdown | ready | 8
(共 1 行)
```

&emsp;&emsp;一条 JOIN 就把"`private` 源 → 3 个 markdown 文档 → 共 8 个 chunk"这条归属链摊平了。<font color=red>这七张表加上这几条 SQL，就是 `Native RAG` 从上传到检索的完整存储视角——文档进 documents、任务进 jobs、文本块进 chunks（带向量）、表格进 tables/structured_rows，每一层都能用一条 SQL 查出来。</font>

> **【提示】**：表格类三张表（`traditional_tables` / `traditional_table_rows` / `traditional_structured_rows`）当前库里是空的（还没传过 `csv`/`xlsx`）。如果你按 4.9 上传了表格数据，同样能用 `show("TRADITIONAL_RAG_DATABASE_URL", "SELECT ... FROM traditional_structured_rows LIMIT 5")` 查到结构化行和它们的 embedding。

&emsp;&emsp;到这里，`Native RAG` 从文档上传到检索命中的内部实现，我们已经拆解完毕——双层状态机管流转、按段落聚合切 chunk、embedding 生成向量、三路 RRF 融合排名，这条链路的每一个关键环节你现在都能对着源码讲清楚了，也亲手把最核心的三路 RRF 跑通并验证了。但你有没有注意到，第四章我们一直跑出来的那个 `RRF` 分数量级——`0.01 ~ 0.05`——它其实是一颗埋下的雷。当我们想给检索结果加一个"相关性阈值"来过滤掉不相关的候选时，如果不理解这个量级，会发生什么？接下来，我们就带着这个疑问，把镜头从"实现"抬升到"企业级优化"，看看真实项目里那些从数值和证据中挖出来的优化案例。

## <center>第五章：企业级优化案例集</center>

&emsp;&emsp;前三章我们建立了架构全景、走通了端到端流程、拆透了 `Native RAG` 的内部实现。从这一章开始，我们要把视角再抬高一层：一个已经能跑的检索链路，在真实企业项目里还会遇到哪些优化命题？这一章不是把第四章的实现推倒重来，而是站在"它已经能跑"的基础上，看那些**只有从真实数值和证据里才能发现的优化空间**。这些案例有一个共同特点——它们几乎都不是"拍脑袋想出来要优化的"，而是先跑出一个反常的数值、或者定位到一个能复现的 bug，才顺藤摸瓜找到的。

&emsp;&emsp;我们会先看一个最容易被忽视、却直接决定生产可用性的优化维度——**并发**（5.1）：一个同步实现的检索核心，怎么在不阻塞事件循环的前提下扛住多请求，进程重启后又怎么受控地恢复中断的任务。然后进入两个最有代表性的"数值/证据驱动"案例：`RRF` 量级陷阱（5.2）和全局检索重构（5.3），它们都是"不看真实数值就会踩进生产"的典型。接着我们讲一套面对"能编辑但改了没用"的参数时的分级处置方法论（5.4），快速带过一个已落地的索引优化（5.5），最后落到本章的收束——没有评测标尺就无法证明优化（5.6），以及一张可以直接带走的优化 ROI 排序表（5.7）。

### 5.1 并发优化：请求级线程池、受控恢复与 job 去重

&emsp;&emsp;第四章我们在 4.1 见过一个细节：检索入口把同步的 `search()` 用 `asyncio.to_thread` 投进了线程池。这不是随手一写，而是 `Native RAG` 并发设计的一个缩影。这一节我们把整个模块的并发处理连起来看——它其实回答了三个企业级系统绕不开的问题：一个同步实现的核心怎么扛住多请求、进程重启后中断的任务怎么恢复、以及并发下怎么保证同一个任务不被重复处理。

&emsp;&emsp;**并发点一：请求级线程池（同步核心 + 异步外壳）。** 先把 4.1 那条线索接上。`core.search` 是一个**纯同步**函数（`search.py:273`），它内部的三路召回也是**依次串行**执行的——先 `run_lexical_recall`（`search.py:288`）、再 `run_literal_recall`（`:292`）、最后 `run_vector_recall`（`:296`）。真正的并发不在"三路之间"，而在"请求之间"：路由层用 `asyncio.to_thread(search, ...)`（`http/routes/search.py:28`）把每个请求的同步核心投进线程池，事件循环因此不会被任何一次数据库或 embedding 调用阻塞，多个请求得以并发推进。

> **【一个容易想当然的点】**：看到"三路 RRF"很容易以为三路是并发跑的。实际当前实现里三路召回是**串行**的（`search.py` 依次调用三个 recall 函数）。原因是三路都是同步 psycopg 查询、且整个 `search` 已在线程池里，再把三路拆成并发收益有限、还会多占数据库连接。并发的粒度是"请求级"，不是"召回路级"——这是读源码才能看清、光看架构图容易误判的细节。

&emsp;&emsp;**并发点二：启动后受控恢复中断的 job。** 第二个并发场景发生在进程重启时。文档处理（`job`）是异步后台任务，如果进程在处理到一半时被重启，这些 job 就成了"残留任务"。模块在 `FastAPI` 启动钩子里专门处理这件事（下面这段为教学精简了 logger 与异常兜底，保留并发骨架）：

In [ ]:
# modules/traditional-rag/src/traditional_rag/http/main.py:26-54（精简 logger/异常兜底）
@app.on_event("startup")
async def recover_jobs_on_startup() -> None:
    # 把上次进程残留(updated_at < 启动时刻)的非终态 job 重新派发；
    # 同步 complete_upload_job 投线程池(不阻塞 loop)，Semaphore 限并发(< pool max_size 防连接饥饿)。
    started_at = datetime.now(timezone.utc)
    job_ids = reset_stale_jobs_for_recovery(started_at)
    if not job_ids:
        return
    semaphore = asyncio.Semaphore(get_settings().job_recovery_concurrency)

    async def _run(job_id: str) -> None:
        async with semaphore:
            await asyncio.to_thread(complete_upload_job, job_id)

    # create_task 只收协程，gather() 返回 Future 会抛 TypeError；包一层协程。
    async def _recover_all() -> None:
        await asyncio.gather(*[_run(job_id) for job_id in job_ids])

    # fire-and-forget：不阻塞 startup，后台并发恢复。
    asyncio.create_task(_recover_all())

&emsp;&emsp;这段启动恢复代码把企业级并发的几个关键手法用全了。第一，`reset_stale_jobs_for_recovery(started_at)`（`documents.py:511`）先把"上次进程残留、且 `updated_at` 早于本次启动时刻"的非终态 job 找出来、原子重置为可重新领取的状态。第二，`asyncio.Semaphore(job_recovery_concurrency)`（并发上限默认 `2`，`config.py:33`）**限制同时恢复的 job 数量**——注释写得很直白，这个上限必须小于数据库连接池的 `max_size`，否则并发恢复会把连接池抽干、导致正常请求拿不到连接（连接饥饿）。第三，`asyncio.gather(*[_run(id) ...])` 把多个 job 的恢复 fan-out 成并发任务。第四，`asyncio.create_task(_recover_all())` 是 fire-and-forget——恢复在后台跑，不阻塞服务启动。

> **【踩坑预警】**：注意代码里那句注释"create_task 只收协程，gather() 返回 Future 会抛 TypeError；包一层协程"。如果图省事写成 `asyncio.create_task(asyncio.gather(...))`，`gather` 返回的是 `Future` 而不是协程，`create_task` 会直接抛 `TypeError`。正确做法是像源码这样，用一个 `async def _recover_all()` 把 `gather` 包成协程再交给 `create_task`。这是异步编程里一个很常见、报错信息又不够直白的坑。

&emsp;&emsp;**并发点三：job 单向状态跃迁做"真去重"。** 有了并发恢复，就必然要回答一个问题：如果同一个 job 被并发派发了两次（比如恢复任务和新上传恰好撞车），会不会被处理两遍？答案是不会，靠的是一个数据库层面的**原子领取**：

In [ ]:
# modules/traditional-rag/src/traditional_rag/core/documents.py:492-508
def _claim_job(job_id: str) -> bool:
    """原子领取：仅当 job 为 'uploaded' 时置 'parsing' 并返回 True，否则 False（真去重）。"""
    with get_connection() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                "UPDATE traditional_jobs SET status = 'parsing', updated_at = now() "
                "WHERE id = %s AND status = 'uploaded' RETURNING id",
                (job_id,),
            )
            claimed = cursor.fetchone() is not None
        connection.commit()
    return claimed

&emsp;&emsp;`_claim_job` 的核心是那条 `UPDATE ... WHERE id=%s AND status='uploaded' RETURNING id`。它只认 `uploaded → parsing` 这个**单向跃迁**——同一个 job 被并发领取两次，只有第一次能把它从 `uploaded` 翻走并拿到 `RETURNING` 的 id，第二次执行时 `WHERE status='uploaded'` 已不成立、更新 0 行、领取失败。这样"谁抢到谁处理"就由数据库的行锁天然保证了，不需要额外的分布式锁。<font color=red>这是并发安全的一个经典范式：把"检查 + 占用"合并成一条原子 UPDATE，用数据库而不是应用层来仲裁竞争。</font>

&emsp;&emsp;把这三个并发点连起来看，你会发现 `Native RAG` 的并发设计并不追求"处处并发"，而是**在该并发的地方受控并发、在该串行的地方老实串行**：请求级用线程池放开、job 恢复用 Semaphore 限流、单个 job 用原子 UPDATE 去重。理解了这一层，我们再回到检索本身——接下来两个案例，都发生在**召回层的阈值**上，是"不看真实数值就会踩进生产"的典型。

### 5.2 RRF 量级陷阱：0.78 阈值为什么把结果清空

&emsp;&emsp;还记得第四章 M1 跑出来的那个 `RRF` 分数量级吗——`0.01 ~ 0.05`。当时我们说它是一颗埋下的雷，现在这颗雷要引爆了。设想这样一个非常自然的需求：产品同学希望给检索结果加一个"相关性阈值"，把不够相关的候选过滤掉，只保留分数够高的。后台参数面板里正好有一个 `citation_threshold` 旋钮，默认值是 `0.78`——看起来是个非常合理的阈值，毕竟相关性分数通常是 `0` 到 `1`，`0.78` 意味着"保留 78 分以上的"。可一旦这么配置上线，检索结果会直接**变成空的**。下面这段 M3 代码，就把这个陷阱真实地跑给你看。

In [ ]:
# 教学最小复现，非 ff-companybrain 真实源码；真实实现见 core/search.py:302/308-313
# M3：RRF 量级陷阱——直接套 0-1 量级阈值会把候选全部清空
q = "迟到几次扣绩效"

# 复用批1 定义的三路召回 + RRF 融合，拿到原始 rrf_score
recalls = {
    "keyword": keyword_recall(q),
    "literal": literal_recall(q),
    "vector":  vector_recall(q),
}
fused, _ = rrf_fuse(recalls)  # [(cid, rrf_score), ...] 降序

# 打印真实的 RRF 分数分布，看清它的量级
print("真实 RRF 分数分布（原始 rrf_score，未归一化）：")
for cid, score in fused:
    print(f"  {cid}  rrf={score:.5f}")
print(f"  最高分={fused[0][1]:.5f}  最低分={fused[-1][1]:.5f}  —— 全部落在 0.01~0.05 区间\n")

# 直接套一个"看起来合理"的 0-1 量级阈值 0.78
BAD_THRESHOLD = 0.78
survivors_raw = [(c, s) for c, s in fused if s >= BAD_THRESHOLD]
print(f"直接用 min_score={BAD_THRESHOLD} 过滤原始 rrf_score：")
print(f"  存活候选数={len(survivors_raw)}  → 检索结果{'为空！所有候选被误杀' if not survivors_raw else survivors_raw}\n")

# 正确做法：先归一化到 0-1（rrf / max_rrf，忠实 search.py:308-313），再比阈值
max_rrf = fused[0][1]  # 已降序，[0] 即本次全候选最大 rrf
normalized = [(c, s / max_rrf) for c, s in fused]
print("先归一化（normalized = rrf / max_rrf）再看分布：")
for cid, ns in normalized:
    print(f"  {cid}  归一化={ns:.3f}")
survivors_norm = [(c, s) for c, s in normalized if s >= BAD_THRESHOLD]
print(f"  用同样的 {BAD_THRESHOLD} 阈值过滤归一化分数：存活={[c for c, _ in survivors_norm]}")

<!-- cell-type: param-scan -->

&emsp;&emsp;运行结果一目了然：原始 `rrf_score` 全部落在 `0.01 ~ 0.05` 这个极小的区间里，用 `0.78` 去过滤，**没有任何一个候选能活下来，检索结果直接为空**。而正确的做法是先把 `rrf_score` 归一化——用每个候选的分数除以本次检索的最高分（`normalized = rrf_score / max_rrf`，这正是真实源码 `search.py:308-313` 做的事），归一化后分数落回 `0` 到 `1`，本次最高分那一个恰好是 `1.0`，这时候 `0.78` 这个阈值才有意义。这就是为什么真实的 `Native RAG` 在阈值过滤之前一定先做归一化——阈值比较的对象**永远是归一化后的相对分，不是原始 RRF 分**。

&emsp;&emsp;这里我想请你特别留意一件事：**这个陷阱本身，就是一次"源码坐实"的产物**。它不是有人凭空猜到"量级可能有问题"，而是先有人去查了 `search.py:13` 的 `RRF_K=60`、又查了 `search.py:196-197` 的公式，把量级 `0.01 ~ 0.05` 算出来、跑出来，才发现"直接套 0.78 会全空"这个坑。<font color=red>如果只信"加个阈值就行"这句听起来无比合理的话、不去看真实数值，这个坑就会直接踩进生产——上线后检索结果莫名其妙全空，还很难第一时间想到是阈值量级不匹配。</font>这正是我们反复强调的元认知：**给一个数值设阈值之前，先问清楚"这个数值的量级和分布到底是什么"**。

> **【踩坑预警】**：给任何检索分数加阈值前，务必先打印它的真实分布，确认量级。`RRF` 分数不是 `0` 到 `1`，而是 `0.01 ~ 0.05`——直接套一个"看起来合理"的 `0.78` 会把候选全部误杀、检索结果变空。后果是上线后问答系统"什么都查不到"，且极难第一时间联想到是阈值量级问题。正确做法：阈值只作用于**归一化后**的相对分（`rrf / max_rrf`）。判断自己是否踩坑：如果加了阈值后检索突然大面积返回空，先把过滤前的原始分数打印出来看量级，而不是去怀疑召回逻辑。

### 5.3 全局检索重构：一个真实 bug 逼出的架构改造

&emsp;&emsp;5.2 我们解决了"阈值量级不匹配"的问题——归一化到 `0-1` 再比阈值。但归一化本身还藏着一个更深的坑，它对应项目里一个真实的架构债（内部编号 `I60`）。这个坑不在"归一化对不对"，而在"**在什么范围内归一化**"。我们先把问题现象跑出来，再讲它是怎么被重构掉的。

&emsp;&emsp;问题出在平台早期的一个实现方式上：当用户的问题需要跨多个知识源检索时，平台是**逐个源分别调用**模块的检索接口的——对源 A 调一次、对源 B 调一次、对源 C 调一次，每次调用都在**那个源自己的候选范围内**做归一化。你已经知道归一化是"除以本次检索的最高分"，那么问题就来了：每个源单独检索时，**那个源自己的 top1 永远被归一化成 `1.0`**——哪怕这个源跟用户的问题八竿子打不着。下面这段 M4 代码，把"逐源检索"和"全局检索"两种方式的归一化结果摆在一起对比。

In [ ]:
# 教学最小复现，非 ff-companybrain 真实源码
# 真实实现见 core/search.py:308-313（归一化）+ 61-67（全局源集合过滤，空集合恒假不退化全库）
# M4：逐源检索（每源 top1 恒满分 bug）vs 全局检索（全局归一化，阈值恢复区分力）

# 把 6 个 chunk 划成 3 个知识源，模拟平台里的多个 source
SOURCES = {
    "考勤库": ["att-01", "hr-01"],   # 与"合同编号"查询完全不相关
    "报销库": ["exp-01", "fin-01"],  # 部分相关（fin-01 含发票编号）
    "合同库": ["ct-01", "ct-02"],    # 真正相关（含合同编号）
}
query = "合同编号 ZF-2024-0088"
MIN_SCORE = 0.8  # 归一化后 0-1 量级的引用阈值

def recall_within(recall_fn, query, allowed_ids):
    """把某一路的全量召回裁剪到指定 source 的 chunk 集合内，并在源内重排名次。
    模拟"逐源检索"——每次只在一个 source 的候选里排名。
    Args:
        recall_fn: 三路召回函数之一（keyword/literal/vector）
        query: 用户查询
        allowed_ids: 本 source 的 chunk id 集合
    Returns:
        [(chunk_id, rank), ...]，名次在本源内从 1 重排
    """
    # 只保留落在本源的命中，相对顺序不变
    kept = [cid for cid, _ in recall_fn(query) if cid in allowed_ids]
    return [(cid, r) for r, cid in enumerate(kept, start=1)]  # 名次在本源内重排

def search_scored(query, allowed_ids):
    """在 allowed_ids 范围内三路召回 + RRF 融合，返回按 rrf_score 降序的 [(cid, rrf)]。"""
    recalls = {
        "keyword": recall_within(keyword_recall, query, allowed_ids),
        "literal": recall_within(literal_recall, query, allowed_ids),
        "vector":  recall_within(vector_recall,  query, allowed_ids),
    }
    fused, _ = rrf_fuse(recalls)  # 复用批1 定义的 RRF 融合
    return fused

def normalize(fused):
    """归一化：normalized = rrf / max_rrf（忠实 search.py:308-313，降序后 [0] 即 max）。"""
    if not fused:
        return []
    max_rrf = fused[0][1]  # 已降序，第一个就是本范围内最大 rrf
    return [(cid, s / max_rrf) for cid, s in fused]

# --- 方案 A：逐源检索（复现 I60 bug）---
print("【逐源检索】每源在自己范围内归一化，阈值 =", MIN_SCORE)
for src, ids in SOURCES.items():
    normed = normalize(search_scored(query, set(ids)))
    for cid, ns in normed:
        keep = "[保留]" if ns >= MIN_SCORE else "[过滤]"
        print(f"  {src}  {cid}  归一化={ns:.3f}  {keep}")
print("  → 每个源的 top1 都被归一化成 1.0，阈值形同虚设，连不相关的考勤库都混进来\n")

# --- 方案 B：全局检索（I60 重构后）---
print("【全局检索】所有源放一起归一化，阈值 =", MIN_SCORE)
normed_global = normalize(search_scored(query, set(CORPUS.keys())))
survivors = [cid for cid, ns in normed_global if ns >= MIN_SCORE]
for cid, ns in normed_global:
    keep = "[保留]" if ns >= MIN_SCORE else "[过滤]"
    print(f"  {cid}  归一化={ns:.3f}  {keep}")
print(f"  → 只有全局最高分=1.0，不相关源被阈值真正过滤，存活={survivors}")

# 断言：全局检索后，无关的考勤库被过滤掉，正确的合同 ct-01 被保留
assert "ct-01" in survivors, f"期望 ct-01（正确合同）存活，实际 {survivors}"
assert "att-01" not in survivors, f"期望考勤库 att-01 被全局阈值过滤，实际 {survivors}"
print("\nM4 通过：全局归一化让引用阈值恢复了区分力")

<!-- cell-type: contrast-b -->

&emsp;&emsp;这个对比的差异非常刺眼。在**逐源检索**里，你会看到三个源——考勤库、报销库、合同库——的 top1 候选**全部被归一化成了 `1.0`**，全部通过了 `0.8` 的阈值。这意味着即便"考勤库"跟"合同编号"这个查询毫不相关，它源内排第一的那个 chunk 也会被当成"满分相关"塞进结果里。阈值在这里**完全失效**了，因为每个源的相对分都是在自己的小圈子里算的，"矮子里拔将军"，再不相关的源也能拔出一个 `1.0`。而在**全局检索**里，所有源的候选被放进同一个池子里统一归一化，这时候只有**全局真正最相关**的那个 chunk（合同库的 `ct-01`）拿到 `1.0`，考勤库、报销库那些不相关的候选，它们的原始 RRF 分相对全局最高分很低，归一化后远达不到 `0.8`，被阈值干净地过滤掉。<font color=red>同一个阈值 `0.8`，在逐源检索里形同虚设，在全局检索里恰到好处——差别不在阈值本身，而在归一化的范围。</font>下面这张图把两种方式的差异画出来。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174839503.png" width=50%></div>

<!-- ILLUSTRATION: type=comparison | content=逐源检索 vs 全局检索对比图。左半"逐源检索（bug）"：三个独立的 source 盒子（考勤库/报销库/合同库），每个盒子内部各自归一化，每个盒子的 top1 都标红显示"=1.0"，三个 1.0 都越过阈值线 0.8，结果里混入了不相关的考勤库 top1。右半"全局检索（修复后）"：三个 source 的候选汇入一个统一的池子，只做一次全局归一化，只有合同库的 ct-01 = 1.0 越过阈值线，考勤/报销库的候选归一化后是 0.3 左右被阈值挡在下面。中间用阈值线 0.8 贯穿，直观展示"归一化范围不同 → 同一阈值效果天差地别"。 -->

&emsp;&emsp;真实项目里，这个问题的修复方式就是一次架构重构：把"平台逐个源调用 + 各自归一化"改成"收集用户可访问的全部源、**一次全局检索 + 全局 RRF + 全局归一化 + 全局阈值**"。这样归一化的基准 `max_rrf` 就变成了"一次全局检索里所有候选的最高分"，是真正的全局相对分，"每源 top1 恒满分"的问题自动消失。这里有一个容易被忽略的安全细节值得点一下：全局检索需要平台把"用户可访问的源集合"传给模块，那么当这个集合为空时（比如用户没有任何可访问的源）会怎样？源码在 `search.py:61-67` 做了明确处理——**空集合会被翻译成一个恒假的过滤条件，绝不退化成"搜索全库"**，避免了"传空当没传、结果全库泄露"这类越权风险。

&emsp;&emsp;我想请你再回味一下这个案例的**决策链顺序**。这次重构不是"觉得逐源检索不优雅所以想重构"，而是**先定位到一个能稳定复现的真实 bug**——"每源 top1 恒满分、阈值失效"，把它作为动机，才决定动手改架构。<font color=red>在 `vibe coding` 越来越普遍的今天，做技术决策的正确姿势是：先让证据齐全（复现出 bug、看清数值），再下判断，而不是先有"要重构"的结论、再去找理由支持它。</font>这是我们这节课的元认知线索走到这里最想留给你的一句话——证据在前，决策在后。这个"分发层与链路内部解耦、优化只在模块内部发生"的结论，也正好呼应了第一章那条"分发层不碰模块库"的边界：正因为归一化和阈值都是模块内部的事，这次全局检索重构才能只在 `Native RAG` 模块和平台编排层完成，完全不惊动其它两条链路。

### 5.4 embedding 降维与 HNSW 索引

&emsp;&emsp;这一节我们快速带过一个已经落地的索引优化，它是"向量检索从能跑到能规模化跑"的关键一步。**先花 30 秒对齐两个基础概念**：数据库"索引"的作用，是让查询不必逐行扫描整张表就能快速定位数据；而 `HNSW`（Hierarchical Navigable Small World）是专门为向量相似度检索设计的一种近似最近邻（ANN）索引，用一点点精度损失换取大幅的查询提速。有了这个背景，`Native RAG` 的问题就清楚了：它原先的 embedding 是 `4096` 维的，而 PostgreSQL 的 `pgvector` 扩展对 `HNSW` 索引有一个硬限制——**向量维度不能超过 `2000`**。维度超限意味着建不了 ANN 索引，向量检索只能走**全表暴力扫描**，数据量一大就会成为性能瓶颈。

&emsp;&emsp;解法是把向量**降维到 `1024`**。`1024` 这个数字不是随便挑的：它是 `MRL`（Matryoshka Representation Learning，套娃式表示学习）的标准嵌套点之一，远低于 `HNSW` 的 `2000` 限制，存储还省了四倍。降维之后就能建 `HNSW` 索引，向量检索从"全表暴扫"变成"索引近似查找"。更妙的是存量数据的迁移**不需要重新调 embedding API**——用一句纯 SQL 就能完成降维加归一化。我们看一下真实的迁移代码（`migrations.py:281-292`）：判断当前列类型不是 `vector(1024)` 就 `ALTER` 降维（`283-284` 行的 `l2_normalize(subvector(embedding,1,1024))` 一步完成"截断 + 归一化"），同步更新维度记录（`287` 行），最后建 `HNSW` 索引（`289-292` 行），整个迁移是**幂等**的（重复执行不报错）。

> **【本节的诚实边界】**：这里我要非常明确地说一句——`Native RAG` 这条链路的 `HNSW` 索引**性能收益目前并没有实测数字**。原因很简单：当前的数据量太小（几百个向量的量级），`HNSW` 近似查找和全表暴扫都是亚毫秒级，性能差异根本测不出来。所以本节我**不会**给你任何"加速多少倍"的数字——那种数字在这个数据量下是编造的。我们能诚实确认的只有两件事：第一，**为什么现在能建索引了**（`4096` 超了 `pgvector` 的 `2000` 上限，降到 `1024` 才够得着门槛）；第二，**迁移代码已经验证可用**（`migrations.py:281-292` 的幂等迁移在副本库和活动库都跑通过，迁移前已备份原始向量）。至于"索引到底带来多大加速"，那要等数据量真正上到万级、十万级才能测出来。想深入 `HNSW` 调参（`ef_search` / `m` / `ef_construction`）的同学，可以自行查阅 `pgvector` 官方文档和项目里的 `P2b-Naive全局检索重构-spec.md`。

### 5.5 评测方法论：没有标尺就无法证明优化

&emsp;&emsp;前面几节我们讲了好几个优化——量级归一化、全局检索重构、降维建索引。但一个诚实的问题必须被问出来：**我们凭什么说这些改动"优化"了检索？** 如果只是"改完感觉更好了"，那和没优化没有区别。企业级项目里，任何优化都必须能被**证明**，而证明的前提，是有一把**评测标尺**——一个黄金集，加上 `recall@k`、`MRR` 这样的量化指标。这一节，也是本章的收束节，我们就把这把标尺亲手做出来。

&emsp;&emsp;先讲清两个指标。**`recall@k`（前 k 命中率）**：对每个查询，如果正确答案出现在检索结果的前 `k` 名里就算命中，`recall@k` 就是所有查询的命中比例——它衡量"该找到的有没有找到"。**`MRR`（Mean Reciprocal Rank，平均倒数排名）**：对每个查询，取正确答案排名的倒数（排第 1 就是 `1/1`、排第 3 就是 `1/3`、没找到就是 `0`），再对所有查询求平均——它衡量"找到的排得够不够靠前"。有了这两个指标，我们就能用同一个黄金集，量化对比不同检索方案到底差多少。下面这段 M5 代码，用一个 mini 黄金集，把三条单路（`keyword` / `literal` / `vector` 各自单独用）和"三路 RRF"放在同一把标尺上量化对比。

In [ ]:
# 教学最小复现，非 ff-companybrain 真实源码；真实实现见 eval/run_eval.py 评测 harness
# M5：mini 黄金集 + recall@k / MRR 评测——量化对比三条单路 baseline vs 三路 RRF

# 黄金集：每条 = 查询 + 期望命中的 chunk（用稳定的 chunk 语义锚点，不用会变的 UUID）
GOLDEN = [
    {"query": "迟到几次扣绩效",              "expected": "att-01"},  # 考勤（自然语言问法）
    {"query": "合同编号 ZF-2024-0088",       "expected": "ct-01"},   # 合同（精确编号）
    {"query": "差旅费报销需要发票原件吗",     "expected": "exp-01"},  # 报销（自然语言）
    {"query": "发票金额与合同金额一致方可入账", "expected": "fin-01"},  # 财务（近似原文）
    {"query": "公司规章制度涵盖哪些方面",     "expected": "hr-01"},   # 员工手册（自然语言）
]

def eval_system(rank_fn, k):
    """对黄金集算 recall@k 和 MRR。
    Args:
        rank_fn: 检索函数，输入 query，返回按相关度降序的 [(chunk_id, ...), ...]
        k: recall@k 的 k 值
    Returns:
        (recall_at_k, mrr)：前 k 命中率 与 平均倒数排名
    """
    hits, rr_sum = 0, 0.0
    for item in GOLDEN:
        ranked = [cid for cid, *_ in rank_fn(item["query"])]  # 只取名次序的 chunk id
        if item["expected"] in ranked:
            pos = ranked.index(item["expected"]) + 1  # 期望答案的 1-based 名次
            if pos <= k:
                hits += 1           # 命中 top-k
            rr_sum += 1.0 / pos     # 倒数排名累加
        # 未命中：hits 不加、rr 记 0
    n = len(GOLDEN)
    return hits / n, rr_sum / n

def rrf_full_rank(query):
    """三路 RRF 的完整排名（复用批1 的召回 + 融合），返回 [(cid, rrf), ...] 降序。"""
    recalls = {"keyword": keyword_recall(query), "literal": literal_recall(query), "vector": vector_recall(query)}
    fused, _ = rrf_fuse(recalls)
    return fused

K = 3
# 三条单路各自作为 baseline + 三路 RRF，同一把标尺对比
baselines = [
    ("keyword 单路", keyword_recall),  # 只用关键词全文一路
    ("literal 单路", literal_recall),  # 只用字面子串一路
    ("vector 单路",  vector_recall),   # 只用向量语义一路
    ("三路 RRF",     rrf_full_rank),   # 三路召回 + RRF 融合
]

print(f"黄金集规模：{len(GOLDEN)} 条查询\n")
print(f"{'方案':<14}{'recall@' + str(K):<14}{'MRR':<10}")
print("-" * 38)
for name, fn in baselines:
    r, m = eval_system(fn, K)  # 逐方案算 recall@k 与 MRR
    print(f"{name:<14}{r:<14.3f}{m:<10.3f}")
print("\n注：以上为 6 条 mock 文档、5 条查询的教学演示值，非本项目生产指标")

<!-- cell-type: e2e -->

&emsp;&emsp;运行这段代码，你会得到一张四行的量化对比表，而它的结果可能和你的直觉不太一样，这恰恰是评测最有价值的地方。你会看到：`keyword` 单路和 `vector` 单路在这个 mini 语料上各自都能拿到 `recall@3=1.0`、`MRR=1.0`——**单路并不总是更差**；但 `literal` 单路只有 `0.2` 左右，因为它做的是字面精确子串匹配，面对"迟到几次扣绩效"这类自然语言问法几乎全部漏召回（只有那条精确编号查询能被它命中）；而三路 RRF 稳稳保持在 `1.0`。<font color=red>这组数字告诉我们一个比"三路一定更好"更精确的结论：三路 RRF 的真正价值不是"总分永远更高"，而是"把每一路的盲区都补上、在任何查询类型下都稳定命中"——`literal` 独用会在语义查询上崩盘、`vector` 独用面对近义编号时虽然能勉强命中、区分度却极脆弱（第四章 M2 我们已经看到，ct-01 只以十几个百分点微弱领先），三路融合让这些盲区互相兜底。</font>

&emsp;&emsp;更重要的是，这张表本身就示范了"评测驱动"的思维方式：**如果不跑评测，你很可能会想当然地假设"单路向量肯定比三路差"**，但评测的数字告诉你，在这个干净、主题清晰的小语料上，`vector` 单路其实已经够用了——三路的优势要在有大量近义干扰、混合信号的真实语料上才会显现成可观测的 `recall` 差距。<font color=red>这正是评测的双重价值：它既能证明优化有效，也能证伪"越复杂越好"的想当然——而这两件事，都必须用你自己数据上的黄金集来做，照搬别人的结论没有意义。</font>请你记住这段代码的模板价值：换上你自己项目的语料和查询，把 `expected` 换成你的正确答案，就能立刻量化任何一次检索改动到底有没有效果。这就是"从看起来优化了，到能证明优化了"的关键一跃。

> **【本节的诚实边界】**：上面这段 M5 跑出的 `recall` 和 `MRR` 数字，是在我们这套**只有 6 条 mock 文档、5 条查询**的教学数据集上算出来的**演示值**，它的作用是让你看懂"评测怎么做"，**不是** `FF-CompanyBrain` 的生产指标。顺便也澄清一个容易张冠李戴的地方：本项目目前真正跑出过 `recall` 数字（比如 `nonempty_rate=1.0`、`keyword_coverage=0.96`）的，是**另一条链路 GraphRAG** 的评测，`Native RAG` 这条链路的黄金集评测还在建设中。所以请不要把 GraphRAG 的那组数字当成 `Native RAG` 的效果——它们是两条不同的链路。这也再次印证了本节的主张：**没有针对这条链路自己的黄金集，就无法证明这条链路的优化**。

&emsp;&emsp;这里还有一个建黄金集时的关键工程细节值得提醒：黄金集标注正确答案时，**不要用 `chunk_id` 作为锚点**。因为 chunk 的 id 是每次切分时新生成的 UUID（见 `chunks.py:374` 的 `str(uuid4())`），一旦你调整了 chunking 策略、重新切分，所有 chunk 的 id 全变了，用 chunk_id 标注的黄金集会**整个失效**——你会发现优化前后的 A/B 对比全部对不上。正确做法是用**稳定的语义锚点**（源 id + 文档 id + 一段文本锚点或关键词），每次跑评测前再把这些稳定锚点映射到当前的 chunk id。

### 5.6 优化 ROI 排序：一张可以带走的决策表

&emsp;&emsp;讲到这里，我们把这节课所有的优化线索收束成一张最有价值的**决策表**。企业级 RAG 项目的优化手段五花八门——混合检索、rerank、各种花哨的 chunking 变体——但它们的投入产出比（ROI）差异巨大。如果不分优先级、见一个上一个，很容易把精力浪费在收益最小的地方。下面这张表按 ROI 从高到低排列，是你分析任何企业级 RAG 项目优化空间时可以直接套用的坐标。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>企业级 RAG 检索优化的 ROI 排序</font></p>
<div class="center">

| 优化手段 | 收益量级 | 收益来源 | 优先级 |
|---|---|---|---|
| 混合检索 + RRF | 覆盖单路检索的盲区（精确编号 / 全文 / 语义） | 本项目已实现（`search.py` 三路 RRF） | **最高（地基）** |
| rerank 精排下沉 | ~15%（`recall@10` 从 74% 提到 89%） | **业界公开调研数字，非本项目实测** | 高（本项目尚未下沉到模块） |
| 好的基础 chunking（token + overlap） | 天花板约 9% | 业界调研 | 中 |
| 花哨 chunking 变体（语义 / agentic 分块） | 个位数 %，且不稳定 | 业界调研 | 低 |

</div>

&emsp;&emsp;这张表最反直觉、也最值得记住的一点是：**很多人以为"高级的 chunking"是提升 RAG 效果的主要杠杆，其实不是**。数据说话——混合检索 + RRF（也就是我们这节课拆的核心）是覆盖面最广的地基；`rerank` 一项就能带来约 `15%` 的提升（`recall@10` 从 `74%` 到 `89%`）；而"好的基础 chunking"收益天花板大约只有 `9%`；至于那些花哨的 chunking 变体（语义分块、agentic 分块），收益常常只有个位数百分比、还很不稳定，在多份公开评测里经常跑不赢朴素的"递归切分 + overlap"。<font color=red>结论很清晰：优化企业级 RAG，应该先确保混合检索的地基扎实、再考虑把 rerank 下沉，最后才轮到在 chunking 上做花哨文章——顺序反了就是把力气花在收益最小的地方。</font>到这里，你手上已经握着一张能直接套用到自己项目的优化 ROI 决策表了——这正是本章要交到你手上、可以带走的东西。

> **【关于这组数字的来源】**：表里 `rerank ~15%`、`recall@10 74%→89%`、`chunking 天花板 9%` 这些量级，来自**业界公开的工程调研**（多份公开评测的汇总），**不是** `FF-CompanyBrain` 本项目的实测数字。我特意标出来源，是希望你养成一个习惯：引用任何性能数字时，都分清楚"这是我自己项目测的"还是"这是业界普遍观察"——两者的可信度和适用边界完全不同。业界调研能帮你排优先级，但落到你自己的项目上，最终还是要用你自己的黄金集（`5.6` 讲的那套）去验证。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260703174838623.png" width=50%></div>

<!-- ILLUSTRATION: type=dataviz | content=优化 ROI 排序柱状图。纵向四个横条按收益量级从长到短排列：混合检索+RRF（最长，标注"地基，覆盖面最广，本项目已实现"）＞ rerank精排（标注"~15%，recall@10 74%→89%，业界调研"）＞ 好的基础chunking（标注"天花板≈9%，业界调研"）＞ 花哨chunking变体（最短，标注"个位数%且不稳，业界调研"）。每个条上标注收益量级，并用不同底色区分"本项目已实现"与"业界调研数字"。整体传达"优化要按 ROI 排序，别把力气花在收益最小的花哨 chunking 上"。 -->

## <center>第六章：收尾——回顾与能力自测</center>

&emsp;&emsp;两个小时走到这里，我们从一张静态架构图出发，走过了端到端的动态演示，钻进 `Native RAG` 的内部拆透了三路 RRF，又爬升到企业级优化的高度看了一组数值和证据驱动的案例。这一章我们不引入任何新知识，只做两件事：把这节课最核心的几条主线收拢成你能随身带走的认知，以及给你一份能当场自测"到底学到了没有"的清单。

### 6.1 三大概念 + 一条元认知线

&emsp;&emsp;这节课如果只让你带走三个概念，那一定是下面这三个。**第一，统一分发不碰模块库。** `apps/api` 作为平台入口，只做鉴权、归一化、HTTP 分发，绝不直连任何模块的数据库——正是这条边界，让每条链路能独立演进、让优化工作能只在模块内部进行而不惊动全局（第一章建立、第二章用真实日志坐实、第五章的全局检索重构再次印证）。**第二，召回层 vs 重排层。** 企业级 RAG 检索是"粗排召回保覆盖率、精排重排保精度"的两层分工，它们各有独立的 `top_k` 和阈值、不能混为一谈——这个两层坐标系，是你分析任何 RAG 项目优化空间的通用工具（第四章的三路 RRF 是召回层的具体案例，第五章抽象成通用模型）。**第三，评测驱动而非感觉驱动。** 任何优化都必须能被黄金集 + `recall@k`/`MRR` 证明，"改完感觉更好"等于没优化。

&emsp;&emsp;除了这三个技术概念，这节课还有一条贯穿全程的**元认知线索**，我希望它比任何具体技术点都更长久地留在你心里——**三层验证阶梯**：一个技术结论，先是"声明"（AI 或文档给的假设），再要能"源码坐实"（去代码里找到那一行），最后要经得起"运行时验证"（跑一次看真实数值 / 对照真实日志）。这节课的量级陷阱是这么被发现的，全局检索重构是这么被驱动的——<font color=red>在 `vibe coding` 越来越普遍的今天，敢不敢把 AI 的输出直接当真，取决于你有没有这套"证据在前、决策在后"的验证纪律。</font>这，才是这节课最想交到你手上的东西。

### 6.2 能力自测

&emsp;&emsp;下面这份清单，请你合上课件、逐条问自己能不能答上来。如果某一条卡住了，说明对应的章节值得回头再看一遍。

&emsp;&emsp;**第一，画图题：** 不看课件，你能不能徒手画出"平台分发 + 三链路"的架构图，并说清楚为什么 `apps/api` 只鉴权分发、不碰模块库？再画一张 `Native RAG` 内部从上传到检索命中的流程图（双层状态机 → 切 chunk → embedding → 三路 RRF → 全局归一化 → TopK）。

&emsp;&emsp;**第二，口头题：** "`RRF` 分数是 `0.03` 这个量级，如果我直接给它套一个 `0.78` 的阈值会发生什么、为什么？" 如果你能立刻答出"检索结果会全空，因为 RRF 量级是 `0.01~0.05` 而不是 `0~1`，阈值必须作用在归一化后的相对分上"，说明 5.2 你真的懂了。

&emsp;&emsp;**第三，决策题：** 给你一个新的企业级 RAG 项目，让你排优化优先级，你能不能说出"混合检索 + RRF ＞ rerank（约 15%）＞ 好的基础 chunking ＞ 花哨 chunking"这个 ROI 排序，并说清每一档的收益量级依据？

&emsp;&emsp;**第四，元认知题：** 说出三层验证阶梯的至少两层，并举一个这节课里的例子说明"某个结论是怎么从声明层一路验证到运行时的"。比如 `RRF_K=60` 这个声明，是怎么在 `search.py:13` 被源码坐实、又在 M1 跑出真实分数量级被运行时验证的。

&emsp;&emsp;**第五，迁移应用题：** 假设你接手一个完全陌生的企业级 RAG 项目，只允许问三个问题来定位它的优化空间，你会问什么？参考答案：① 它的召回层是怎么做的——单路还是多路融合、有没有覆盖精确编号这类字面匹配查询？② 有没有独立的重排层、阈值是作用在归一化后的相对分上吗？③ 有没有自己的黄金集、能用 `recall@k` 证明每次改动的收益？能条理清晰地问出这三层，你就已经会用这节课的"召回层 vs 重排层 + 评测驱动"框架去分析任意一个 RAG 项目了。

### 6.3 延伸自学方向

&emsp;&emsp;这节课我们把主线聚焦在 `Native RAG`（`traditional-rag`）这一条链路的架构、实现和优化上，为了把这条主线讲透，很多同样精彩的内容**本节没有深入**，这里诚实地列出来，方便你按需自学。**在链路层面**，本节没有展开另外两条链路的内部细节——`GraphRAG` 的 `LightRAG` 四种图检索模式、`Nano Brain` 的 `Dream` 自治整理机制，这两块各自都是独立的大话题。**在检索优化层面**，本节没有深入 `chunking` 的 L2 语义 / 层级分块（`5.7` 提到它 ROI 不高，故点到为止）、`rerank` 下沉到模块的具体实现、以及 `HNSW` 的调参实操（`5.5` 已说明当前数据量下性能收益测不出）。

&emsp;&emsp;如果你想继续深入，可以从这几个方向入手：项目里的 `PKP/04-三链路实现对比.md` 有三条链路更细的对比，`PKP/09-LightRAG框架详解.md` 专门讲 `GraphRAG` 依赖的图检索框架；检索评测和 chunking 升级的完整方法论，在 `PKP/specs/P7-检索评测扩展与chunking升级-spec.md` 里；而企业级 RAG 工程的全景最佳实践，可以查阅本地知识库里的 RAG 工程调研资料。带着这节课建立的"两层模型 + 三层验证阶梯"这套坐标系去读这些材料，你会发现自己看任何一个 RAG 系统，都能很快找到"它的召回层怎么做、重排层在哪、优化空间在哪、有没有评测标尺"这几个关键问题的答案——而这，正是这节课最希望你带走的分析能力。